In [1]:
# Cell 0 | 设置 Data_split 和 IL_split 内层30折 MPNN 表征路径与配置

import gc
import hashlib
import json
import os
import random
import sys
import time
import traceback
import warnings

from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from IPython.display import display

from sklearn.model_selection import (
    GroupShuffleSplit,
    train_test_split,
)
from sklearn.preprocessing import StandardScaler

from torch.utils.data import (
    DataLoader,
    Dataset,
    Subset,
)

from torch_geometric.data import Batch
from torch_geometric.nn import (
    NNConv,
    global_mean_pool,
)


warnings.filterwarnings("ignore")


def find_project_root(
    start_path: Union[str, Path] = ".",
) -> Path:
    """向上查找同时包含 data 和 notebooks 的项目根目录。"""
    current_path = Path(
        start_path
    ).resolve()

    for _ in range(20):
        if (
            (current_path / "data").exists()
            and (current_path / "notebooks").exists()
        ):
            return current_path

        if current_path.parent == current_path:
            break

        current_path = current_path.parent

    raise FileNotFoundError(
        "未找到项目根目录。请确认 Notebook 位于：\n"
        r"D:\jupyter\IL_ML_Project\notebooks"
    )


def seed_everything(
    seed: int,
) -> None:
    """固定 Python、NumPy 和 PyTorch 随机种子。"""
    random.seed(seed)

    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def calculate_sha256(
    path: Union[str, Path],
) -> str:
    """计算文件 SHA256。"""
    path = Path(path)

    sha256 = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        for block in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b"",
        ):
            sha256.update(block)

    return sha256.hexdigest()


def read_json(
    path: Union[str, Path],
) -> dict:
    """读取 JSON 文件。"""
    with open(
        path,
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(file)


def read_csv_robust(
    path: Union[str, Path],
) -> pd.DataFrame:
    """依次尝试常见编码读取 CSV。"""
    encodings = [
        "utf-8-sig",
        "utf-8",
        "gb18030",
    ]

    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(
                path,
                encoding=encoding,
            )

        except UnicodeDecodeError as error:
            last_error = error

    raise last_error


def to_jsonable(
    value,
):
    """将 NumPy 类型递归转换为 JSON 可写类型。"""
    if isinstance(
        value,
        dict,
    ):
        return {
            str(key): to_jsonable(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple),
    ):
        return [
            to_jsonable(item)
            for item in value
        ]

    if isinstance(
        value,
        np.integer,
    ):
        return int(value)

    if isinstance(
        value,
        np.floating,
    ):
        return float(value)

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    return value


# ============================================================
# 1. 项目路径
# ============================================================

PROJECT_ROOT = find_project_root(
    Path.cwd()
)

EXPERIMENT_ROOT = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
)

GNN_DIR = (
    EXPERIMENT_ROOT
    / "features"
    / "mpnn_graph"
)

OUTER_FOLD_FEATURE_ROOT = (
    EXPERIMENT_ROOT
    / "outputs"
    / "foldwise_mpnn_cv_features"
)

MODEL_OUTPUT_ROOT = (
    EXPERIMENT_ROOT
    / "outputs"
    / "model_training_3models_cv_no_leakage_2splits"
)

HYPERPARAMETER_TUNING_ROOT = (
    MODEL_OUTPUT_ROOT
    / "hyperparameter_tuning"
)

INNER_SPLIT_ROOT = (
    HYPERPARAMETER_TUNING_ROOT
    / "inner_splits"
)

INNER_MPNN_OUTPUT_ROOT = (
    MODEL_OUTPUT_ROOT
    / "inner_foldwise_mpnn_tuning_features"
)

INNER_MPNN_LOG_DIR = (
    INNER_MPNN_OUTPUT_ROOT
    / "_run_logs"
)


INNER_MPNN_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

INNER_MPNN_LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. 分子图输入
# ============================================================

CATION_GRAPHS_PATH = (
    GNN_DIR
    / "cation_graphs.pt"
)

ANION_GRAPHS_PATH = (
    GNN_DIR
    / "anion_graphs.pt"
)

GRAPH_MANIFEST_PATH = (
    GNN_DIR
    / "graph_manifest.csv"
)


# ============================================================
# 3. 内层划分输入
# ============================================================

INNER_SPLIT_AUDIT_PATH = (
    INNER_SPLIT_ROOT
    / "inner_split_audit_eta_Data_IL.csv"
)

INNER_SPLIT_MANIFEST_PATH = (
    INNER_SPLIT_ROOT
    / "inner_split_manifest_eta_Data_IL.json"
)


def get_outer_fold_dir(
    split_name: str,
    outer_fold: int,
) -> Path:
    """返回某个外层折的现有折专属输出目录。"""
    return (
        OUTER_FOLD_FEATURE_ROOT
        / split_name
        / f"Fold_{outer_fold}"
    )


def get_outer_bundle_paths(
    split_name: str,
    outer_fold: int,
) -> Dict[str, Path]:
    """
    返回内层 MPNN 运行所需的外层折文件。

    不返回外层测试特征、测试标签或旧 MPNN 特征矩阵。
    """
    fold_dir = get_outer_fold_dir(
        split_name=split_name,
        outer_fold=outer_fold,
    )

    return {
        "fold_dir": fold_dir,

        "outer_train_manifest": (
            fold_dir
            / "outer_train_manifest.csv"
        ),

        "fold_indices": (
            fold_dir
            / "fold_indices.npz"
        ),

        "feature_metadata": (
            fold_dir
            / "fold_feature_metadata.json"
        ),

        "output_audit": (
            fold_dir
            / "fold_output_audit.json"
        ),
    }


def get_inner_index_path(
    split_name: str,
    outer_fold: int,
) -> Path:
    """返回一个外层训练折对应的3折内层索引文件。"""
    return (
        INNER_SPLIT_ROOT
        / split_name
        / f"OuterFold_{outer_fold}"
        / (
            f"inner_cv_indices_{split_name}_"
            f"OuterFold{outer_fold}.npz"
        )
    )


def get_inner_task_output_dir(
    split_name: str,
    outer_fold: int,
    inner_fold: int,
) -> Path:
    """返回一套内层专属 MPNN 的独立输出目录。"""
    return (
        INNER_MPNN_OUTPUT_ROOT
        / split_name
        / f"OuterFold_{outer_fold}"
        / f"InnerFold_{inner_fold}"
    )


# ============================================================
# 4. 明确禁止读取的文件
# ============================================================

FORBIDDEN_GLOBAL_FEATURE_PATH = (
    EXPERIMENT_ROOT
    / "outputs"
    / "representation_compare"
    / "mpnn_features_eta.npy"
)

FORBIDDEN_OUTER_FEATURE_FILENAMES = {
    "X_outer_train_mpnn.npy",
    "X_outer_test_mpnn.npy",
    "y_outer_test_lg_eta.npy",
    "outer_test_manifest.csv",
}


# ============================================================
# 5. 任务和训练配置
# ============================================================

SPLIT_ORDER = [
    "Data_split",
    "IL_split",
]

N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3

BASE_SEED = 20260722

MPNN_INTERNAL_VALIDATION_FRACTION = 0.20

BATCH_SIZE = 128
NUM_WORKERS = 0

MAX_EPOCHS = 150
EARLY_STOP_PATIENCE = 20
MIN_DELTA = 1e-6

INITIAL_LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5

RUN_CODE_VERSION = (
    "10_inner_foldwise_mpnn_"
    "Data_IL_30tasks_no_leakage_v1_20260722"
)


def build_task_seed(
    split_name: str,
    outer_fold: int,
    inner_fold: int,
) -> int:
    """为每套内层 MPNN 生成稳定且互不相同的随机种子。"""
    split_offset = (
        SPLIT_ORDER.index(
            split_name
        )
        * 10000
    )

    return int(
        BASE_SEED
        + split_offset
        + outer_fold * 100
        + inner_fold
    )


# 第一轮只检查和运行这一套任务。
TEST_SPLIT_NAME = "Data_split"
TEST_OUTER_FOLD = 1
TEST_INNER_FOLD = 1

TEST_TASK_SEED = build_task_seed(
    split_name=TEST_SPLIT_NAME,
    outer_fold=TEST_OUTER_FOLD,
    inner_fold=TEST_INNER_FOLD,
)


# ============================================================
# 6. 建立30套任务清单
# ============================================================

task_rows = []


for split_name in SPLIT_ORDER:
    for outer_fold in range(
        1,
        N_OUTER_FOLDS + 1,
    ):
        outer_paths = (
            get_outer_bundle_paths(
                split_name=split_name,
                outer_fold=outer_fold,
            )
        )

        inner_index_path = (
            get_inner_index_path(
                split_name=split_name,
                outer_fold=outer_fold,
            )
        )

        for inner_fold in range(
            1,
            N_INNER_FOLDS + 1,
        ):
            output_dir = (
                get_inner_task_output_dir(
                    split_name=split_name,
                    outer_fold=outer_fold,
                    inner_fold=inner_fold,
                )
            )

            task_rows.append(
                {
                    "Task_id": (
                        f"{split_name}_"
                        f"OuterFold{outer_fold}_"
                        f"InnerFold{inner_fold}"
                    ),
                    "Split": split_name,
                    "Outer_fold": outer_fold,
                    "Inner_fold": inner_fold,
                    "Task_seed": build_task_seed(
                        split_name=split_name,
                        outer_fold=outer_fold,
                        inner_fold=inner_fold,
                    ),
                    "Outer_train_manifest": str(
                        outer_paths[
                            "outer_train_manifest"
                        ]
                    ),
                    "Outer_fold_indices": str(
                        outer_paths[
                            "fold_indices"
                        ]
                    ),
                    "Inner_index_path": str(
                        inner_index_path
                    ),
                    "Output_dir": str(
                        output_dir
                    ),
                }
            )


df_inner_mpnn_tasks = pd.DataFrame(
    task_rows
)


expected_task_count = (
    len(SPLIT_ORDER)
    * N_OUTER_FOLDS
    * N_INNER_FOLDS
)


if len(
    df_inner_mpnn_tasks
) != expected_task_count:
    raise RuntimeError(
        "内层 MPNN 任务数错误：\n"
        f"实际={len(df_inner_mpnn_tasks)}\n"
        f"预期={expected_task_count}"
    )


if df_inner_mpnn_tasks[
    "Task_id"
].duplicated().any():
    raise RuntimeError(
        "内层 MPNN 任务编号存在重复。"
    )


# ============================================================
# 7. 检查全部输入文件
# ============================================================

required_common_files = [
    CATION_GRAPHS_PATH,
    ANION_GRAPHS_PATH,
    GRAPH_MANIFEST_PATH,
    INNER_SPLIT_AUDIT_PATH,
    INNER_SPLIT_MANIFEST_PATH,
]

required_outer_files = []
required_inner_index_files = []


for split_name in SPLIT_ORDER:
    for outer_fold in range(
        1,
        N_OUTER_FOLDS + 1,
    ):
        outer_paths = (
            get_outer_bundle_paths(
                split_name=split_name,
                outer_fold=outer_fold,
            )
        )

        required_outer_files.extend(
            [
                outer_paths[
                    "outer_train_manifest"
                ],
                outer_paths[
                    "fold_indices"
                ],
                outer_paths[
                    "feature_metadata"
                ],
                outer_paths[
                    "output_audit"
                ],
            ]
        )

        required_inner_index_files.append(
            get_inner_index_path(
                split_name=split_name,
                outer_fold=outer_fold,
            )
        )


required_files = (
    required_common_files
    + required_outer_files
    + required_inner_index_files
)


missing_files = [
    path
    for path in required_files
    if not Path(path).exists()
]


if missing_files:
    raise FileNotFoundError(
        "缺少以下内层 MPNN 输入文件：\n"
        + "\n".join(
            str(path)
            for path in missing_files
        )
    )


# ============================================================
# 8. 路径安全检查
# ============================================================

experiment_text = str(
    EXPERIMENT_ROOT.resolve()
).lower()

output_text = str(
    INNER_MPNN_OUTPUT_ROOT.resolve()
).lower()


if (
    "small_paper_data"
    not in experiment_text
    or "viscosity_10point_holdout"
    not in experiment_text
):
    raise RuntimeError(
        "实验目录异常，已停止运行。"
    )


if (
    "model_training_3models_cv_no_leakage_2splits"
    not in output_text
    or "inner_foldwise_mpnn_tuning_features"
    not in output_text
):
    raise RuntimeError(
        "内层 MPNN 输出目录异常，"
        "已停止运行以防止覆盖其他结果。"
    )


allowed_required_names = {
    Path(path).name
    for path in required_files
}


for forbidden_name in (
    FORBIDDEN_OUTER_FEATURE_FILENAMES
):
    if forbidden_name in allowed_required_names:
        raise RuntimeError(
            "输入文件列表错误地包含禁止读取文件："
            f"{forbidden_name}"
        )


# ============================================================
# 9. 运行环境
# ============================================================

seed_everything(
    TEST_TASK_SEED
)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

PIN_MEMORY = (
    DEVICE.type == "cuda"
)

CPU_THREADS = max(
    1,
    min(
        8,
        os.cpu_count() or 1,
    ),
)

if DEVICE.type == "cpu":
    torch.set_num_threads(
        CPU_THREADS
    )


# ============================================================
# 10. 输出检查
# ============================================================

task_summary = (
    df_inner_mpnn_tasks
    .groupby(
        "Split",
        as_index=False,
    )
    .agg(
        Outer_fold_count=(
            "Outer_fold",
            "nunique",
        ),
        Inner_task_count=(
            "Task_id",
            "size",
        ),
        Min_seed=(
            "Task_seed",
            "min",
        ),
        Max_seed=(
            "Task_seed",
            "max",
        ),
    )
)


print("=" * 100)
print("Cell 0 内层30折 MPNN 路径与配置检查通过 ✓")
print("=" * 100)

print(
    "python                         =",
    sys.executable,
)

print(
    "torch                          =",
    torch.__version__,
)

print(
    "device                         =",
    DEVICE,
)

print(
    "CPU threads                    =",
    CPU_THREADS,
)

print(
    "PROJECT_ROOT                   =",
    PROJECT_ROOT,
)

print(
    "EXPERIMENT_ROOT                =",
    EXPERIMENT_ROOT,
)

print(
    "OUTER_FOLD_FEATURE_ROOT        =",
    OUTER_FOLD_FEATURE_ROOT,
)

print(
    "INNER_SPLIT_ROOT               =",
    INNER_SPLIT_ROOT,
)

print(
    "INNER_MPNN_OUTPUT_ROOT         =",
    INNER_MPNN_OUTPUT_ROOT,
)

print(
    "splits                         =",
    SPLIT_ORDER,
)

print(
    "outer folds/split              =",
    N_OUTER_FOLDS,
)

print(
    "inner folds/outer fold         =",
    N_INNER_FOLDS,
)

print(
    "expected inner MPNN tasks      =",
    expected_task_count,
)

print(
    "outer input bundles            =",
    len(SPLIT_ORDER)
    * N_OUTER_FOLDS,
)

print(
    "inner index files              =",
    len(
        required_inner_index_files
    ),
)

print(
    "first test task                =",
    (
        TEST_SPLIT_NAME,
        TEST_OUTER_FOLD,
        TEST_INNER_FOLD,
    ),
)

print(
    "first test seed                =",
    TEST_TASK_SEED,
)

print(
    "MPNN internal val fraction     =",
    MPNN_INTERNAL_VALIDATION_FRACTION,
)

print(
    "batch size                     =",
    BATCH_SIZE,
)

print(
    "maximum epochs                 =",
    MAX_EPOCHS,
)

print(
    "early stopping patience        =",
    EARLY_STOP_PATIENCE,
)

print(
    "old global MPNN feature read   =",
    False,
)

print(
    "existing outer MPNN X read     =",
    False,
)

print(
    "outer test manifest read       =",
    False,
)

print(
    "outer test target read         =",
    False,
)

print(
    "external 10 points read        =",
    False,
)

print(
    "outer test indices used only   =",
    "overlap audit",
)

print(
    "run code version               =",
    RUN_CODE_VERSION,
)


print(
    "\n30套任务汇总："
)

display(
    task_summary
)


print(
    "\n前6套任务："
)

display(
    df_inner_mpnn_tasks.head(6)
)

Cell 0 内层30折 MPNN 路径与配置检查通过 ✓
python                         = D:\conda\envs\rdkit-env\python.exe
torch                          = 2.2.1+cpu
device                         = cpu
CPU threads                    = 8
PROJECT_ROOT                   = D:\jupyter\IL_ML_Project
EXPERIMENT_ROOT                = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout
OUTER_FOLD_FEATURE_ROOT        = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\foldwise_mpnn_cv_features
INNER_SPLIT_ROOT               = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning\inner_splits
INNER_MPNN_OUTPUT_ROOT         = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\inner_foldwise_mpnn_tuning_features
splits                         = ['Data_split', 'IL_split']
outer folds/split              = 5
inner folds/outer f

,Split,Outer_fold_count,Inner_task_count,Min_seed,Max_seed
0,Data_split,5,15,20260823,20261225
1,IL_split,5,15,20270823,20271225



前6套任务：


,Task_id,Split,Outer_fold,Inner_fold,Task_seed,Outer_train_manifest,Outer_fold_indices,Inner_index_path,Output_dir
0,Data_split_OuterFold1_InnerFold1,Data_split,1,1,20260823,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...
1,Data_split_OuterFold1_InnerFold2,Data_split,1,2,20260824,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...
2,Data_split_OuterFold1_InnerFold3,Data_split,1,3,20260825,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...
3,Data_split_OuterFold2_InnerFold1,Data_split,2,1,20260923,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...
4,Data_split_OuterFold2_InnerFold2,Data_split,2,2,20260924,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...
5,Data_split_OuterFold2_InnerFold3,Data_split,2,3,20260925,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...


In [2]:
# Cell 1 | 读取无标签分子图并核对30套内层训练与验证索引


# ============================================================
# 1. 辅助函数
# ============================================================

def read_selected_csv_robust(
    path: Union[str, Path],
    required_columns: List[str],
) -> pd.DataFrame:
    """
    只读取指定字段，避免从 graph_manifest.csv
    读取外层测试样本的 eta 和 lg_eta。
    """
    required_set = set(
        required_columns
    )

    encodings = [
        "utf-8-sig",
        "utf-8",
        "gb18030",
    ]

    last_error = None

    for encoding in encodings:
        try:
            dataframe = pd.read_csv(
                path,
                encoding=encoding,
                usecols=lambda column: (
                    str(column).strip()
                    in required_set
                ),
            )

            dataframe.columns = (
                dataframe.columns
                .astype(str)
                .str.strip()
            )

            missing_columns = [
                column
                for column in required_columns
                if column not in dataframe.columns
            ]

            if missing_columns:
                raise KeyError(
                    f"{Path(path).name} 缺少字段："
                    f"{missing_columns}"
                )

            return dataframe[
                required_columns
            ].copy()

        except UnicodeDecodeError as error:
            last_error = error

    raise last_error


def load_torch_object(
    path: Union[str, Path],
):
    """兼容不同 PyTorch 版本读取图对象。"""
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

    except TypeError:
        return torch.load(
            path,
            map_location="cpu",
        )


def require_columns(
    dataframe: pd.DataFrame,
    required_columns: List[str],
    dataframe_name: str,
) -> None:
    """检查 DataFrame 必需字段。"""
    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise KeyError(
            f"{dataframe_name} 缺少字段："
            f"{missing_columns}"
        )


def validate_relative_indices(
    train_idx: np.ndarray,
    valid_idx: np.ndarray,
    expected_rows: int,
    task_name: str,
) -> None:
    """检查一套内层训练和验证相对索引。"""
    train_idx = np.asarray(
        train_idx,
        dtype=np.int64,
    )

    valid_idx = np.asarray(
        valid_idx,
        dtype=np.int64,
    )

    if (
        train_idx.ndim != 1
        or valid_idx.ndim != 1
    ):
        raise ValueError(
            f"{task_name}：内层索引不是一维数组。"
        )

    if (
        len(train_idx) == 0
        or len(valid_idx) == 0
    ):
        raise ValueError(
            f"{task_name}：内层训练或验证索引为空。"
        )

    if (
        len(np.unique(train_idx))
        != len(train_idx)
    ):
        raise ValueError(
            f"{task_name}：内层训练索引存在重复。"
        )

    if (
        len(np.unique(valid_idx))
        != len(valid_idx)
    ):
        raise ValueError(
            f"{task_name}：内层验证索引存在重复。"
        )

    if (
        train_idx.min() < 0
        or valid_idx.min() < 0
        or train_idx.max() >= expected_rows
        or valid_idx.max() >= expected_rows
    ):
        raise IndexError(
            f"{task_name}：内层索引超出外层训练集范围。"
        )

    overlap_count = int(
        np.intersect1d(
            train_idx,
            valid_idx,
        ).size
    )

    if overlap_count != 0:
        raise RuntimeError(
            f"{task_name}：内层训练与验证"
            f"存在 {overlap_count} 条行重叠。"
        )

    combined_idx = np.concatenate(
        [
            train_idx,
            valid_idx,
        ]
    )

    if (
        len(combined_idx) != expected_rows
        or len(np.unique(combined_idx))
        != expected_rows
    ):
        raise RuntimeError(
            f"{task_name}：内层训练与验证"
            "没有完整且唯一地覆盖外层训练集。"
        )


# ============================================================
# 2. 只读取 graph_manifest 中无标签结构字段
# ============================================================

GRAPH_INDEX_COLUMNS = [
    "source_row_idx",
    "graph_valid",
    "graph_list_idx",
    "IL_key",
    "cation_smiles",
    "anion_smiles",
]


graph_index_manifest = (
    read_selected_csv_robust(
        path=GRAPH_MANIFEST_PATH,
        required_columns=(
            GRAPH_INDEX_COLUMNS
        ),
    )
)


for column in [
    "source_row_idx",
    "graph_valid",
    "graph_list_idx",
]:
    graph_index_manifest[
        column
    ] = pd.to_numeric(
        graph_index_manifest[
            column
        ],
        errors="raise",
    ).astype(
        np.int64
    )


for column in [
    "IL_key",
    "cation_smiles",
    "anion_smiles",
]:
    graph_index_manifest[
        column
    ] = (
        graph_index_manifest[
            column
        ]
        .astype(str)
        .str.strip()
    )


if not graph_index_manifest[
    "graph_valid"
].eq(
    1
).all():
    raise RuntimeError(
        "graph_manifest.csv 中存在"
        "未成功构图的样本。"
    )


graph_index_rows = (
    graph_index_manifest
    .sort_values(
        "graph_list_idx"
    )
    .reset_index(
        drop=True
    )
    .copy()
)


expected_graph_indices = np.arange(
    len(
        graph_index_rows
    ),
    dtype=np.int64,
)


if not np.array_equal(
    graph_index_rows[
        "graph_list_idx"
    ].to_numpy(
        dtype=np.int64
    ),
    expected_graph_indices,
):
    raise RuntimeError(
        "graph_list_idx 不是从0开始的连续索引。"
    )


if graph_index_rows[
    "graph_list_idx"
].duplicated().any():
    raise RuntimeError(
        "graph_list_idx 存在重复。"
    )


# ============================================================
# 3. 加载不含标签的阳、阴离子分子图
# ============================================================

all_cation_graphs = load_torch_object(
    CATION_GRAPHS_PATH
)

all_anion_graphs = load_torch_object(
    ANION_GRAPHS_PATH
)


n_graph_rows = len(
    graph_index_rows
)


if not (
    len(all_cation_graphs)
    == len(all_anion_graphs)
    == n_graph_rows
):
    raise RuntimeError(
        "图清单与阳、阴离子图数量不一致：\n"
        f"graph rows    = {n_graph_rows}\n"
        f"cation graphs = {len(all_cation_graphs)}\n"
        f"anion graphs  = {len(all_anion_graphs)}"
    )


# 只检查图对象结构，不读取任何目标值。
for graph_name, graph_list in {
    "cation": all_cation_graphs,
    "anion": all_anion_graphs,
}.items():
    if len(graph_list) == 0:
        raise RuntimeError(
            f"{graph_name} 图列表为空。"
        )

    first_graph = graph_list[0]

    for attribute in [
        "x",
        "edge_index",
        "edge_attr",
    ]:
        if not hasattr(
            first_graph,
            attribute,
        ):
            raise AttributeError(
                f"{graph_name} 图缺少属性："
                f"{attribute}"
            )


# ============================================================
# 4. 逐外层折和内层折完成索引审计
# ============================================================

OUTER_CONTEXT_CACHE = {}
INNER_TASK_INDEX_CACHE = {}

outer_input_manifest_records = []
inner_task_audit_rows = []


OUTER_TRAIN_REQUIRED_COLUMNS = [
    "split_name",
    "fold",
    "subset",
    "original_feature_row_idx",
    "fold_feature_row_idx",
    "source_row_idx",
    "graph_list_idx",
    "IL_key",
    "cation_smiles",
    "anion_smiles",
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]


for split_name in SPLIT_ORDER:
    for outer_fold in range(
        1,
        N_OUTER_FOLDS + 1,
    ):
        outer_paths = (
            get_outer_bundle_paths(
                split_name=split_name,
                outer_fold=outer_fold,
            )
        )

        outer_train_manifest = (
            read_csv_robust(
                outer_paths[
                    "outer_train_manifest"
                ]
            )
        )

        outer_train_manifest.columns = (
            outer_train_manifest.columns
            .astype(str)
            .str.strip()
        )


        require_columns(
            dataframe=outer_train_manifest,
            required_columns=(
                OUTER_TRAIN_REQUIRED_COLUMNS
            ),
            dataframe_name=(
                f"{split_name} "
                f"Outer Fold {outer_fold} "
                "outer_train_manifest"
            ),
        )


        # ----------------------------------------------------
        # 4.1 基本身份字段
        # ----------------------------------------------------

        if not outer_train_manifest[
            "split_name"
        ].astype(str).eq(
            split_name
        ).all():
            raise RuntimeError(
                f"{split_name} Outer Fold {outer_fold}："
                "split_name 字段不一致。"
            )


        if not pd.to_numeric(
            outer_train_manifest[
                "fold"
            ],
            errors="raise",
        ).astype(int).eq(
            outer_fold
        ).all():
            raise RuntimeError(
                f"{split_name} Outer Fold {outer_fold}："
                "fold 字段不一致。"
            )


        if not outer_train_manifest[
            "subset"
        ].astype(str).eq(
            "outer_train"
        ).all():
            raise RuntimeError(
                f"{split_name} Outer Fold {outer_fold}："
                "清单中存在非 outer_train 样本。"
            )


        # ----------------------------------------------------
        # 4.2 数值字段与行顺序
        # ----------------------------------------------------

        integer_columns = [
            "original_feature_row_idx",
            "fold_feature_row_idx",
            "source_row_idx",
            "graph_list_idx",
        ]


        for column in integer_columns:
            outer_train_manifest[
                column
            ] = pd.to_numeric(
                outer_train_manifest[
                    column
                ],
                errors="raise",
            ).astype(
                np.int64
            )


        float_columns = [
            "T_rep",
            "P_rep",
            "eta",
            "lg_eta",
        ]


        for column in float_columns:
            outer_train_manifest[
                column
            ] = pd.to_numeric(
                outer_train_manifest[
                    column
                ],
                errors="raise",
            ).astype(
                float
            )


            if not np.isfinite(
                outer_train_manifest[
                    column
                ].to_numpy(
                    dtype=float
                )
            ).all():
                raise ValueError(
                    f"{split_name} Outer Fold {outer_fold}："
                    f"{column} 中存在 NaN 或无穷值。"
                )


        for column in [
            "IL_key",
            "cation_smiles",
            "anion_smiles",
        ]:
            outer_train_manifest[
                column
            ] = (
                outer_train_manifest[
                    column
                ]
                .astype(str)
                .str.strip()
            )


        n_outer_train = len(
            outer_train_manifest
        )


        expected_fold_row_idx = np.arange(
            n_outer_train,
            dtype=np.int64,
        )


        actual_fold_row_idx = (
            outer_train_manifest[
                "fold_feature_row_idx"
            ].to_numpy(
                dtype=np.int64
            )
        )


        if not np.array_equal(
            actual_fold_row_idx,
            expected_fold_row_idx,
        ):
            raise RuntimeError(
                f"{split_name} Outer Fold {outer_fold}："
                "fold_feature_row_idx 不是"
                "从0开始的连续索引。"
            )


        outer_train_global_idx = (
            outer_train_manifest[
                "original_feature_row_idx"
            ].to_numpy(
                dtype=np.int64
            )
        )


        if (
            outer_train_global_idx.min() < 0
            or outer_train_global_idx.max()
            >= n_graph_rows
        ):
            raise IndexError(
                f"{split_name} Outer Fold {outer_fold}："
                "original_feature_row_idx 超出图清单范围。"
            )


        if (
            len(
                np.unique(
                    outer_train_global_idx
                )
            )
            != n_outer_train
        ):
            raise RuntimeError(
                f"{split_name} Outer Fold {outer_fold}："
                "外层训练全局索引存在重复。"
            )


        # ----------------------------------------------------
        # 4.3 与无标签 graph_manifest 结构字段核对
        # ----------------------------------------------------

        selected_graph_rows = (
            graph_index_rows.iloc[
                outer_train_global_idx
            ]
            .reset_index(
                drop=True
            )
        )


        for column in [
            "source_row_idx",
            "graph_list_idx",
        ]:
            if not np.array_equal(
                outer_train_manifest[
                    column
                ].to_numpy(
                    dtype=np.int64
                ),
                selected_graph_rows[
                    column
                ].to_numpy(
                    dtype=np.int64
                ),
            ):
                raise RuntimeError(
                    f"{split_name} Outer Fold {outer_fold}："
                    f"{column} 与 graph_manifest 不一致。"
                )


        for column in [
            "IL_key",
            "cation_smiles",
            "anion_smiles",
        ]:
            if not np.array_equal(
                outer_train_manifest[
                    column
                ].astype(str).to_numpy(),
                selected_graph_rows[
                    column
                ].astype(str).to_numpy(),
            ):
                raise RuntimeError(
                    f"{split_name} Outer Fold {outer_fold}："
                    f"{column} 与 graph_manifest 不一致。"
                )


        # ----------------------------------------------------
        # 4.4 读取外层索引，仅使用测试索引做重叠审计
        # ----------------------------------------------------

        with np.load(
            outer_paths[
                "fold_indices"
            ],
            allow_pickle=False,
        ) as fold_index_file:
            required_fold_keys = {
                "outer_train_idx",
                "outer_test_idx",
            }


            if not required_fold_keys.issubset(
                set(
                    fold_index_file.files
                )
            ):
                raise KeyError(
                    f"{split_name} Outer Fold {outer_fold}："
                    "fold_indices.npz 缺少"
                    " outer_train_idx 或 outer_test_idx。"
                )


            saved_outer_train_idx = (
                fold_index_file[
                    "outer_train_idx"
                ]
                .astype(
                    np.int64
                )
            )


            outer_test_idx_audit_only = (
                fold_index_file[
                    "outer_test_idx"
                ]
                .astype(
                    np.int64
                )
            )


        if not np.array_equal(
            saved_outer_train_idx,
            outer_train_global_idx,
        ):
            raise RuntimeError(
                f"{split_name} Outer Fold {outer_fold}："
                "outer_train_manifest 与"
                " fold_indices.npz 行顺序不一致。"
            )


        outer_train_test_overlap = int(
            np.intersect1d(
                outer_train_global_idx,
                outer_test_idx_audit_only,
            ).size
        )


        if outer_train_test_overlap != 0:
            raise RuntimeError(
                f"{split_name} Outer Fold {outer_fold}："
                "外层训练和测试索引存在重叠。"
            )


        combined_outer_indices = np.concatenate(
            [
                outer_train_global_idx,
                outer_test_idx_audit_only,
            ]
        )


        if (
            len(combined_outer_indices)
            != n_graph_rows
            or len(
                np.unique(
                    combined_outer_indices
                )
            )
            != n_graph_rows
        ):
            raise RuntimeError(
                f"{split_name} Outer Fold {outer_fold}："
                "外层训练与测试没有完整且唯一地"
                "覆盖全部正式建模数据。"
            )


        # ----------------------------------------------------
        # 4.5 核对原外层表征的审计文件
        # ----------------------------------------------------

        feature_metadata = read_json(
            outer_paths[
                "feature_metadata"
            ]
        )

        output_audit = read_json(
            outer_paths[
                "output_audit"
            ]
        )


        if (
            feature_metadata.get(
                "split_name"
            )
            != split_name
        ):
            raise RuntimeError(
                f"{split_name} Outer Fold {outer_fold}："
                "feature_metadata 的 split_name 不一致。"
            )


        if int(
            feature_metadata.get(
                "fold",
                -1,
            )
        ) != outer_fold:
            raise RuntimeError(
                f"{split_name} Outer Fold {outer_fold}："
                "feature_metadata 的 fold 不一致。"
            )


        if (
            output_audit.get(
                "audit_status"
            )
            != "passed"
        ):
            raise RuntimeError(
                f"{split_name} Outer Fold {outer_fold}："
                "原外层 MPNN 输出审计未通过。"
            )


        leakage_checks = (
            output_audit.get(
                "leakage_checks",
                {},
            )
        )


        for check_name in [
            "outer_test_used_for_mpnn_training",
            "outer_test_used_for_early_stopping",
        ]:
            if bool(
                leakage_checks.get(
                    check_name,
                    True,
                )
            ):
                raise RuntimeError(
                    f"{split_name} Outer Fold {outer_fold}："
                    f"{check_name}=True。"
                )


        # ----------------------------------------------------
        # 4.6 读取该外层折的3套内层索引
        # ----------------------------------------------------

        inner_index_path = (
            get_inner_index_path(
                split_name=split_name,
                outer_fold=outer_fold,
            )
        )


        with np.load(
            inner_index_path,
            allow_pickle=False,
        ) as inner_index_file:
            expected_inner_keys = set()


            for inner_fold in range(
                1,
                N_INNER_FOLDS + 1,
            ):
                expected_inner_keys.add(
                    f"inner_train_idx_{inner_fold}"
                )

                expected_inner_keys.add(
                    f"inner_valid_idx_{inner_fold}"
                )


            if set(
                inner_index_file.files
            ) != expected_inner_keys:
                raise KeyError(
                    f"{split_name} Outer Fold {outer_fold}："
                    "内层索引键名不完整。\n"
                    f"实际={sorted(inner_index_file.files)}\n"
                    f"预期={sorted(expected_inner_keys)}"
                )


            validation_coverage = np.zeros(
                n_outer_train,
                dtype=np.int64,
            )


            for inner_fold in range(
                1,
                N_INNER_FOLDS + 1,
            ):
                task_id = (
                    f"{split_name}_"
                    f"OuterFold{outer_fold}_"
                    f"InnerFold{inner_fold}"
                )


                inner_train_relative_idx = (
                    inner_index_file[
                        f"inner_train_idx_{inner_fold}"
                    ]
                    .astype(
                        np.int64
                    )
                )


                inner_valid_relative_idx = (
                    inner_index_file[
                        f"inner_valid_idx_{inner_fold}"
                    ]
                    .astype(
                        np.int64
                    )
                )


                validate_relative_indices(
                    train_idx=(
                        inner_train_relative_idx
                    ),
                    valid_idx=(
                        inner_valid_relative_idx
                    ),
                    expected_rows=(
                        n_outer_train
                    ),
                    task_name=task_id,
                )


                validation_coverage[
                    inner_valid_relative_idx
                ] += 1


                inner_train_global_idx = (
                    outer_train_global_idx[
                        inner_train_relative_idx
                    ]
                )


                inner_valid_global_idx = (
                    outer_train_global_idx[
                        inner_valid_relative_idx
                    ]
                )


                train_outer_test_overlap = int(
                    np.intersect1d(
                        inner_train_global_idx,
                        outer_test_idx_audit_only,
                    ).size
                )


                valid_outer_test_overlap = int(
                    np.intersect1d(
                        inner_valid_global_idx,
                        outer_test_idx_audit_only,
                    ).size
                )


                if (
                    train_outer_test_overlap != 0
                    or valid_outer_test_overlap != 0
                ):
                    raise RuntimeError(
                        f"{task_id}：内层训练或验证"
                        "与外层测试索引存在重叠。"
                    )


                inner_train_manifest = (
                    outer_train_manifest.iloc[
                        inner_train_relative_idx
                    ]
                )


                inner_valid_manifest = (
                    outer_train_manifest.iloc[
                        inner_valid_relative_idx
                    ]
                )


                if split_name == "IL_split":
                    train_il_set = set(
                        inner_train_manifest[
                            "IL_key"
                        ].astype(str)
                    )

                    valid_il_set = set(
                        inner_valid_manifest[
                            "IL_key"
                        ].astype(str)
                    )

                    il_group_overlap_count = int(
                        len(
                            train_il_set
                            & valid_il_set
                        )
                    )

                    if (
                        il_group_overlap_count
                        != 0
                    ):
                        raise RuntimeError(
                            f"{task_id}：IL_split 内层训练"
                            "与验证存在 IL_key 重叠。"
                        )

                else:
                    il_group_overlap_count = (
                        np.nan
                    )


                INNER_TASK_INDEX_CACHE[
                    (
                        split_name,
                        outer_fold,
                        inner_fold,
                    )
                ] = {
                    "task_id": task_id,
                    "task_seed": (
                        build_task_seed(
                            split_name=split_name,
                            outer_fold=outer_fold,
                            inner_fold=inner_fold,
                        )
                    ),
                    "inner_train_relative_idx": (
                        inner_train_relative_idx.copy()
                    ),
                    "inner_valid_relative_idx": (
                        inner_valid_relative_idx.copy()
                    ),
                    "inner_train_global_idx": (
                        inner_train_global_idx.copy()
                    ),
                    "inner_valid_global_idx": (
                        inner_valid_global_idx.copy()
                    ),
                    "output_dir": (
                        get_inner_task_output_dir(
                            split_name=split_name,
                            outer_fold=outer_fold,
                            inner_fold=inner_fold,
                        )
                    ),
                }


                inner_task_audit_rows.append(
                    {
                        "Task_id": task_id,
                        "Split": split_name,
                        "Outer_fold": outer_fold,
                        "Inner_fold": inner_fold,
                        "Task_seed": (
                            build_task_seed(
                                split_name=split_name,
                                outer_fold=outer_fold,
                                inner_fold=inner_fold,
                            )
                        ),
                        "Total_model_rows": (
                            n_graph_rows
                        ),
                        "Outer_train_rows": (
                            n_outer_train
                        ),
                        "Outer_test_rows": int(
                            len(
                                outer_test_idx_audit_only
                            )
                        ),
                        "Inner_train_rows": int(
                            len(
                                inner_train_relative_idx
                            )
                        ),
                        "Inner_valid_rows": int(
                            len(
                                inner_valid_relative_idx
                            )
                        ),
                        "Inner_train_unique_IL": int(
                            inner_train_manifest[
                                "IL_key"
                            ].nunique()
                        ),
                        "Inner_valid_unique_IL": int(
                            inner_valid_manifest[
                                "IL_key"
                            ].nunique()
                        ),
                        "Inner_train_valid_row_overlap": 0,
                        "Inner_train_outer_test_overlap": (
                            train_outer_test_overlap
                        ),
                        "Inner_valid_outer_test_overlap": (
                            valid_outer_test_overlap
                        ),
                        "IL_group_overlap_count": (
                            il_group_overlap_count
                        ),
                        "Outer_MPNN_audit_status": (
                            output_audit.get(
                                "audit_status"
                            )
                        ),
                        "Graph_archive_contains_labels": False,
                        "Graph_manifest_eta_read": False,
                        "Graph_manifest_lg_eta_read": False,
                        "Outer_test_manifest_read": False,
                        "Outer_test_target_read": False,
                        "Existing_outer_MPNN_X_read": False,
                        "External_10_points_read": False,
                    }
                )


        validation_coverage_error_count = int(
            np.sum(
                validation_coverage != 1
            )
        )


        if (
            validation_coverage_error_count
            != 0
        ):
            raise RuntimeError(
                f"{split_name} Outer Fold {outer_fold}："
                "3个内层验证折没有完整且唯一地"
                "覆盖外层训练集，"
                f"异常行数="
                f"{validation_coverage_error_count}"
            )


        OUTER_CONTEXT_CACHE[
            (
                split_name,
                outer_fold,
            )
        ] = {
            "outer_train_manifest": (
                outer_train_manifest
            ),
            "outer_train_global_idx": (
                outer_train_global_idx.copy()
            ),
            "outer_test_idx_audit_only": (
                outer_test_idx_audit_only.copy()
            ),
            "inner_index_path": (
                inner_index_path
            ),
            "outer_paths": (
                outer_paths
            ),
        }


        outer_input_manifest_records.append(
            {
                "split_name": split_name,
                "outer_fold": outer_fold,
                "outer_train_rows": (
                    n_outer_train
                ),
                "outer_test_rows": int(
                    len(
                        outer_test_idx_audit_only
                    )
                ),
                "outer_train_manifest": str(
                    outer_paths[
                        "outer_train_manifest"
                    ]
                ),
                "outer_train_manifest_sha256": (
                    calculate_sha256(
                        outer_paths[
                            "outer_train_manifest"
                        ]
                    )
                ),
                "fold_indices": str(
                    outer_paths[
                        "fold_indices"
                    ]
                ),
                "fold_indices_sha256": (
                    calculate_sha256(
                        outer_paths[
                            "fold_indices"
                        ]
                    )
                ),
                "inner_index_path": str(
                    inner_index_path
                ),
                "inner_index_sha256": (
                    calculate_sha256(
                        inner_index_path
                    )
                ),
                "outer_test_indices_used_only_for": (
                    "overlap audit"
                ),
            }
        )


# ============================================================
# 5. 汇总与最终强制检查
# ============================================================

df_inner_mpnn_input_audit = pd.DataFrame(
    inner_task_audit_rows
)


if len(
    df_inner_mpnn_input_audit
) != expected_task_count:
    raise RuntimeError(
        "内层 MPNN 输入审计行数错误：\n"
        f"实际={len(df_inner_mpnn_input_audit)}\n"
        f"预期={expected_task_count}"
    )


for column in [
    "Inner_train_valid_row_overlap",
    "Inner_train_outer_test_overlap",
    "Inner_valid_outer_test_overlap",
]:
    if int(
        df_inner_mpnn_input_audit[
            column
        ].sum()
    ) != 0:
        raise RuntimeError(
            f"{column} 总数不为0。"
        )


il_rows = (
    df_inner_mpnn_input_audit[
        "Split"
    ].eq(
        "IL_split"
    )
)


if int(
    df_inner_mpnn_input_audit.loc[
        il_rows,
        "IL_group_overlap_count",
    ].sum()
) != 0:
    raise RuntimeError(
        "IL_split 内层训练与验证"
        "存在 IL_key 重叠。"
    )


if not (
    df_inner_mpnn_input_audit[
        "Outer_MPNN_audit_status"
    ].eq(
        "passed"
    )
).all():
    raise RuntimeError(
        "部分原外层 MPNN 审计未通过。"
    )


# ============================================================
# 6. 建立第一套测试任务上下文
# ============================================================

TEST_TASK_KEY = (
    TEST_SPLIT_NAME,
    TEST_OUTER_FOLD,
    TEST_INNER_FOLD,
)


TEST_OUTER_CONTEXT = (
    OUTER_CONTEXT_CACHE[
        (
            TEST_SPLIT_NAME,
            TEST_OUTER_FOLD,
        )
    ]
)

TEST_TASK_CONTEXT = (
    INNER_TASK_INDEX_CACHE[
        TEST_TASK_KEY
    ]
)


TEST_OUTER_TRAIN_MANIFEST = (
    TEST_OUTER_CONTEXT[
        "outer_train_manifest"
    ]
)


TEST_INNER_TRAIN_RELATIVE_IDX = (
    TEST_TASK_CONTEXT[
        "inner_train_relative_idx"
    ]
)

TEST_INNER_VALID_RELATIVE_IDX = (
    TEST_TASK_CONTEXT[
        "inner_valid_relative_idx"
    ]
)


TEST_INNER_TRAIN_MANIFEST = (
    TEST_OUTER_TRAIN_MANIFEST.iloc[
        TEST_INNER_TRAIN_RELATIVE_IDX
    ]
    .reset_index(
        drop=True
    )
    .copy()
)


TEST_INNER_VALID_MANIFEST = (
    TEST_OUTER_TRAIN_MANIFEST.iloc[
        TEST_INNER_VALID_RELATIVE_IDX
    ]
    .reset_index(
        drop=True
    )
    .copy()
)


# ============================================================
# 7. 保存输入审计文件
# ============================================================

INNER_MPNN_TASK_TABLE_PATH = (
    INNER_MPNN_LOG_DIR
    / "inner_mpnn_task_table_Data_IL.csv"
)

INNER_MPNN_INPUT_AUDIT_PATH = (
    INNER_MPNN_LOG_DIR
    / "inner_mpnn_task_input_audit_Data_IL.csv"
)

INNER_MPNN_INPUT_MANIFEST_PATH = (
    INNER_MPNN_LOG_DIR
    / "inner_mpnn_input_manifest_Data_IL.json"
)


df_inner_mpnn_tasks.to_csv(
    INNER_MPNN_TASK_TABLE_PATH,
    index=False,
    encoding="utf-8-sig",
)


df_inner_mpnn_input_audit.to_csv(
    INNER_MPNN_INPUT_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)


graph_file_metadata = {}


for graph_role, graph_path in {
    "cation_graphs": CATION_GRAPHS_PATH,
    "anion_graphs": ANION_GRAPHS_PATH,
    "graph_manifest": GRAPH_MANIFEST_PATH,
}.items():
    graph_path = Path(
        graph_path
    )

    graph_file_metadata[
        graph_role
    ] = {
        "path": str(
            graph_path
        ),
        "size_bytes": int(
            graph_path.stat().st_size
        ),
        "modified_time": (
            datetime.fromtimestamp(
                graph_path.stat().st_mtime
            ).isoformat(
                timespec="seconds"
            )
        ),
    }


inner_mpnn_input_manifest = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),
    "run_code_version": (
        RUN_CODE_VERSION
    ),
    "property": "viscosity",
    "target": "lg_eta",
    "representation": (
        "inner-fold-specific supervised MPNN"
    ),
    "splits": SPLIT_ORDER,
    "n_outer_folds": (
        N_OUTER_FOLDS
    ),
    "n_inner_folds": (
        N_INNER_FOLDS
    ),
    "expected_task_count": (
        expected_task_count
    ),
    "model_rows": (
        n_graph_rows
    ),
    "graph_input_policy": {
        "graph_archives_contain": (
            "SMILES-derived graph structures only"
        ),
        "graph_archives_contain_targets": False,
        "graph_manifest_columns_read": (
            GRAPH_INDEX_COLUMNS
        ),
        "graph_manifest_eta_read": False,
        "graph_manifest_lg_eta_read": False,
    },
    "leakage_control": {
        "old_global_mpnn_feature_read": False,
        "existing_outer_mpnn_feature_read": False,
        "outer_test_manifest_read": False,
        "outer_test_target_read": False,
        "external_10point_holdout_read": False,
        "outer_test_indices_use": (
            "overlap audit only"
        ),
        "inner_validation_used_for_mpnn_training": False,
        "inner_validation_used_for_mpnn_early_stopping": False,
    },
    "graph_file_metadata": (
        graph_file_metadata
    ),
    "outer_input_records": (
        outer_input_manifest_records
    ),
    "outputs": {
        "task_table": str(
            INNER_MPNN_TASK_TABLE_PATH
        ),
        "input_audit": str(
            INNER_MPNN_INPUT_AUDIT_PATH
        ),
        "input_manifest": str(
            INNER_MPNN_INPUT_MANIFEST_PATH
        ),
    },
}


with open(
    INNER_MPNN_INPUT_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        to_jsonable(
            inner_mpnn_input_manifest
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 8. 输出结果
# ============================================================

input_audit_summary = (
    df_inner_mpnn_input_audit
    .groupby(
        "Split",
        as_index=False,
    )
    .agg(
        Outer_fold_count=(
            "Outer_fold",
            "nunique",
        ),
        Inner_task_count=(
            "Task_id",
            "size",
        ),
        Min_inner_train_rows=(
            "Inner_train_rows",
            "min",
        ),
        Max_inner_train_rows=(
            "Inner_train_rows",
            "max",
        ),
        Min_inner_valid_rows=(
            "Inner_valid_rows",
            "min",
        ),
        Max_inner_valid_rows=(
            "Inner_valid_rows",
            "max",
        ),
        Train_test_overlap_total=(
            "Inner_train_outer_test_overlap",
            "sum",
        ),
        Valid_test_overlap_total=(
            "Inner_valid_outer_test_overlap",
            "sum",
        ),
    )
)


print("=" * 100)
print("Cell 1 分子图和30套内层索引审计通过 ✓")
print("=" * 100)

print(
    "graph rows                         =",
    n_graph_rows,
)

print(
    "cation graph objects               =",
    len(
        all_cation_graphs
    ),
)

print(
    "anion graph objects                =",
    len(
        all_anion_graphs
    ),
)

print(
    "outer contexts checked             =",
    len(
        OUTER_CONTEXT_CACHE
    ),
)

print(
    "inner task contexts checked        =",
    len(
        INNER_TASK_INDEX_CACHE
    ),
)

print(
    "input audit rows                   =",
    len(
        df_inner_mpnn_input_audit
    ),
)

print(
    "inner train/valid overlap total    =",
    int(
        df_inner_mpnn_input_audit[
            "Inner_train_valid_row_overlap"
        ].sum()
    ),
)

print(
    "inner train/outer test overlap     =",
    int(
        df_inner_mpnn_input_audit[
            "Inner_train_outer_test_overlap"
        ].sum()
    ),
)

print(
    "inner valid/outer test overlap     =",
    int(
        df_inner_mpnn_input_audit[
            "Inner_valid_outer_test_overlap"
        ].sum()
    ),
)

print(
    "IL_split group overlap total       =",
    int(
        df_inner_mpnn_input_audit.loc[
            il_rows,
            "IL_group_overlap_count",
        ].sum()
    ),
)

print(
    "graph archives contain targets     =",
    False,
)

print(
    "graph_manifest eta read            =",
    False,
)

print(
    "graph_manifest lg_eta read         =",
    False,
)

print(
    "outer test manifest read           =",
    False,
)

print(
    "outer test target read             =",
    False,
)

print(
    "existing outer MPNN X read         =",
    False,
)

print(
    "external 10 points read            =",
    False,
)

print(
    "\n第一套任务：",
    TEST_TASK_CONTEXT[
        "task_id"
    ],
)

print(
    "first inner train rows             =",
    len(
        TEST_INNER_TRAIN_MANIFEST
    ),
)

print(
    "first inner valid rows             =",
    len(
        TEST_INNER_VALID_MANIFEST
    ),
)

print(
    "first inner train unique IL        =",
    TEST_INNER_TRAIN_MANIFEST[
        "IL_key"
    ].nunique(),
)

print(
    "first inner valid unique IL        =",
    TEST_INNER_VALID_MANIFEST[
        "IL_key"
    ].nunique(),
)


print(
    "\n30套输入审计汇总："
)

display(
    input_audit_summary
)


print(
    "\n前6套输入审计："
)

display(
    df_inner_mpnn_input_audit.head(6)
)


print(
    "\n任务清单："
)

print(
    INNER_MPNN_TASK_TABLE_PATH
)


print(
    "\n输入审计表："
)

print(
    INNER_MPNN_INPUT_AUDIT_PATH
)


print(
    "\n输入 manifest："
)

print(
    INNER_MPNN_INPUT_MANIFEST_PATH
)

Cell 1 分子图和30套内层索引审计通过 ✓
graph rows                         = 16216
cation graph objects               = 16216
anion graph objects                = 16216
outer contexts checked             = 10
inner task contexts checked        = 30
input audit rows                   = 30
inner train/valid overlap total    = 0
inner train/outer test overlap     = 0
inner valid/outer test overlap     = 0
IL_split group overlap total       = 0
graph archives contain targets     = False
graph_manifest eta read            = False
graph_manifest lg_eta read         = False
outer test manifest read           = False
outer test target read             = False
existing outer MPNN X read         = False
external 10 points read            = False

第一套任务： Data_split_OuterFold1_InnerFold1
first inner train rows             = 8648
first inner valid rows             = 4324
first inner train unique IL        = 1180
first inner valid unique IL        = 1047

30套输入审计汇总：


,Split,Outer_fold_count,Inner_task_count,Min_inner_train_rows,Max_inner_train_rows,Min_inner_valid_rows,Max_inner_valid_rows,Train_test_overlap_total,Valid_test_overlap_total
0,Data_split,5,15,8648,8649,4324,4325,0,0
1,IL_split,5,15,8648,8649,4324,4325,0,0



前6套输入审计：


,Task_id,Split,Outer_fold,Inner_fold,Task_seed,Total_model_rows,Outer_train_rows,Outer_test_rows,Inner_train_rows,Inner_valid_rows,...,Inner_valid_outer_test_overlap,IL_group_overlap_count,Outer_MPNN_audit_status,Graph_archive_contains_labels,Graph_manifest_eta_read,Graph_manifest_lg_eta_read,Outer_test_manifest_read,Outer_test_target_read,Existing_outer_MPNN_X_read,External_10_points_read
0,Data_split_OuterFold1_InnerFold1,Data_split,1,1,20260823,16216,12972,3244,8648,4324,...,0,NaN,passed,False,False,False,False,False,False,False
1,Data_split_OuterFold1_InnerFold2,Data_split,1,2,20260824,16216,12972,3244,8648,4324,...,0,NaN,passed,False,False,False,False,False,False,False
2,Data_split_OuterFold1_InnerFold3,Data_split,1,3,20260825,16216,12972,3244,8648,4324,...,0,NaN,passed,False,False,False,False,False,False,False
3,Data_split_OuterFold2_InnerFold1,Data_split,2,1,20260923,16216,12973,3243,8648,4325,...,0,NaN,passed,False,False,False,False,False,False,False
4,Data_split_OuterFold2_InnerFold2,Data_split,2,2,20260924,16216,12973,3243,8649,4324,...,0,NaN,passed,False,False,False,False,False,False,False
5,Data_split_OuterFold2_InnerFold3,Data_split,2,3,20260925,16216,12973,3243,8649,4324,...,0,NaN,passed,False,False,False,False,False,False,False



任务清单：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\inner_foldwise_mpnn_tuning_features\_run_logs\inner_mpnn_task_table_Data_IL.csv

输入审计表：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\inner_foldwise_mpnn_tuning_features\_run_logs\inner_mpnn_task_input_audit_Data_IL.csv

输入 manifest：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\inner_foldwise_mpnn_tuning_features\_run_logs\inner_mpnn_input_manifest_Data_IL.json


In [3]:
# Cell 2 | 为第一套任务建立MPNN监督训练集、早停集和标准化器


def fitted_sample_count(
    scaler,
) -> int:
    """返回 sklearn 标准化器实际拟合的样本数。"""
    value = np.asarray(
        scaler.n_samples_seen_
    ).reshape(-1)

    return int(
        value[0]
    )


def build_inner_mpnn_preprocessing_context(
    split_name: str,
    outer_fold: int,
    inner_fold: int,
    save_outputs: bool = True,
) -> dict:
    """
    为一套内层任务建立无泄露的 MPNN 数据边界。

    内层验证集：
    - 不参与 MPNN 反向传播；
    - 不参与 MPNN 早停；
    - 不参与 T/P 标准化器拟合；
    - 不参与目标标准化器拟合。
    """
    task_key = (
        split_name,
        outer_fold,
        inner_fold,
    )

    outer_key = (
        split_name,
        outer_fold,
    )


    if outer_key not in OUTER_CONTEXT_CACHE:
        raise KeyError(
            f"找不到外层任务上下文：{outer_key}"
        )


    if task_key not in INNER_TASK_INDEX_CACHE:
        raise KeyError(
            f"找不到内层任务上下文：{task_key}"
        )


    outer_context = (
        OUTER_CONTEXT_CACHE[
            outer_key
        ]
    )

    task_context = (
        INNER_TASK_INDEX_CACHE[
            task_key
        ]
    )


    task_id = task_context[
        "task_id"
    ]

    task_seed = int(
        task_context[
            "task_seed"
        ]
    )


    seed_everything(
        task_seed
    )


    outer_train_manifest = (
        outer_context[
            "outer_train_manifest"
        ]
        .copy()
    )


    outer_test_idx_audit_only = (
        outer_context[
            "outer_test_idx_audit_only"
        ]
        .copy()
    )


    inner_train_relative_idx = (
        task_context[
            "inner_train_relative_idx"
        ]
        .astype(
            np.int64
        )
    )


    inner_valid_relative_idx = (
        task_context[
            "inner_valid_relative_idx"
        ]
        .astype(
            np.int64
        )
    )


    inner_train_manifest = (
        outer_train_manifest.iloc[
            inner_train_relative_idx
        ]
        .reset_index(
            drop=True
        )
        .copy()
    )


    inner_valid_manifest = (
        outer_train_manifest.iloc[
            inner_valid_relative_idx
        ]
        .reset_index(
            drop=True
        )
        .copy()
    )


    # 专供 MPNN 特征提取的无目标内层验证清单。
    inner_valid_structure_manifest = (
        inner_valid_manifest.drop(
            columns=[
                "eta",
                "lg_eta",
            ],
            errors="raise",
        )
        .copy()
    )


    # --------------------------------------------------------
    # 1. 再次核对内层清单与 Cell 1 索引
    # --------------------------------------------------------

    inner_train_global_idx = (
        inner_train_manifest[
            "original_feature_row_idx"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    inner_valid_global_idx = (
        inner_valid_manifest[
            "original_feature_row_idx"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    if not np.array_equal(
        inner_train_global_idx,
        task_context[
            "inner_train_global_idx"
        ],
    ):
        raise RuntimeError(
            f"{task_id}：内层训练全局行号"
            "与 Cell 1 缓存不一致。"
        )


    if not np.array_equal(
        inner_valid_global_idx,
        task_context[
            "inner_valid_global_idx"
        ],
    ):
        raise RuntimeError(
            f"{task_id}：内层验证全局行号"
            "与 Cell 1 缓存不一致。"
        )


    if np.intersect1d(
        inner_train_global_idx,
        inner_valid_global_idx,
    ).size != 0:
        raise RuntimeError(
            f"{task_id}：内层训练与验证存在行重叠。"
        )


    if np.intersect1d(
        inner_train_global_idx,
        outer_test_idx_audit_only,
    ).size != 0:
        raise RuntimeError(
            f"{task_id}：内层训练与外层测试存在重叠。"
        )


    if np.intersect1d(
        inner_valid_global_idx,
        outer_test_idx_audit_only,
    ).size != 0:
        raise RuntimeError(
            f"{task_id}：内层验证与外层测试存在重叠。"
        )


    # --------------------------------------------------------
    # 2. 只在内层训练集内部划分：
    #    MPNN监督训练集 + MPNN早停集
    # --------------------------------------------------------

    n_inner_train = len(
        inner_train_manifest
    )


    inner_train_positions = np.arange(
        n_inner_train,
        dtype=np.int64,
    )


    if split_name == "Data_split":
        (
            mpnn_supervised_train_pos,
            mpnn_early_stop_pos,
        ) = train_test_split(
            inner_train_positions,
            test_size=(
                MPNN_INTERNAL_VALIDATION_FRACTION
            ),
            random_state=task_seed,
            shuffle=True,
        )

        internal_split_rule = (
            "train_test_split by data point"
        )

        internal_group_column = None

        internal_group_overlap_count = (
            np.nan
        )


    elif split_name == "IL_split":
        inner_train_il_groups = (
            inner_train_manifest[
                "IL_key"
            ]
            .astype(str)
            .to_numpy()
        )


        internal_splitter = (
            GroupShuffleSplit(
                n_splits=1,
                test_size=(
                    MPNN_INTERNAL_VALIDATION_FRACTION
                ),
                random_state=task_seed,
            )
        )


        (
            mpnn_supervised_train_pos,
            mpnn_early_stop_pos,
        ) = next(
            internal_splitter.split(
                X=np.zeros(
                    n_inner_train,
                    dtype=np.float32,
                ),
                groups=(
                    inner_train_il_groups
                ),
            )
        )


        internal_split_rule = (
            "GroupShuffleSplit by IL_key"
        )

        internal_group_column = (
            "IL_key"
        )


        supervised_il_set = set(
            inner_train_manifest.iloc[
                mpnn_supervised_train_pos
            ][
                "IL_key"
            ].astype(str)
        )


        early_stop_il_set = set(
            inner_train_manifest.iloc[
                mpnn_early_stop_pos
            ][
                "IL_key"
            ].astype(str)
        )


        internal_group_overlap_count = int(
            len(
                supervised_il_set
                & early_stop_il_set
            )
        )


        if (
            internal_group_overlap_count
            != 0
        ):
            raise RuntimeError(
                f"{task_id}：MPNN监督训练集"
                "与早停集存在 IL_key 重叠。"
            )


    else:
        raise ValueError(
            f"不支持的划分方式：{split_name}"
        )


    mpnn_supervised_train_pos = (
        np.sort(
            np.asarray(
                mpnn_supervised_train_pos,
                dtype=np.int64,
            )
        )
    )


    mpnn_early_stop_pos = (
        np.sort(
            np.asarray(
                mpnn_early_stop_pos,
                dtype=np.int64,
            )
        )
    )


    # --------------------------------------------------------
    # 3. 检查内部训练与早停边界
    # --------------------------------------------------------

    train_stop_overlap_count = int(
        np.intersect1d(
            mpnn_supervised_train_pos,
            mpnn_early_stop_pos,
        ).size
    )


    if train_stop_overlap_count != 0:
        raise RuntimeError(
            f"{task_id}：MPNN监督训练集"
            "与早停集存在行重叠。"
        )


    internal_union = np.sort(
        np.concatenate(
            [
                mpnn_supervised_train_pos,
                mpnn_early_stop_pos,
            ]
        )
    )


    if not np.array_equal(
        internal_union,
        inner_train_positions,
    ):
        raise RuntimeError(
            f"{task_id}：MPNN监督训练集"
            "和早停集没有完整且唯一地"
            "覆盖内层训练集。"
        )


    mpnn_supervised_train_manifest = (
        inner_train_manifest.iloc[
            mpnn_supervised_train_pos
        ]
        .reset_index(
            drop=True
        )
        .copy()
    )


    mpnn_early_stop_manifest = (
        inner_train_manifest.iloc[
            mpnn_early_stop_pos
        ]
        .reset_index(
            drop=True
        )
        .copy()
    )


    mpnn_supervised_train_global_idx = (
        mpnn_supervised_train_manifest[
            "original_feature_row_idx"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    mpnn_early_stop_global_idx = (
        mpnn_early_stop_manifest[
            "original_feature_row_idx"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    overlap_checks = {
        "mpnn_train_early_stop_overlap": int(
            np.intersect1d(
                mpnn_supervised_train_global_idx,
                mpnn_early_stop_global_idx,
            ).size
        ),

        "mpnn_train_inner_valid_overlap": int(
            np.intersect1d(
                mpnn_supervised_train_global_idx,
                inner_valid_global_idx,
            ).size
        ),

        "mpnn_early_stop_inner_valid_overlap": int(
            np.intersect1d(
                mpnn_early_stop_global_idx,
                inner_valid_global_idx,
            ).size
        ),

        "mpnn_train_outer_test_overlap": int(
            np.intersect1d(
                mpnn_supervised_train_global_idx,
                outer_test_idx_audit_only,
            ).size
        ),

        "mpnn_early_stop_outer_test_overlap": int(
            np.intersect1d(
                mpnn_early_stop_global_idx,
                outer_test_idx_audit_only,
            ).size
        ),
    }


    for (
        overlap_name,
        overlap_value,
    ) in overlap_checks.items():
        if overlap_value != 0:
            raise RuntimeError(
                f"{task_id}："
                f"{overlap_name}="
                f"{overlap_value}"
            )


    # --------------------------------------------------------
    # 4. 只用 MPNN 监督训练样本拟合标准化器
    # --------------------------------------------------------

    inner_train_tp_raw = (
        inner_train_manifest[
            [
                "T_rep",
                "P_rep",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )


    inner_valid_tp_raw = (
        inner_valid_structure_manifest[
            [
                "T_rep",
                "P_rep",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )


    inner_train_target_raw = (
        inner_train_manifest[
            "lg_eta"
        ]
        .to_numpy(
            dtype=float
        )
    )


    tp_scaler = StandardScaler()

    tp_scaler.fit(
        inner_train_tp_raw[
            mpnn_supervised_train_pos
        ]
    )


    target_scaler = StandardScaler()

    target_scaler.fit(
        inner_train_target_raw[
            mpnn_supervised_train_pos
        ].reshape(
            -1,
            1,
        )
    )


    if fitted_sample_count(
        tp_scaler
    ) != len(
        mpnn_supervised_train_pos
    ):
        raise RuntimeError(
            f"{task_id}：T/P标准化器"
            "拟合样本数错误。"
        )


    if fitted_sample_count(
        target_scaler
    ) != len(
        mpnn_supervised_train_pos
    ):
        raise RuntimeError(
            f"{task_id}：目标标准化器"
            "拟合样本数错误。"
        )


    # 对内层训练集和内层验证集的T/P只做transform。
    inner_train_tp_scaled = (
        tp_scaler.transform(
            inner_train_tp_raw
        )
        .astype(
            np.float32
        )
    )


    inner_valid_tp_scaled = (
        tp_scaler.transform(
            inner_valid_tp_raw
        )
        .astype(
            np.float32
        )
    )


    # 只有内层训练集目标被转换。
    # 内层验证集不建立MPNN目标张量。
    inner_train_target_scaled = (
        target_scaler.transform(
            inner_train_target_raw.reshape(
                -1,
                1,
            )
        )
        .reshape(-1)
        .astype(
            np.float32
        )
    )


    if not np.isfinite(
        inner_train_tp_scaled
    ).all():
        raise RuntimeError(
            f"{task_id}：标准化后的"
            "内层训练T/P存在NaN或无穷值。"
        )


    if not np.isfinite(
        inner_valid_tp_scaled
    ).all():
        raise RuntimeError(
            f"{task_id}：标准化后的"
            "内层验证T/P存在NaN或无穷值。"
        )


    if not np.isfinite(
        inner_train_target_scaled
    ).all():
        raise RuntimeError(
            f"{task_id}：标准化后的"
            "内层训练目标存在NaN或无穷值。"
        )


    # --------------------------------------------------------
    # 5. 图索引
    # --------------------------------------------------------

    inner_train_graph_idx = (
        inner_train_manifest[
            "graph_list_idx"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    inner_valid_graph_idx = (
        inner_valid_structure_manifest[
            "graph_list_idx"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    for index_name, index_array in {
        "inner_train_graph_idx": (
            inner_train_graph_idx
        ),
        "inner_valid_graph_idx": (
            inner_valid_graph_idx
        ),
    }.items():
        if (
            index_array.min() < 0
            or index_array.max()
            >= len(
                all_cation_graphs
            )
        ):
            raise IndexError(
                f"{task_id}：{index_name}"
                "超出图对象范围。"
            )


    # --------------------------------------------------------
    # 6. 保存第一套任务的预处理结果
    # --------------------------------------------------------

    output_dir = Path(
        task_context[
            "output_dir"
        ]
    )


    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    path_map = {
        "inner_train_manifest": (
            output_dir
            / "inner_train_manifest.csv"
        ),

        "inner_valid_structure_manifest": (
            output_dir
            / (
                "inner_valid_structure_manifest_"
                "no_targets.csv"
            )
        ),

        "mpnn_supervised_train_manifest": (
            output_dir
            / (
                "mpnn_supervised_train_"
                "manifest.csv"
            )
        ),

        "mpnn_early_stop_manifest": (
            output_dir
            / "mpnn_early_stop_manifest.csv"
        ),

        "internal_indices": (
            output_dir
            / "mpnn_internal_indices.npz"
        ),

        "tp_scaler": (
            output_dir
            / "tp_scaler.joblib"
        ),

        "target_scaler": (
            output_dir
            / "target_scaler.joblib"
        ),

        "preprocessing_audit": (
            output_dir
            / "mpnn_preprocessing_audit.json"
        ),
    }


    if save_outputs:
        inner_train_manifest.to_csv(
            path_map[
                "inner_train_manifest"
            ],
            index=False,
            encoding="utf-8-sig",
        )


        inner_valid_structure_manifest.to_csv(
            path_map[
                "inner_valid_structure_manifest"
            ],
            index=False,
            encoding="utf-8-sig",
        )


        mpnn_supervised_train_manifest.to_csv(
            path_map[
                "mpnn_supervised_train_manifest"
            ],
            index=False,
            encoding="utf-8-sig",
        )


        mpnn_early_stop_manifest.to_csv(
            path_map[
                "mpnn_early_stop_manifest"
            ],
            index=False,
            encoding="utf-8-sig",
        )


        np.savez_compressed(
            path_map[
                "internal_indices"
            ],

            inner_train_relative_to_outer_train_idx=(
                inner_train_relative_idx
            ),

            inner_valid_relative_to_outer_train_idx=(
                inner_valid_relative_idx
            ),

            mpnn_supervised_train_relative_to_inner_train_idx=(
                mpnn_supervised_train_pos
            ),

            mpnn_early_stop_relative_to_inner_train_idx=(
                mpnn_early_stop_pos
            ),

            inner_train_global_idx=(
                inner_train_global_idx
            ),

            inner_valid_global_idx=(
                inner_valid_global_idx
            ),

            mpnn_supervised_train_global_idx=(
                mpnn_supervised_train_global_idx
            ),

            mpnn_early_stop_global_idx=(
                mpnn_early_stop_global_idx
            ),
        )


        joblib.dump(
            tp_scaler,
            path_map[
                "tp_scaler"
            ],
        )


        joblib.dump(
            target_scaler,
            path_map[
                "target_scaler"
            ],
        )


    preprocessing_audit = {
        "created_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),

        "run_code_version": (
            RUN_CODE_VERSION
        ),

        "status": (
            "preprocessing_passed"
        ),

        "task_id": task_id,

        "split_name": split_name,

        "outer_fold": int(
            outer_fold
        ),

        "inner_fold": int(
            inner_fold
        ),

        "task_seed": int(
            task_seed
        ),

        "internal_split_rule": (
            internal_split_rule
        ),

        "internal_group_column": (
            internal_group_column
        ),

        "counts": {
            "total_model_rows": int(
                len(
                    graph_index_rows
                )
            ),

            "outer_train_rows": int(
                len(
                    outer_train_manifest
                )
            ),

            "outer_test_rows": int(
                len(
                    outer_test_idx_audit_only
                )
            ),

            "inner_ml_train_rows": int(
                len(
                    inner_train_manifest
                )
            ),

            "inner_ml_valid_rows": int(
                len(
                    inner_valid_manifest
                )
            ),

            "mpnn_supervised_train_rows": int(
                len(
                    mpnn_supervised_train_pos
                )
            ),

            "mpnn_early_stop_rows": int(
                len(
                    mpnn_early_stop_pos
                )
            ),

            "mpnn_supervised_train_unique_IL": int(
                mpnn_supervised_train_manifest[
                    "IL_key"
                ].nunique()
            ),

            "mpnn_early_stop_unique_IL": int(
                mpnn_early_stop_manifest[
                    "IL_key"
                ].nunique()
            ),
        },

        "overlap_checks": {
            **overlap_checks,

            "internal_group_overlap_count": (
                None
                if np.isnan(
                    internal_group_overlap_count
                )
                else int(
                    internal_group_overlap_count
                )
            ),
        },

        "standardization": {
            "tp_scaler_fitted_rows": (
                fitted_sample_count(
                    tp_scaler
                )
            ),

            "target_scaler_fitted_rows": (
                fitted_sample_count(
                    target_scaler
                )
            ),

            "tp_scaler_fitted_only_on_"
            "mpnn_supervised_train": True,

            "target_scaler_fitted_only_on_"
            "mpnn_supervised_train": True,
        },

        "leakage_control": {
            "inner_validation_target_present_"
            "in_source_outer_train_manifest": True,

            "inner_validation_target_used_for_"
            "mpnn_training": False,

            "inner_validation_target_used_for_"
            "mpnn_early_stopping": False,

            "inner_validation_used_for_"
            "tp_scaler_fit": False,

            "inner_validation_used_for_"
            "target_scaler_fit": False,

            "inner_validation_mpnn_target_"
            "tensor_created": False,

            "outer_test_manifest_read": False,

            "outer_test_target_read": False,

            "outer_test_indices_used_only_for": (
                "overlap audit"
            ),

            "existing_outer_mpnn_features_read": (
                False
            ),

            "old_global_mpnn_feature_read": False,

            "external_10point_holdout_read": False,
        },

        "outputs": {
            key: str(
                value
            )
            for key, value in (
                path_map.items()
            )
        },
    }


    if save_outputs:
        with open(
            path_map[
                "preprocessing_audit"
            ],
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                to_jsonable(
                    preprocessing_audit
                ),
                file,
                ensure_ascii=False,
                indent=2,
            )


    return {
        "task_id": task_id,
        "task_seed": task_seed,
        "split_name": split_name,
        "outer_fold": outer_fold,
        "inner_fold": inner_fold,

        "inner_train_manifest": (
            inner_train_manifest
        ),

        "inner_valid_structure_manifest": (
            inner_valid_structure_manifest
        ),

        "mpnn_supervised_train_manifest": (
            mpnn_supervised_train_manifest
        ),

        "mpnn_early_stop_manifest": (
            mpnn_early_stop_manifest
        ),

        "inner_train_graph_idx": (
            inner_train_graph_idx
        ),

        "inner_valid_graph_idx": (
            inner_valid_graph_idx
        ),

        "mpnn_supervised_train_pos": (
            mpnn_supervised_train_pos
        ),

        "mpnn_early_stop_pos": (
            mpnn_early_stop_pos
        ),

        "inner_train_tp_scaled": (
            inner_train_tp_scaled
        ),

        "inner_valid_tp_scaled": (
            inner_valid_tp_scaled
        ),

        "inner_train_target_scaled": (
            inner_train_target_scaled
        ),

        "tp_scaler": tp_scaler,

        "target_scaler": target_scaler,

        "overlap_checks": (
            overlap_checks
        ),

        "path_map": path_map,

        "preprocessing_audit": (
            preprocessing_audit
        ),
    }


# ============================================================
# 7. 只建立第一套测试任务
# ============================================================

TEST_TASK_PREPROCESSING = (
    build_inner_mpnn_preprocessing_context(
        split_name=(
            TEST_SPLIT_NAME
        ),
        outer_fold=(
            TEST_OUTER_FOLD
        ),
        inner_fold=(
            TEST_INNER_FOLD
        ),
        save_outputs=True,
    )
)


TEST_MPNN_SUPERVISED_TRAIN_POS = (
    TEST_TASK_PREPROCESSING[
        "mpnn_supervised_train_pos"
    ]
)

TEST_MPNN_EARLY_STOP_POS = (
    TEST_TASK_PREPROCESSING[
        "mpnn_early_stop_pos"
    ]
)

TEST_INNER_TRAIN_GRAPH_IDX = (
    TEST_TASK_PREPROCESSING[
        "inner_train_graph_idx"
    ]
)

TEST_INNER_VALID_GRAPH_IDX = (
    TEST_TASK_PREPROCESSING[
        "inner_valid_graph_idx"
    ]
)

TEST_INNER_TRAIN_TP_SCALED = (
    TEST_TASK_PREPROCESSING[
        "inner_train_tp_scaled"
    ]
)

TEST_INNER_VALID_TP_SCALED = (
    TEST_TASK_PREPROCESSING[
        "inner_valid_tp_scaled"
    ]
)

TEST_INNER_TRAIN_TARGET_SCALED = (
    TEST_TASK_PREPROCESSING[
        "inner_train_target_scaled"
    ]
)


# ============================================================
# 8. 输出检查
# ============================================================

audit = TEST_TASK_PREPROCESSING[
    "preprocessing_audit"
]


print("=" * 100)
print("Cell 2 第一套任务MPNN预处理和数据边界检查通过 ✓")
print("=" * 100)

print(
    "task id                           =",
    audit[
        "task_id"
    ],
)

print(
    "internal split rule               =",
    audit[
        "internal_split_rule"
    ],
)

print(
    "inner ML train rows               =",
    audit[
        "counts"
    ][
        "inner_ml_train_rows"
    ],
)

print(
    "inner ML valid rows               =",
    audit[
        "counts"
    ][
        "inner_ml_valid_rows"
    ],
)

print(
    "MPNN supervised train rows        =",
    audit[
        "counts"
    ][
        "mpnn_supervised_train_rows"
    ],
)

print(
    "MPNN early-stop rows              =",
    audit[
        "counts"
    ][
        "mpnn_early_stop_rows"
    ],
)

print(
    "MPNN train/early-stop overlap     =",
    audit[
        "overlap_checks"
    ][
        "mpnn_train_early_stop_overlap"
    ],
)

print(
    "MPNN train/inner-valid overlap    =",
    audit[
        "overlap_checks"
    ][
        "mpnn_train_inner_valid_overlap"
    ],
)

print(
    "MPNN early-stop/inner-valid overlap =",
    audit[
        "overlap_checks"
    ][
        "mpnn_early_stop_inner_valid_overlap"
    ],
)

print(
    "MPNN train/outer-test overlap     =",
    audit[
        "overlap_checks"
    ][
        "mpnn_train_outer_test_overlap"
    ],
)

print(
    "MPNN early-stop/outer-test overlap =",
    audit[
        "overlap_checks"
    ][
        "mpnn_early_stop_outer_test_overlap"
    ],
)

print(
    "T/P scaler fitted rows            =",
    audit[
        "standardization"
    ][
        "tp_scaler_fitted_rows"
    ],
)

print(
    "target scaler fitted rows         =",
    audit[
        "standardization"
    ][
        "target_scaler_fitted_rows"
    ],
)

print(
    "inner valid target used for MPNN  =",
    audit[
        "leakage_control"
    ][
        "inner_validation_target_used_for_"
        "mpnn_training"
    ],
)

print(
    "inner valid used for early stop   =",
    audit[
        "leakage_control"
    ][
        "inner_validation_target_used_for_"
        "mpnn_early_stopping"
    ],
)

print(
    "inner valid MPNN target created   =",
    audit[
        "leakage_control"
    ][
        "inner_validation_mpnn_target_"
        "tensor_created"
    ],
)

print(
    "outer test target read            =",
    audit[
        "leakage_control"
    ][
        "outer_test_target_read"
    ],
)

print(
    "existing outer MPNN X read        =",
    audit[
        "leakage_control"
    ][
        "existing_outer_mpnn_features_read"
    ],
)

print(
    "\n保存文件："
)

for (
    output_name,
    output_path,
) in TEST_TASK_PREPROCESSING[
    "path_map"
].items():
    print(
        f"{output_name:33s} = "
        f"{output_path}"
    )

Cell 2 第一套任务MPNN预处理和数据边界检查通过 ✓
task id                           = Data_split_OuterFold1_InnerFold1
internal split rule               = train_test_split by data point
inner ML train rows               = 8648
inner ML valid rows               = 4324
MPNN supervised train rows        = 6918
MPNN early-stop rows              = 1730
MPNN train/early-stop overlap     = 0
MPNN train/inner-valid overlap    = 0
MPNN early-stop/inner-valid overlap = 0
MPNN train/outer-test overlap     = 0
MPNN early-stop/outer-test overlap = 0
T/P scaler fitted rows            = 6918
target scaler fitted rows         = 6918
inner valid target used for MPNN  = False
inner valid used for early stop   = False
inner valid MPNN target created   = False
outer test target read            = False
existing outer MPNN X read        = False

保存文件：
inner_train_manifest              = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\inner_foldwise_mpnn_

In [4]:
# Cell 3 | 为第一套任务建立无泄露MPNN数据集和DataLoader


class ILGraphTargetDataset(Dataset):
    """用于MPNN监督训练和早停的带目标数据集。"""

    def __init__(
        self,
        cation_graph_list,
        anion_graph_list,
        graph_indices,
        environment_array,
        target_array,
    ):
        graph_indices = np.asarray(
            graph_indices,
            dtype=np.int64,
        )

        environment_array = np.asarray(
            environment_array,
            dtype=np.float32,
        )

        target_array = np.asarray(
            target_array,
            dtype=np.float32,
        ).reshape(-1)


        if not (
            len(graph_indices)
            == len(environment_array)
            == len(target_array)
        ):
            raise ValueError(
                "图索引、环境变量和目标值数量不一致。"
            )


        if environment_array.ndim != 2:
            raise ValueError(
                "环境变量应为二维数组。"
            )


        if environment_array.shape[1] != 2:
            raise ValueError(
                "环境变量应包含T和P两列。"
            )


        if (
            graph_indices.min() < 0
            or graph_indices.max()
            >= len(cation_graph_list)
            or graph_indices.max()
            >= len(anion_graph_list)
        ):
            raise IndexError(
                "图索引超出阳、阴离子图列表范围。"
            )


        if not np.isfinite(
            environment_array
        ).all():
            raise ValueError(
                "环境变量中存在NaN或无穷值。"
            )


        if not np.isfinite(
            target_array
        ).all():
            raise ValueError(
                "监督目标中存在NaN或无穷值。"
            )


        self.cation_graph_list = (
            cation_graph_list
        )

        self.anion_graph_list = (
            anion_graph_list
        )

        self.graph_indices = (
            graph_indices
        )

        self.environment_tensor = (
            torch.as_tensor(
                environment_array,
                dtype=torch.float32,
            )
        )

        self.target_tensor = (
            torch.as_tensor(
                target_array,
                dtype=torch.float32,
            )
            .reshape(-1, 1)
        )


    def __len__(self):
        return len(
            self.graph_indices
        )


    def __getitem__(
        self,
        index,
    ):
        graph_index = int(
            self.graph_indices[
                index
            ]
        )

        return (
            self.cation_graph_list[
                graph_index
            ],
            self.anion_graph_list[
                graph_index
            ],
            self.environment_tensor[
                index
            ],
            self.target_tensor[
                index
            ],
        )


class ILGraphFeatureDataset(Dataset):
    """
    用于提取MPNN表征的无目标数据集。

    内层验证集只允许通过本类进入模型，
    不创建或返回任何目标张量。
    """

    def __init__(
        self,
        cation_graph_list,
        anion_graph_list,
        graph_indices,
        environment_array,
    ):
        graph_indices = np.asarray(
            graph_indices,
            dtype=np.int64,
        )

        environment_array = np.asarray(
            environment_array,
            dtype=np.float32,
        )


        if len(
            graph_indices
        ) != len(
            environment_array
        ):
            raise ValueError(
                "图索引与环境变量数量不一致。"
            )


        if (
            environment_array.ndim != 2
            or environment_array.shape[1] != 2
        ):
            raise ValueError(
                "环境变量应为包含T和P的二维数组。"
            )


        if (
            graph_indices.min() < 0
            or graph_indices.max()
            >= len(cation_graph_list)
            or graph_indices.max()
            >= len(anion_graph_list)
        ):
            raise IndexError(
                "特征提取图索引超出范围。"
            )


        if not np.isfinite(
            environment_array
        ).all():
            raise ValueError(
                "特征提取环境变量中存在NaN或无穷值。"
            )


        self.cation_graph_list = (
            cation_graph_list
        )

        self.anion_graph_list = (
            anion_graph_list
        )

        self.graph_indices = (
            graph_indices
        )

        self.environment_tensor = (
            torch.as_tensor(
                environment_array,
                dtype=torch.float32,
            )
        )


    def __len__(self):
        return len(
            self.graph_indices
        )


    def __getitem__(
        self,
        index,
    ):
        graph_index = int(
            self.graph_indices[
                index
            ]
        )

        return (
            self.cation_graph_list[
                graph_index
            ],
            self.anion_graph_list[
                graph_index
            ],
            self.environment_tensor[
                index
            ],
        )


def collate_with_target(
    batch,
):
    """合并带目标的监督训练或早停批次。"""
    cation_batch = Batch.from_data_list(
        [
            item[0]
            for item in batch
        ]
    )

    anion_batch = Batch.from_data_list(
        [
            item[1]
            for item in batch
        ]
    )

    environment_batch = torch.stack(
        [
            item[2]
            for item in batch
        ]
    )

    target_batch = torch.stack(
        [
            item[3]
            for item in batch
        ]
    )

    return (
        cation_batch,
        anion_batch,
        environment_batch,
        target_batch,
    )


def collate_without_target(
    batch,
):
    """合并无目标的MPNN特征提取批次。"""
    cation_batch = Batch.from_data_list(
        [
            item[0]
            for item in batch
        ]
    )

    anion_batch = Batch.from_data_list(
        [
            item[1]
            for item in batch
        ]
    )

    environment_batch = torch.stack(
        [
            item[2]
            for item in batch
        ]
    )

    return (
        cation_batch,
        anion_batch,
        environment_batch,
    )


def graph_contains_target(
    graph,
) -> bool:
    """检查PyG图对象中是否嵌入目标属性。"""
    possible_target_attributes = [
        "y",
        "target",
        "eta",
        "lg_eta",
    ]

    for attribute in (
        possible_target_attributes
    ):
        value = getattr(
            graph,
            attribute,
            None,
        )

        if value is not None:
            return True

    return False


def build_task_dataloaders(
    preprocessing_context: dict,
) -> dict:
    """为一套内层任务建立训练、早停和特征提取DataLoader。"""
    task_id = preprocessing_context[
        "task_id"
    ]

    task_seed = int(
        preprocessing_context[
            "task_seed"
        ]
    )


    inner_train_graph_idx = (
        preprocessing_context[
            "inner_train_graph_idx"
        ]
    )

    inner_valid_graph_idx = (
        preprocessing_context[
            "inner_valid_graph_idx"
        ]
    )


    inner_train_tp_scaled = (
        preprocessing_context[
            "inner_train_tp_scaled"
        ]
    )

    inner_valid_tp_scaled = (
        preprocessing_context[
            "inner_valid_tp_scaled"
        ]
    )


    inner_train_target_scaled = (
        preprocessing_context[
            "inner_train_target_scaled"
        ]
    )


    mpnn_supervised_train_pos = (
        preprocessing_context[
            "mpnn_supervised_train_pos"
        ]
    )

    mpnn_early_stop_pos = (
        preprocessing_context[
            "mpnn_early_stop_pos"
        ]
    )


    # --------------------------------------------------------
    # 1. 带目标数据集只包含内层训练集
    # --------------------------------------------------------

    inner_train_target_dataset = (
        ILGraphTargetDataset(
            cation_graph_list=(
                all_cation_graphs
            ),
            anion_graph_list=(
                all_anion_graphs
            ),
            graph_indices=(
                inner_train_graph_idx
            ),
            environment_array=(
                inner_train_tp_scaled
            ),
            target_array=(
                inner_train_target_scaled
            ),
        )
    )


    mpnn_supervised_train_dataset = (
        Subset(
            inner_train_target_dataset,
            mpnn_supervised_train_pos.tolist(),
        )
    )


    mpnn_early_stop_dataset = (
        Subset(
            inner_train_target_dataset,
            mpnn_early_stop_pos.tolist(),
        )
    )


    # --------------------------------------------------------
    # 2. 特征提取数据集均不包含目标
    # --------------------------------------------------------

    inner_train_feature_dataset = (
        ILGraphFeatureDataset(
            cation_graph_list=(
                all_cation_graphs
            ),
            anion_graph_list=(
                all_anion_graphs
            ),
            graph_indices=(
                inner_train_graph_idx
            ),
            environment_array=(
                inner_train_tp_scaled
            ),
        )
    )


    inner_valid_feature_dataset = (
        ILGraphFeatureDataset(
            cation_graph_list=(
                all_cation_graphs
            ),
            anion_graph_list=(
                all_anion_graphs
            ),
            graph_indices=(
                inner_valid_graph_idx
            ),
            environment_array=(
                inner_valid_tp_scaled
            ),
        )
    )


    if hasattr(
        inner_valid_feature_dataset,
        "target_tensor",
    ):
        raise RuntimeError(
            f"{task_id}：内层验证特征数据集"
            "意外包含目标张量。"
        )


    # --------------------------------------------------------
    # 3. 建立DataLoader
    # --------------------------------------------------------

    train_generator = (
        torch.Generator()
        .manual_seed(
            task_seed
        )
    )


    mpnn_train_loader = DataLoader(
        mpnn_supervised_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=(
            collate_with_target
        ),
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        generator=train_generator,
        drop_last=False,
    )


    # 只用于无随机顺序地检查训练集或计算训练指标。
    mpnn_train_eval_loader = DataLoader(
        mpnn_supervised_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=(
            collate_with_target
        ),
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )


    mpnn_early_stop_loader = DataLoader(
        mpnn_early_stop_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=(
            collate_with_target
        ),
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )


    inner_train_feature_loader = (
        DataLoader(
            inner_train_feature_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            collate_fn=(
                collate_without_target
            ),
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
            drop_last=False,
        )
    )


    inner_valid_feature_loader = (
        DataLoader(
            inner_valid_feature_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            collate_fn=(
                collate_without_target
            ),
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
            drop_last=False,
        )
    )


    # --------------------------------------------------------
    # 4. 强制数量检查
    # --------------------------------------------------------

    expected_counts = {
        "inner_train_target_dataset": len(
            inner_train_graph_idx
        ),

        "mpnn_supervised_train_dataset": len(
            mpnn_supervised_train_pos
        ),

        "mpnn_early_stop_dataset": len(
            mpnn_early_stop_pos
        ),

        "inner_train_feature_dataset": len(
            inner_train_graph_idx
        ),

        "inner_valid_feature_dataset": len(
            inner_valid_graph_idx
        ),
    }


    actual_counts = {
        "inner_train_target_dataset": len(
            inner_train_target_dataset
        ),

        "mpnn_supervised_train_dataset": len(
            mpnn_supervised_train_dataset
        ),

        "mpnn_early_stop_dataset": len(
            mpnn_early_stop_dataset
        ),

        "inner_train_feature_dataset": len(
            inner_train_feature_dataset
        ),

        "inner_valid_feature_dataset": len(
            inner_valid_feature_dataset
        ),
    }


    if actual_counts != expected_counts:
        raise RuntimeError(
            f"{task_id}：数据集行数检查失败。\n"
            f"实际={actual_counts}\n"
            f"预期={expected_counts}"
        )


    return {
        "inner_train_target_dataset": (
            inner_train_target_dataset
        ),

        "mpnn_supervised_train_dataset": (
            mpnn_supervised_train_dataset
        ),

        "mpnn_early_stop_dataset": (
            mpnn_early_stop_dataset
        ),

        "inner_train_feature_dataset": (
            inner_train_feature_dataset
        ),

        "inner_valid_feature_dataset": (
            inner_valid_feature_dataset
        ),

        "mpnn_train_loader": (
            mpnn_train_loader
        ),

        "mpnn_train_eval_loader": (
            mpnn_train_eval_loader
        ),

        "mpnn_early_stop_loader": (
            mpnn_early_stop_loader
        ),

        "inner_train_feature_loader": (
            inner_train_feature_loader
        ),

        "inner_valid_feature_loader": (
            inner_valid_feature_loader
        ),

        "counts": actual_counts,
    }


# ============================================================
# 5. 建立第一套测试任务的DataLoader
# ============================================================

TEST_TASK_DATALOADERS = (
    build_task_dataloaders(
        preprocessing_context=(
            TEST_TASK_PREPROCESSING
        )
    )
)


TEST_MPNN_TRAIN_LOADER = (
    TEST_TASK_DATALOADERS[
        "mpnn_train_loader"
    ]
)

TEST_MPNN_TRAIN_EVAL_LOADER = (
    TEST_TASK_DATALOADERS[
        "mpnn_train_eval_loader"
    ]
)

TEST_MPNN_EARLY_STOP_LOADER = (
    TEST_TASK_DATALOADERS[
        "mpnn_early_stop_loader"
    ]
)

TEST_INNER_TRAIN_FEATURE_LOADER = (
    TEST_TASK_DATALOADERS[
        "inner_train_feature_loader"
    ]
)

TEST_INNER_VALID_FEATURE_LOADER = (
    TEST_TASK_DATALOADERS[
        "inner_valid_feature_loader"
    ]
)


# ============================================================
# 6. 实际批次检查
# ============================================================

(
    first_cation_batch,
    first_anion_batch,
    first_environment_batch,
    first_target_batch,
) = next(
    iter(
        TEST_MPNN_TRAIN_EVAL_LOADER
    )
)


(
    first_valid_cation_batch,
    first_valid_anion_batch,
    first_valid_environment_batch,
) = next(
    iter(
        TEST_INNER_VALID_FEATURE_LOADER
    )
)


if first_target_batch.ndim != 2:
    raise RuntimeError(
        "监督训练目标批次维度错误。"
    )


if (
    first_environment_batch.shape[1]
    != 2
    or first_valid_environment_batch.shape[1]
    != 2
):
    raise RuntimeError(
        "T/P批次维度错误。"
    )


if len(
    (
        first_valid_cation_batch,
        first_valid_anion_batch,
        first_valid_environment_batch,
    )
) != 3:
    raise RuntimeError(
        "内层验证特征批次结构异常。"
    )


# 检查所有公共图对象是否携带目标属性。
embedded_target_graph_count = 0


for graph_list in [
    all_cation_graphs,
    all_anion_graphs,
]:
    embedded_target_graph_count += sum(
        int(
            graph_contains_target(
                graph
            )
        )
        for graph in graph_list
    )


if embedded_target_graph_count != 0:
    raise RuntimeError(
        "分子图对象中发现目标属性，"
        f"数量={embedded_target_graph_count}"
    )


# ============================================================
# 7. 保存DataLoader审计
# ============================================================

TEST_DATALOADER_AUDIT_PATH = (
    Path(
        TEST_TASK_PREPROCESSING[
            "path_map"
        ][
            "preprocessing_audit"
        ]
    ).parent
    / "mpnn_dataloader_audit.json"
)


dataloader_audit = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "status": (
        "dataloader_passed"
    ),

    "task_id": (
        TEST_TASK_PREPROCESSING[
            "task_id"
        ]
    ),

    "counts": (
        TEST_TASK_DATALOADERS[
            "counts"
        ]
    ),

    "loader_batches": {
        "mpnn_train_batches": int(
            len(
                TEST_MPNN_TRAIN_LOADER
            )
        ),

        "mpnn_train_eval_batches": int(
            len(
                TEST_MPNN_TRAIN_EVAL_LOADER
            )
        ),

        "mpnn_early_stop_batches": int(
            len(
                TEST_MPNN_EARLY_STOP_LOADER
            )
        ),

        "inner_train_feature_batches": int(
            len(
                TEST_INNER_TRAIN_FEATURE_LOADER
            )
        ),

        "inner_valid_feature_batches": int(
            len(
                TEST_INNER_VALID_FEATURE_LOADER
            )
        ),
    },

    "first_supervised_batch": {
        "graph_count": int(
            first_environment_batch.shape[0]
        ),

        "environment_shape": list(
            first_environment_batch.shape
        ),

        "target_shape": list(
            first_target_batch.shape
        ),

        "cation_node_feature_dim": int(
            first_cation_batch.x.shape[1]
        ),

        "anion_node_feature_dim": int(
            first_anion_batch.x.shape[1]
        ),

        "cation_edge_feature_dim": int(
            first_cation_batch.edge_attr.shape[1]
        ),

        "anion_edge_feature_dim": int(
            first_anion_batch.edge_attr.shape[1]
        ),
    },

    "first_inner_valid_feature_batch": {
        "graph_count": int(
            first_valid_environment_batch.shape[0]
        ),

        "environment_shape": list(
            first_valid_environment_batch.shape
        ),

        "returned_items": 3,

        "target_returned": False,
    },

    "leakage_control": {
        "graph_objects_with_embedded_targets": int(
            embedded_target_graph_count
        ),

        "inner_valid_dataset_has_target_tensor": bool(
            hasattr(
                TEST_TASK_DATALOADERS[
                    "inner_valid_feature_dataset"
                ],
                "target_tensor",
            )
        ),

        "inner_valid_loader_returns_target": False,

        "outer_test_dataset_created": False,

        "outer_test_loader_created": False,

        "outer_test_target_tensor_created": False,
    },

    "loader_configuration": {
        "batch_size": int(
            BATCH_SIZE
        ),

        "num_workers": int(
            NUM_WORKERS
        ),

        "pin_memory": bool(
            PIN_MEMORY
        ),

        "training_shuffle": True,

        "evaluation_shuffle": False,
    },
}


with open(
    TEST_DATALOADER_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        to_jsonable(
            dataloader_audit
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 8. 输出检查
# ============================================================

print("=" * 100)
print("Cell 3 第一套任务MPNN数据集和DataLoader检查通过 ✓")
print("=" * 100)

print(
    "task id                         =",
    dataloader_audit[
        "task_id"
    ],
)

print(
    "inner train target dataset     =",
    dataloader_audit[
        "counts"
    ][
        "inner_train_target_dataset"
    ],
)

print(
    "MPNN supervised train dataset =",
    dataloader_audit[
        "counts"
    ][
        "mpnn_supervised_train_dataset"
    ],
)

print(
    "MPNN early-stop dataset       =",
    dataloader_audit[
        "counts"
    ][
        "mpnn_early_stop_dataset"
    ],
)

print(
    "inner train feature dataset   =",
    dataloader_audit[
        "counts"
    ][
        "inner_train_feature_dataset"
    ],
)

print(
    "inner valid feature dataset   =",
    dataloader_audit[
        "counts"
    ][
        "inner_valid_feature_dataset"
    ],
)

print(
    "MPNN train batches            =",
    dataloader_audit[
        "loader_batches"
    ][
        "mpnn_train_batches"
    ],
)

print(
    "MPNN early-stop batches       =",
    dataloader_audit[
        "loader_batches"
    ][
        "mpnn_early_stop_batches"
    ],
)

print(
    "inner valid feature batches   =",
    dataloader_audit[
        "loader_batches"
    ][
        "inner_valid_feature_batches"
    ],
)

print(
    "first environment shape       =",
    dataloader_audit[
        "first_supervised_batch"
    ][
        "environment_shape"
    ],
)

print(
    "first target shape            =",
    dataloader_audit[
        "first_supervised_batch"
    ][
        "target_shape"
    ],
)

print(
    "cation node feature dim       =",
    dataloader_audit[
        "first_supervised_batch"
    ][
        "cation_node_feature_dim"
    ],
)

print(
    "anion node feature dim        =",
    dataloader_audit[
        "first_supervised_batch"
    ][
        "anion_node_feature_dim"
    ],
)

print(
    "cation edge feature dim       =",
    dataloader_audit[
        "first_supervised_batch"
    ][
        "cation_edge_feature_dim"
    ],
)

print(
    "anion edge feature dim        =",
    dataloader_audit[
        "first_supervised_batch"
    ][
        "anion_edge_feature_dim"
    ],
)

print(
    "graph objects with targets    =",
    dataloader_audit[
        "leakage_control"
    ][
        "graph_objects_with_embedded_targets"
    ],
)

print(
    "inner valid has target tensor =",
    dataloader_audit[
        "leakage_control"
    ][
        "inner_valid_dataset_has_target_tensor"
    ],
)

print(
    "inner valid loader target     =",
    dataloader_audit[
        "leakage_control"
    ][
        "inner_valid_loader_returns_target"
    ],
)

print(
    "outer test dataset created    =",
    dataloader_audit[
        "leakage_control"
    ][
        "outer_test_dataset_created"
    ],
)

print(
    "outer test loader created     =",
    dataloader_audit[
        "leakage_control"
    ][
        "outer_test_loader_created"
    ],
)

print(
    "\nDataLoader审计文件："
)

print(
    TEST_DATALOADER_AUDIT_PATH
)

Cell 3 第一套任务MPNN数据集和DataLoader检查通过 ✓
task id                         = Data_split_OuterFold1_InnerFold1
inner train target dataset     = 8648
MPNN supervised train dataset = 6918
MPNN early-stop dataset       = 1730
inner train feature dataset   = 8648
inner valid feature dataset   = 4324
MPNN train batches            = 55
MPNN early-stop batches       = 14
inner valid feature batches   = 34
first environment shape       = [128, 2]
first target shape            = [128, 1]
cation node feature dim       = 7
anion node feature dim        = 7
cation edge feature dim       = 4
anion edge feature dim        = 4
graph objects with targets    = 0
inner valid has target tensor = False
inner valid loader target     = False
outer test dataset created    = False
outer test loader created     = False

DataLoader审计文件：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\inner_foldwise_mpnn_tuning_features\Data_split\OuterFold_1\Inne

In [5]:
# Cell 4 | 定义与外层流程一致的双分支MPNN并检查前向传播


class MPNNEncoder(nn.Module):
    """将单个离子分子图编码为64维图表征。"""

    def __init__(
        self,
        node_dim: int = 7,
        edge_dim: int = 4,
        hidden_dim: int = 64,
    ):
        super().__init__()

        self.node_dim = int(
            node_dim
        )

        self.edge_dim = int(
            edge_dim
        )

        self.hidden_dim = int(
            hidden_dim
        )


        self.lin0 = nn.Linear(
            self.node_dim,
            self.hidden_dim,
        )


        edge_network = nn.Sequential(
            nn.Linear(
                self.edge_dim,
                32,
            ),
            nn.ReLU(),
            nn.Linear(
                32,
                self.hidden_dim
                * self.hidden_dim,
            ),
        )


        self.conv = NNConv(
            in_channels=(
                self.hidden_dim
            ),
            out_channels=(
                self.hidden_dim
            ),
            nn=edge_network,
            aggr="mean",
        )


        self.lin1 = nn.Linear(
            self.hidden_dim,
            self.hidden_dim,
        )


    def forward(
        self,
        graph_batch,
    ):
        node_state = self.lin0(
            graph_batch.x.float()
        )


        node_state = torch.relu(
            self.conv(
                node_state,
                graph_batch.edge_index,
                graph_batch.edge_attr.float(),
            )
        )


        node_state = self.lin1(
            node_state
        )


        graph_embedding = (
            global_mean_pool(
                node_state,
                graph_batch.batch,
            )
        )


        return graph_embedding


class ILPredictor(nn.Module):
    """
    双分支MPNN监督训练模型。

    阳离子图表征64维，阴离子图表征64维，
    与标准化T/P两维拼接形成130维表征。
    """

    def __init__(
        self,
    ):
        super().__init__()


        self.cation_encoder = (
            MPNNEncoder(
                node_dim=7,
                edge_dim=4,
                hidden_dim=64,
            )
        )


        self.anion_encoder = (
            MPNNEncoder(
                node_dim=7,
                edge_dim=4,
                hidden_dim=64,
            )
        )


        self.regression_head = (
            nn.Sequential(
                nn.Linear(
                    130,
                    64,
                ),
                nn.ReLU(),
                nn.Linear(
                    64,
                    32,
                ),
                nn.ReLU(),
                nn.Linear(
                    32,
                    1,
                ),
            )
        )


    def extract_features(
        self,
        cation_batch,
        anion_batch,
        environment_batch,
    ):
        """提取130维表征，不经过回归头。"""

        cation_embedding = (
            self.cation_encoder(
                cation_batch
            )
        )


        anion_embedding = (
            self.anion_encoder(
                anion_batch
            )
        )


        features = torch.cat(
            [
                cation_embedding,
                anion_embedding,
                environment_batch,
            ],
            dim=1,
        )


        return features


    def forward(
        self,
        cation_batch,
        anion_batch,
        environment_batch,
    ):
        features = (
            self.extract_features(
                cation_batch,
                anion_batch,
                environment_batch,
            )
        )


        prediction = (
            self.regression_head(
                features
            )
        )


        return prediction


def count_model_parameters(
    model,
) -> dict:
    """统计模型总参数量与可训练参数量。"""

    total_parameters = int(
        sum(
            parameter.numel()
            for parameter
            in model.parameters()
        )
    )


    trainable_parameters = int(
        sum(
            parameter.numel()
            for parameter
            in model.parameters()
            if parameter.requires_grad
        )
    )


    return {
        "total_parameters": (
            total_parameters
        ),
        "trainable_parameters": (
            trainable_parameters
        ),
    }


def build_fresh_mpnn_model(
    task_seed: int,
):
    """按照任务随机种子重新初始化一个全新MPNN。"""

    seed_everything(
        int(
            task_seed
        )
    )


    new_model = (
        ILPredictor()
        .to(
            DEVICE
        )
    )


    return new_model


# ============================================================
# 1. 初始化第一套任务模型
# ============================================================

TEST_MPNN_MODEL = (
    build_fresh_mpnn_model(
        task_seed=(
            TEST_TASK_PREPROCESSING[
                "task_seed"
            ]
        )
    )
)


parameter_counts = (
    count_model_parameters(
        TEST_MPNN_MODEL
    )
)


EXPECTED_TOTAL_PARAMETERS = (
    298817
)


if (
    parameter_counts[
        "total_parameters"
    ]
    != EXPECTED_TOTAL_PARAMETERS
):
    raise RuntimeError(
        "当前内层MPNN参数量"
        "与已完成外层MPNN结构不一致：\n"
        f"实际="
        f"{parameter_counts['total_parameters']}\n"
        f"预期="
        f"{EXPECTED_TOTAL_PARAMETERS}"
    )


if (
    parameter_counts[
        "trainable_parameters"
    ]
    != EXPECTED_TOTAL_PARAMETERS
):
    raise RuntimeError(
        "部分MPNN参数没有参与训练。"
    )


# ============================================================
# 2. 使用监督训练批次检查完整前向传播
# ============================================================

(
    supervised_cation_batch,
    supervised_anion_batch,
    supervised_environment_batch,
    supervised_target_batch,
) = next(
    iter(
        TEST_MPNN_TRAIN_EVAL_LOADER
    )
)


supervised_cation_batch = (
    supervised_cation_batch.to(
        DEVICE
    )
)

supervised_anion_batch = (
    supervised_anion_batch.to(
        DEVICE
    )
)

supervised_environment_batch = (
    supervised_environment_batch.to(
        DEVICE
    )
)

supervised_target_batch = (
    supervised_target_batch.to(
        DEVICE
    )
)


TEST_MPNN_MODEL.eval()


with torch.no_grad():
    supervised_features = (
        TEST_MPNN_MODEL.extract_features(
            supervised_cation_batch,
            supervised_anion_batch,
            supervised_environment_batch,
        )
    )


    supervised_prediction = (
        TEST_MPNN_MODEL(
            supervised_cation_batch,
            supervised_anion_batch,
            supervised_environment_batch,
        )
    )


expected_supervised_rows = int(
    supervised_environment_batch.shape[
        0
    ]
)


if tuple(
    supervised_features.shape
) != (
    expected_supervised_rows,
    130,
):
    raise RuntimeError(
        "监督训练批次MPNN表征维度错误："
        f"{tuple(supervised_features.shape)}"
    )


if tuple(
    supervised_prediction.shape
) != (
    expected_supervised_rows,
    1,
):
    raise RuntimeError(
        "监督训练批次回归输出维度错误："
        f"{tuple(supervised_prediction.shape)}"
    )


if tuple(
    supervised_target_batch.shape
) != (
    expected_supervised_rows,
    1,
):
    raise RuntimeError(
        "监督训练目标批次维度错误。"
    )


if not torch.isfinite(
    supervised_features
).all():
    raise RuntimeError(
        "监督训练批次MPNN表征"
        "中存在NaN或无穷值。"
    )


if not torch.isfinite(
    supervised_prediction
).all():
    raise RuntimeError(
        "监督训练批次预测输出"
        "中存在NaN或无穷值。"
    )


# ============================================================
# 3. 使用无目标内层验证批次检查特征提取
# ============================================================

(
    valid_cation_batch,
    valid_anion_batch,
    valid_environment_batch,
) = next(
    iter(
        TEST_INNER_VALID_FEATURE_LOADER
    )
)


valid_cation_batch = (
    valid_cation_batch.to(
        DEVICE
    )
)

valid_anion_batch = (
    valid_anion_batch.to(
        DEVICE
    )
)

valid_environment_batch = (
    valid_environment_batch.to(
        DEVICE
    )
)


with torch.no_grad():
    valid_features = (
        TEST_MPNN_MODEL.extract_features(
            valid_cation_batch,
            valid_anion_batch,
            valid_environment_batch,
        )
    )


expected_valid_rows = int(
    valid_environment_batch.shape[
        0
    ]
)


if tuple(
    valid_features.shape
) != (
    expected_valid_rows,
    130,
):
    raise RuntimeError(
        "内层验证批次MPNN表征维度错误："
        f"{tuple(valid_features.shape)}"
    )


if not torch.isfinite(
    valid_features
).all():
    raise RuntimeError(
        "内层验证批次MPNN表征"
        "中存在NaN或无穷值。"
    )


if hasattr(
    TEST_TASK_DATALOADERS[
        "inner_valid_feature_dataset"
    ],
    "target_tensor",
):
    raise RuntimeError(
        "内层验证特征数据集"
        "意外包含目标张量。"
    )


# ============================================================
# 4. 核对模型结构
# ============================================================

architecture_checks = {
    "cation_embedding_dim": int(
        supervised_features.shape[1]
        - 64
        - 2
    ),

    "anion_embedding_dim": 64,

    "environment_dim": int(
        supervised_environment_batch.shape[
            1
        ]
    ),

    "descriptor_dim": int(
        supervised_features.shape[
            1
        ]
    ),

    "prediction_dim": int(
        supervised_prediction.shape[
            1
        ]
    ),

    "total_parameters": (
        parameter_counts[
            "total_parameters"
        ]
    ),

    "trainable_parameters": (
        parameter_counts[
            "trainable_parameters"
        ]
    ),
}


expected_architecture = {
    "cation_embedding_dim": 64,
    "anion_embedding_dim": 64,
    "environment_dim": 2,
    "descriptor_dim": 130,
    "prediction_dim": 1,
    "total_parameters": 298817,
    "trainable_parameters": 298817,
}


if (
    architecture_checks
    != expected_architecture
):
    raise RuntimeError(
        "MPNN结构检查不一致：\n"
        f"实际={architecture_checks}\n"
        f"预期={expected_architecture}"
    )


# ============================================================
# 5. 保存模型结构审计
# ============================================================

TEST_MODEL_STRUCTURE_AUDIT_PATH = (
    Path(
        TEST_TASK_PREPROCESSING[
            "path_map"
        ][
            "preprocessing_audit"
        ]
    ).parent
    / "mpnn_model_structure_audit.json"
)


model_structure_audit = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "status": (
        "model_structure_passed"
    ),

    "task_id": (
        TEST_TASK_PREPROCESSING[
            "task_id"
        ]
    ),

    "task_seed": int(
        TEST_TASK_PREPROCESSING[
            "task_seed"
        ]
    ),

    "device": str(
        DEVICE
    ),

    "architecture": {
        "encoder_class": (
            "MPNNEncoder"
        ),

        "predictor_class": (
            "ILPredictor"
        ),

        "message_passing_layer": (
            "NNConv"
        ),

        "aggregation": "mean",

        "graph_pooling": (
            "global_mean_pool"
        ),

        "node_feature_dim": 7,

        "edge_feature_dim": 4,

        **architecture_checks,
    },

    "supervised_forward_check": {
        "batch_rows": (
            expected_supervised_rows
        ),

        "feature_shape": list(
            supervised_features.shape
        ),

        "prediction_shape": list(
            supervised_prediction.shape
        ),

        "target_shape": list(
            supervised_target_batch.shape
        ),

        "features_finite": True,

        "predictions_finite": True,
    },

    "inner_validation_feature_check": {
        "batch_rows": (
            expected_valid_rows
        ),

        "feature_shape": list(
            valid_features.shape
        ),

        "target_tensor_present": False,

        "target_returned_by_loader": False,

        "features_finite": True,
    },

    "leakage_control": {
        "model_trained_in_this_cell": False,

        "optimizer_created": False,

        "loss_function_applied": False,

        "backward_called": False,

        "inner_validation_target_read": False,

        "inner_validation_target_tensor_created": (
            False
        ),

        "outer_test_dataset_created": False,

        "outer_test_loader_created": False,

        "outer_test_target_read": False,

        "existing_outer_mpnn_features_read": (
            False
        ),

        "old_global_mpnn_feature_read": False,

        "external_10point_holdout_read": False,
    },
}


with open(
    TEST_MODEL_STRUCTURE_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        to_jsonable(
            model_structure_audit
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 6. 输出检查
# ============================================================

print("=" * 100)
print("Cell 4 第一套任务双分支MPNN结构和前向传播检查通过 ✓")
print("=" * 100)

print(
    "task id                    =",
    model_structure_audit[
        "task_id"
    ],
)

print(
    "device                     =",
    model_structure_audit[
        "device"
    ],
)

print(
    "cation embedding dim       =",
    architecture_checks[
        "cation_embedding_dim"
    ],
)

print(
    "anion embedding dim        =",
    architecture_checks[
        "anion_embedding_dim"
    ],
)

print(
    "T/P feature dim            =",
    architecture_checks[
        "environment_dim"
    ],
)

print(
    "descriptor dim             =",
    architecture_checks[
        "descriptor_dim"
    ],
)

print(
    "total parameters           =",
    architecture_checks[
        "total_parameters"
    ],
)

print(
    "trainable parameters       =",
    architecture_checks[
        "trainable_parameters"
    ],
)

print(
    "supervised feature shape   =",
    tuple(
        supervised_features.shape
    ),
)

print(
    "supervised prediction shape =",
    tuple(
        supervised_prediction.shape
    ),
)

print(
    "inner valid feature shape  =",
    tuple(
        valid_features.shape
    ),
)

print(
    "inner valid target tensor  =",
    model_structure_audit[
        "inner_validation_feature_check"
    ][
        "target_tensor_present"
    ],
)

print(
    "model trained in Cell 4    =",
    model_structure_audit[
        "leakage_control"
    ][
        "model_trained_in_this_cell"
    ],
)

print(
    "backward called            =",
    model_structure_audit[
        "leakage_control"
    ][
        "backward_called"
    ],
)

print(
    "outer test target read     =",
    model_structure_audit[
        "leakage_control"
    ][
        "outer_test_target_read"
    ],
)

print(
    "\n模型结构审计文件："
)

print(
    TEST_MODEL_STRUCTURE_AUDIT_PATH
)

Cell 4 第一套任务双分支MPNN结构和前向传播检查通过 ✓
task id                    = Data_split_OuterFold1_InnerFold1
device                     = cpu
cation embedding dim       = 64
anion embedding dim        = 64
T/P feature dim            = 2
descriptor dim             = 130
total parameters           = 298817
trainable parameters       = 298817
supervised feature shape   = (128, 130)
supervised prediction shape = (128, 1)
inner valid feature shape  = (128, 130)
inner valid target tensor  = False
model trained in Cell 4    = False
backward called            = False
outer test target read     = False

模型结构审计文件：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\inner_foldwise_mpnn_tuning_features\Data_split\OuterFold_1\InnerFold_1\mpnn_model_structure_audit.json


In [6]:
# Cell 5 | 训练第一套内层MPNN并保存断点、最佳模型和训练记录


def clone_state_dict_to_cpu(
    state_dict,
):
    """将模型状态复制到CPU，便于稳定保存和恢复。"""
    return {
        key: value
        .detach()
        .cpu()
        .clone()

        for key, value
        in state_dict.items()
    }


def move_optimizer_state_to_device(
    optimizer,
    device,
):
    """将恢复后的优化器状态移动到当前计算设备。"""
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(
                value
            ):
                state[key] = value.to(
                    device
                )


def save_json_atomic(
    data,
    output_path,
):
    """先写临时文件，再替换正式JSON，降低中断损坏风险。"""
    output_path = Path(
        output_path
    )

    temporary_path = (
        output_path.parent
        / (
            output_path.name
            + ".tmp"
        )
    )


    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            to_jsonable(
                data
            ),
            file,
            ensure_ascii=False,
            indent=2,
        )


    os.replace(
        temporary_path,
        output_path,
    )


def save_torch_atomic(
    data,
    output_path,
):
    """先保存临时文件，再替换正式PyTorch文件。"""
    output_path = Path(
        output_path
    )

    temporary_path = (
        output_path.parent
        / (
            output_path.name
            + ".tmp"
        )
    )


    torch.save(
        data,
        temporary_path,
    )


    os.replace(
        temporary_path,
        output_path,
    )


def train_one_inner_mpnn(
    preprocessing_context: dict,
    dataloader_context: dict,
    max_epochs: int = MAX_EPOCHS,
    early_stop_patience: int = (
        EARLY_STOP_PATIENCE
    ),
    min_delta: float = MIN_DELTA,
) -> dict:
    """
    训练一套内层专属MPNN。

    训练只使用：
    - MPNN监督训练集；
    - MPNN早停集。

    不接收内层ML验证Loader或外层测试Loader。
    """
    task_id = (
        preprocessing_context[
            "task_id"
        ]
    )

    task_seed = int(
        preprocessing_context[
            "task_seed"
        ]
    )


    output_dir = Path(
        preprocessing_context[
            "path_map"
        ][
            "preprocessing_audit"
        ]
    ).parent


    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    training_history_path = (
        output_dir
        / "mpnn_training_history.csv"
    )

    best_state_path = (
        output_dir
        / "mpnn_best_state.pt"
    )

    training_summary_path = (
        output_dir
        / "mpnn_training_summary.json"
    )

    checkpoint_path = (
        output_dir
        / "mpnn_training_checkpoint.pt"
    )

    progress_path = (
        output_dir
        / "mpnn_training_progress.json"
    )


    # --------------------------------------------------------
    # 1. 防止误覆盖已经完成的结果
    # --------------------------------------------------------

    if training_summary_path.exists():
        existing_summary = read_json(
            training_summary_path
        )

        if (
            existing_summary.get(
                "status"
            )
            == "training_completed"
        ):
            raise RuntimeError(
                f"{task_id} 已经完成训练。\n"
                "为避免覆盖现有最佳模型，"
                "Cell 5 已停止重复运行。\n"
                f"完成记录：{training_summary_path}"
            )


    # --------------------------------------------------------
    # 2. 只取得训练集和早停集
    # --------------------------------------------------------

    supervised_train_dataset = (
        dataloader_context[
            "mpnn_supervised_train_dataset"
        ]
    )

    early_stop_loader = (
        dataloader_context[
            "mpnn_early_stop_loader"
        ]
    )


    expected_train_rows = int(
        preprocessing_context[
            "preprocessing_audit"
        ][
            "counts"
        ][
            "mpnn_supervised_train_rows"
        ]
    )

    expected_early_stop_rows = int(
        preprocessing_context[
            "preprocessing_audit"
        ][
            "counts"
        ][
            "mpnn_early_stop_rows"
        ]
    )


    if len(
        supervised_train_dataset
    ) != expected_train_rows:
        raise RuntimeError(
            f"{task_id}：监督训练集数量异常。"
        )


    if len(
        early_stop_loader.dataset
    ) != expected_early_stop_rows:
        raise RuntimeError(
            f"{task_id}：早停集数量异常。"
        )


    if expected_train_rows != 6918:
        raise RuntimeError(
            f"{task_id}：第一套任务监督训练数据"
            f"应为6918条，实际为{expected_train_rows}条。"
        )


    if expected_early_stop_rows != 1730:
        raise RuntimeError(
            f"{task_id}：第一套任务早停数据"
            f"应为1730条，实际为{expected_early_stop_rows}条。"
        )


    # --------------------------------------------------------
    # 3. 初始化模型、优化器、调度器和损失函数
    # --------------------------------------------------------

    seed_everything(
        task_seed
    )


    model = build_fresh_mpnn_model(
        task_seed=task_seed
    )


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=INITIAL_LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )


    criterion = nn.MSELoss()


    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=8,
            min_lr=1e-7,
        )
    )


    start_epoch = 1

    best_epoch = 0

    best_val_loss = float(
        "inf"
    )

    best_state_dict = None

    no_improve_count = 0

    training_history = []

    cumulative_minutes_before = 0.0

    resumed_from_checkpoint = False

    resumed_after_epoch = 0


    # --------------------------------------------------------
    # 4. 检查并恢复断点
    # --------------------------------------------------------

    checkpoint = None


    if checkpoint_path.exists():
        checkpoint = load_torch_object(
            checkpoint_path
        )


        required_checkpoint_keys = {
            "task_id",
            "run_code_version",
            "epoch_completed",
            "model_state_dict",
            "optimizer_state_dict",
            "scheduler_state_dict",
            "best_epoch",
            "best_val_loss",
            "best_state_dict",
            "no_improve_count",
            "training_history",
            "train_generator_state",
            "cumulative_training_minutes",
        }


        missing_checkpoint_keys = (
            required_checkpoint_keys
            - set(
                checkpoint.keys()
            )
        )


        if missing_checkpoint_keys:
            raise RuntimeError(
                "训练断点不完整，缺少字段："
                f"{sorted(missing_checkpoint_keys)}"
            )


        if (
            checkpoint[
                "task_id"
            ]
            != task_id
        ):
            raise RuntimeError(
                "训练断点属于其他任务，"
                "已停止恢复。"
            )


        if (
            checkpoint[
                "run_code_version"
            ]
            != RUN_CODE_VERSION
        ):
            raise RuntimeError(
                "训练断点代码版本与当前版本不一致，"
                "已停止恢复。"
            )


        model.load_state_dict(
            checkpoint[
                "model_state_dict"
            ]
        )


        optimizer.load_state_dict(
            checkpoint[
                "optimizer_state_dict"
            ]
        )


        move_optimizer_state_to_device(
            optimizer=optimizer,
            device=DEVICE,
        )


        scheduler.load_state_dict(
            checkpoint[
                "scheduler_state_dict"
            ]
        )


        resumed_after_epoch = int(
            checkpoint[
                "epoch_completed"
            ]
        )


        start_epoch = (
            resumed_after_epoch
            + 1
        )


        best_epoch = int(
            checkpoint[
                "best_epoch"
            ]
        )


        best_val_loss = float(
            checkpoint[
                "best_val_loss"
            ]
        )


        best_state_dict = (
            checkpoint[
                "best_state_dict"
            ]
        )


        no_improve_count = int(
            checkpoint[
                "no_improve_count"
            ]
        )


        training_history = list(
            checkpoint[
                "training_history"
            ]
        )


        cumulative_minutes_before = float(
            checkpoint[
                "cumulative_training_minutes"
            ]
        )


        if (
            "python_random_state"
            in checkpoint
        ):
            random.setstate(
                checkpoint[
                    "python_random_state"
                ]
            )


        if (
            "numpy_random_state"
            in checkpoint
        ):
            np.random.set_state(
                checkpoint[
                    "numpy_random_state"
                ]
            )


        if (
            "torch_cpu_rng_state"
            in checkpoint
        ):
            torch.set_rng_state(
                checkpoint[
                    "torch_cpu_rng_state"
                ]
            )


        if (
            DEVICE.type == "cuda"
            and "torch_cuda_rng_state_all"
            in checkpoint
            and checkpoint[
                "torch_cuda_rng_state_all"
            ] is not None
        ):
            torch.cuda.set_rng_state_all(
                checkpoint[
                    "torch_cuda_rng_state_all"
                ]
            )


        resumed_from_checkpoint = True


    # --------------------------------------------------------
    # 5. 建立可恢复随机顺序的训练Loader
    # --------------------------------------------------------

    train_generator = (
        torch.Generator()
    )


    if resumed_from_checkpoint:
        train_generator.set_state(
            checkpoint[
                "train_generator_state"
            ]
        )

    else:
        train_generator.manual_seed(
            task_seed
        )


    train_loader = DataLoader(
        supervised_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=(
            collate_with_target
        ),
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        generator=train_generator,
        drop_last=False,
    )


    if start_epoch > max_epochs:
        raise RuntimeError(
            "断点中的已完成Epoch"
            "已经达到或超过当前最大Epoch。"
        )


    # --------------------------------------------------------
    # 6. 开始或恢复训练
    # --------------------------------------------------------

    session_start_time = (
        time.perf_counter()
    )


    stopped_early = False

    epochs_run_this_session = 0


    print("=" * 100)

    print(
        "开始第一套内层MPNN监督训练"
    )

    print("=" * 100)

    print(
        "task id                       =",
        task_id,
    )

    print(
        "device                        =",
        DEVICE,
    )

    print(
        "MPNN supervised train rows    =",
        expected_train_rows,
    )

    print(
        "MPNN early-stop rows          =",
        expected_early_stop_rows,
    )

    print(
        "inner ML valid rows used      =",
        0,
    )

    print(
        "outer test rows used          =",
        0,
    )

    print(
        "resume from checkpoint        =",
        resumed_from_checkpoint,
    )

    print(
        "resume after epoch            =",
        resumed_after_epoch,
    )

    print(
        "start epoch                   =",
        start_epoch,
    )

    print(
        "maximum epochs                =",
        max_epochs,
    )

    print(
        "early stopping patience       =",
        early_stop_patience,
    )


    for epoch in range(
        start_epoch,
        max_epochs + 1,
    ):
        epochs_run_this_session += 1


        # ====================================================
        # 训练阶段：仅6,918条监督训练数据
        # ====================================================

        model.train()

        train_loss_sum = 0.0

        train_sample_count = 0


        for (
            cation_batch,
            anion_batch,
            environment_batch,
            target_batch,
        ) in train_loader:

            cation_batch = (
                cation_batch.to(
                    DEVICE
                )
            )

            anion_batch = (
                anion_batch.to(
                    DEVICE
                )
            )

            environment_batch = (
                environment_batch.to(
                    DEVICE
                )
            )

            target_batch = (
                target_batch.to(
                    DEVICE
                )
            )


            if not torch.isfinite(
                target_batch
            ).all():
                raise RuntimeError(
                    f"Epoch {epoch}：监督训练目标"
                    "出现NaN或无穷值。"
                )


            optimizer.zero_grad(
                set_to_none=True
            )


            prediction = model(
                cation_batch,
                anion_batch,
                environment_batch,
            )


            loss = criterion(
                prediction,
                target_batch,
            )


            if not torch.isfinite(
                loss
            ):
                raise RuntimeError(
                    f"Epoch {epoch}：训练损失"
                    "出现NaN或无穷值。"
                )


            loss.backward()

            optimizer.step()


            current_batch_rows = int(
                target_batch.shape[0]
            )


            train_loss_sum += (
                float(
                    loss.item()
                )
                * current_batch_rows
            )


            train_sample_count += (
                current_batch_rows
            )


        if (
            train_sample_count
            != expected_train_rows
        ):
            raise RuntimeError(
                f"Epoch {epoch}：实际训练样本数"
                f"为{train_sample_count}，"
                f"预期为{expected_train_rows}。"
            )


        average_train_loss = (
            train_loss_sum
            / train_sample_count
        )


        # ====================================================
        # 早停阶段：仅1,730条内层训练子集
        # ====================================================

        model.eval()

        early_stop_loss_sum = 0.0

        early_stop_sample_count = 0


        with torch.no_grad():
            for (
                cation_batch,
                anion_batch,
                environment_batch,
                target_batch,
            ) in early_stop_loader:

                cation_batch = (
                    cation_batch.to(
                        DEVICE
                    )
                )

                anion_batch = (
                    anion_batch.to(
                        DEVICE
                    )
                )

                environment_batch = (
                    environment_batch.to(
                        DEVICE
                    )
                )

                target_batch = (
                    target_batch.to(
                        DEVICE
                    )
                )


                if not torch.isfinite(
                    target_batch
                ).all():
                    raise RuntimeError(
                        f"Epoch {epoch}：早停目标"
                        "出现NaN或无穷值。"
                    )


                prediction = model(
                    cation_batch,
                    anion_batch,
                    environment_batch,
                )


                early_stop_loss = (
                    criterion(
                        prediction,
                        target_batch,
                    )
                )


                if not torch.isfinite(
                    early_stop_loss
                ):
                    raise RuntimeError(
                        f"Epoch {epoch}：早停损失"
                        "出现NaN或无穷值。"
                    )


                current_batch_rows = int(
                    target_batch.shape[0]
                )


                early_stop_loss_sum += (
                    float(
                        early_stop_loss.item()
                    )
                    * current_batch_rows
                )


                early_stop_sample_count += (
                    current_batch_rows
                )


        if (
            early_stop_sample_count
            != expected_early_stop_rows
        ):
            raise RuntimeError(
                f"Epoch {epoch}：实际早停样本数"
                f"为{early_stop_sample_count}，"
                f"预期为{expected_early_stop_rows}。"
            )


        average_early_stop_loss = (
            early_stop_loss_sum
            / early_stop_sample_count
        )


        scheduler.step(
            average_early_stop_loss
        )


        current_learning_rate = float(
            optimizer.param_groups[
                0
            ][
                "lr"
            ]
        )


        improved = bool(
            average_early_stop_loss
            < best_val_loss
            - min_delta
        )


        if improved:
            best_val_loss = float(
                average_early_stop_loss
            )

            best_epoch = int(
                epoch
            )

            best_state_dict = (
                clone_state_dict_to_cpu(
                    model.state_dict()
                )
            )

            no_improve_count = 0


            best_state_payload = {
                "property": "viscosity",

                "target_col": "lg_eta",

                "representation_method": (
                    "inner-fold-specific "
                    "supervised MPNN"
                ),

                "task_id": task_id,

                "split_name": (
                    preprocessing_context[
                        "split_name"
                    ]
                ),

                "outer_fold": int(
                    preprocessing_context[
                        "outer_fold"
                    ]
                ),

                "inner_fold": int(
                    preprocessing_context[
                        "inner_fold"
                    ]
                ),

                "task_seed": int(
                    task_seed
                ),

                "model_class": (
                    "ILPredictor"
                ),

                "encoder_class": (
                    "MPNNEncoder"
                ),

                "descriptor_dim": 130,

                "best_epoch": int(
                    best_epoch
                ),

                "best_early_stop_loss": float(
                    best_val_loss
                ),

                "mpnn_supervised_train_rows": (
                    expected_train_rows
                ),

                "mpnn_early_stop_rows": (
                    expected_early_stop_rows
                ),

                "inner_ml_validation_used": (
                    False
                ),

                "outer_test_used": False,

                "model_state_dict": (
                    best_state_dict
                ),
            }


            save_torch_atomic(
                data=best_state_payload,
                output_path=(
                    best_state_path
                ),
            )


        else:
            no_improve_count += 1


        history_row = {
            "task_id": task_id,

            "split_name": (
                preprocessing_context[
                    "split_name"
                ]
            ),

            "outer_fold": int(
                preprocessing_context[
                    "outer_fold"
                ]
            ),

            "inner_fold": int(
                preprocessing_context[
                    "inner_fold"
                ]
            ),

            "task_seed": int(
                task_seed
            ),

            "epoch": int(
                epoch
            ),

            "train_loss": float(
                average_train_loss
            ),

            "early_stop_loss": float(
                average_early_stop_loss
            ),

            "best_early_stop_loss": float(
                best_val_loss
            ),

            "best_epoch": int(
                best_epoch
            ),

            "learning_rate": float(
                current_learning_rate
            ),

            "improved": int(
                improved
            ),

            "no_improve_count": int(
                no_improve_count
            ),

            "train_rows_used": int(
                train_sample_count
            ),

            "early_stop_rows_used": int(
                early_stop_sample_count
            ),

            "inner_ml_valid_rows_used": 0,

            "outer_test_rows_used": 0,
        }


        training_history.append(
            history_row
        )


        pd.DataFrame(
            training_history
        ).to_csv(
            training_history_path,
            index=False,
            encoding="utf-8-sig",
        )


        session_minutes_now = (
            time.perf_counter()
            - session_start_time
        ) / 60.0


        cumulative_minutes_now = (
            cumulative_minutes_before
            + session_minutes_now
        )


        checkpoint_payload = {
            "task_id": task_id,

            "run_code_version": (
                RUN_CODE_VERSION
            ),

            "epoch_completed": int(
                epoch
            ),

            "model_state_dict": (
                clone_state_dict_to_cpu(
                    model.state_dict()
                )
            ),

            "optimizer_state_dict": (
                optimizer.state_dict()
            ),

            "scheduler_state_dict": (
                scheduler.state_dict()
            ),

            "best_epoch": int(
                best_epoch
            ),

            "best_val_loss": float(
                best_val_loss
            ),

            "best_state_dict": (
                best_state_dict
            ),

            "no_improve_count": int(
                no_improve_count
            ),

            "training_history": (
                training_history
            ),

            "train_generator_state": (
                train_generator.get_state()
            ),

            "python_random_state": (
                random.getstate()
            ),

            "numpy_random_state": (
                np.random.get_state()
            ),

            "torch_cpu_rng_state": (
                torch.get_rng_state()
            ),

            "torch_cuda_rng_state_all": (
                torch.cuda.get_rng_state_all()
                if DEVICE.type == "cuda"
                else None
            ),

            "cumulative_training_minutes": (
                float(
                    cumulative_minutes_now
                )
            ),
        }


        save_torch_atomic(
            data=checkpoint_payload,
            output_path=checkpoint_path,
        )


        progress_record = {
            "updated_at": (
                datetime.now().isoformat(
                    timespec="seconds"
                )
            ),

            "status": "training",

            "task_id": task_id,

            "epoch_completed": int(
                epoch
            ),

            "maximum_epochs": int(
                max_epochs
            ),

            "best_epoch": int(
                best_epoch
            ),

            "best_early_stop_loss": float(
                best_val_loss
            ),

            "no_improve_count": int(
                no_improve_count
            ),

            "current_learning_rate": float(
                current_learning_rate
            ),

            "cumulative_training_minutes": (
                float(
                    cumulative_minutes_now
                )
            ),

            "checkpoint_path": str(
                checkpoint_path
            ),
        }


        save_json_atomic(
            data=progress_record,
            output_path=progress_path,
        )


        if (
            epoch == start_epoch
            or epoch % 10 == 0
            or improved
            or epoch == max_epochs
        ):
            print(
                f"Epoch {epoch:03d}/{max_epochs} | "
                f"Train Loss: "
                f"{average_train_loss:.6f} | "
                f"Early-stop Loss: "
                f"{average_early_stop_loss:.6f} | "
                f"Best: {best_val_loss:.6f} "
                f"@ Epoch {best_epoch:03d} | "
                f"LR: {current_learning_rate:.8f}"
            )


        if (
            no_improve_count
            >= early_stop_patience
        ):
            stopped_early = True

            print(
                f"Early stopping at Epoch {epoch}; "
                f"best epoch = {best_epoch}"
            )

            break


    # --------------------------------------------------------
    # 7. 恢复最佳模型并保存最终记录
    # --------------------------------------------------------

    if best_state_dict is None:
        raise RuntimeError(
            "训练结束后没有获得最佳模型状态。"
        )


    model.load_state_dict(
        best_state_dict
    )


    model.to(
        DEVICE
    )

    model.eval()


    final_epoch = int(
        training_history[
            -1
        ][
            "epoch"
        ]
    )


    session_training_minutes = (
        time.perf_counter()
        - session_start_time
    ) / 60.0


    cumulative_training_minutes = (
        cumulative_minutes_before
        + session_training_minutes
    )


    # 结束时再保存一次最佳状态，确保文件与内存一致。
    final_best_payload = {
        "property": "viscosity",

        "target_col": "lg_eta",

        "representation_method": (
            "inner-fold-specific "
            "supervised MPNN"
        ),

        "task_id": task_id,

        "split_name": (
            preprocessing_context[
                "split_name"
            ]
        ),

        "outer_fold": int(
            preprocessing_context[
                "outer_fold"
            ]
        ),

        "inner_fold": int(
            preprocessing_context[
                "inner_fold"
            ]
        ),

        "task_seed": int(
            task_seed
        ),

        "model_class": "ILPredictor",

        "encoder_class": "MPNNEncoder",

        "descriptor_dim": 130,

        "best_epoch": int(
            best_epoch
        ),

        "best_early_stop_loss": float(
            best_val_loss
        ),

        "mpnn_supervised_train_rows": (
            expected_train_rows
        ),

        "mpnn_early_stop_rows": (
            expected_early_stop_rows
        ),

        "inner_ml_validation_used": False,

        "outer_test_used": False,

        "model_state_dict": (
            clone_state_dict_to_cpu(
                model.state_dict()
            )
        ),
    }


    save_torch_atomic(
        data=final_best_payload,
        output_path=best_state_path,
    )


    training_summary = {
        "created_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),

        "run_code_version": (
            RUN_CODE_VERSION
        ),

        "status": (
            "training_completed"
        ),

        "task_id": task_id,

        "split_name": (
            preprocessing_context[
                "split_name"
            ]
        ),

        "outer_fold": int(
            preprocessing_context[
                "outer_fold"
            ]
        ),

        "inner_fold": int(
            preprocessing_context[
                "inner_fold"
            ]
        ),

        "task_seed": int(
            task_seed
        ),

        "device": str(
            DEVICE
        ),

        "training_configuration": {
            "optimizer": "Adam",

            "initial_learning_rate": float(
                INITIAL_LEARNING_RATE
            ),

            "weight_decay": float(
                WEIGHT_DECAY
            ),

            "scheduler": (
                "ReduceLROnPlateau"
            ),

            "scheduler_factor": 0.5,

            "scheduler_patience": 8,

            "maximum_epochs": int(
                max_epochs
            ),

            "early_stopping_patience": int(
                early_stop_patience
            ),

            "minimum_improvement": float(
                min_delta
            ),

            "batch_size": int(
                BATCH_SIZE
            ),
        },

        "data_counts": {
            "mpnn_supervised_train_rows": (
                expected_train_rows
            ),

            "mpnn_early_stop_rows": (
                expected_early_stop_rows
            ),

            "inner_ml_valid_rows_used": 0,

            "outer_test_rows_used": 0,
        },

        "result": {
            "resumed_from_checkpoint": bool(
                resumed_from_checkpoint
            ),

            "resumed_after_epoch": int(
                resumed_after_epoch
            ),

            "epochs_run_this_session": int(
                epochs_run_this_session
            ),

            "final_epoch": int(
                final_epoch
            ),

            "best_epoch": int(
                best_epoch
            ),

            "best_early_stop_loss": float(
                best_val_loss
            ),

            "stopped_early": bool(
                stopped_early
            ),

            "session_training_minutes": float(
                session_training_minutes
            ),

            "cumulative_training_minutes": float(
                cumulative_training_minutes
            ),
        },

        "leakage_control": {
            "inner_ml_validation_used_for_"
            "backpropagation": False,

            "inner_ml_validation_used_for_"
            "early_stopping": False,

            "inner_ml_validation_used_for_"
            "scheduler": False,

            "outer_test_used_for_"
            "backpropagation": False,

            "outer_test_used_for_"
            "early_stopping": False,

            "outer_test_used_for_"
            "scheduler": False,

            "existing_outer_mpnn_features_read": (
                False
            ),

            "old_global_mpnn_feature_read": False,

            "external_10point_holdout_read": False,
        },

        "outputs": {
            "training_history": str(
                training_history_path
            ),

            "best_state": str(
                best_state_path
            ),

            "training_summary": str(
                training_summary_path
            ),

            "training_progress": str(
                progress_path
            ),
        },
    }


    save_json_atomic(
        data=training_summary,
        output_path=training_summary_path,
    )


    final_progress = {
        "updated_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),

        "status": "training_completed",

        "task_id": task_id,

        "final_epoch": int(
            final_epoch
        ),

        "best_epoch": int(
            best_epoch
        ),

        "best_early_stop_loss": float(
            best_val_loss
        ),

        "stopped_early": bool(
            stopped_early
        ),

        "cumulative_training_minutes": float(
            cumulative_training_minutes
        ),

        "checkpoint_removed": True,
    }


    save_json_atomic(
        data=final_progress,
        output_path=progress_path,
    )


    if checkpoint_path.exists():
        checkpoint_path.unlink()


    return {
        "model": model,

        "training_history": pd.DataFrame(
            training_history
        ),

        "training_summary": (
            training_summary
        ),

        "paths": {
            "training_history": (
                training_history_path
            ),

            "best_state": (
                best_state_path
            ),

            "training_summary": (
                training_summary_path
            ),

            "training_progress": (
                progress_path
            ),

            "checkpoint": (
                checkpoint_path
            ),
        },
    }


# ============================================================
# 8. 只训练第一套测试任务
# ============================================================

TEST_TASK_TRAINING_RESULT = (
    train_one_inner_mpnn(
        preprocessing_context=(
            TEST_TASK_PREPROCESSING
        ),
        dataloader_context=(
            TEST_TASK_DATALOADERS
        ),
        max_epochs=MAX_EPOCHS,
        early_stop_patience=(
            EARLY_STOP_PATIENCE
        ),
        min_delta=MIN_DELTA,
    )
)


TEST_TRAINED_MPNN_MODEL = (
    TEST_TASK_TRAINING_RESULT[
        "model"
    ]
)


# ============================================================
# 9. 输出最终结果
# ============================================================

summary = (
    TEST_TASK_TRAINING_RESULT[
        "training_summary"
    ]
)


print("=" * 100)
print("Cell 5 第一套内层MPNN监督训练完成 ✓")
print("=" * 100)

print(
    "task id                      =",
    summary[
        "task_id"
    ],
)

print(
    "supervised train rows        =",
    summary[
        "data_counts"
    ][
        "mpnn_supervised_train_rows"
    ],
)

print(
    "early-stop rows              =",
    summary[
        "data_counts"
    ][
        "mpnn_early_stop_rows"
    ],
)

print(
    "inner ML valid rows used     =",
    summary[
        "data_counts"
    ][
        "inner_ml_valid_rows_used"
    ],
)

print(
    "outer test rows used         =",
    summary[
        "data_counts"
    ][
        "outer_test_rows_used"
    ],
)

print(
    "resumed from checkpoint      =",
    summary[
        "result"
    ][
        "resumed_from_checkpoint"
    ],
)

print(
    "final epoch                  =",
    summary[
        "result"
    ][
        "final_epoch"
    ],
)

print(
    "best epoch                   =",
    summary[
        "result"
    ][
        "best_epoch"
    ],
)

print(
    "best early-stop loss         =",
    summary[
        "result"
    ][
        "best_early_stop_loss"
    ],
)

print(
    "stopped early                =",
    summary[
        "result"
    ][
        "stopped_early"
    ],
)

print(
    "training minutes             =",
    summary[
        "result"
    ][
        "cumulative_training_minutes"
    ],
)

print(
    "inner ML validation used     =",
    False,
)

print(
    "outer test used              =",
    False,
)


print(
    "\n训练历史："
)

print(
    TEST_TASK_TRAINING_RESULT[
        "paths"
    ][
        "training_history"
    ]
)


print(
    "\n最佳模型："
)

print(
    TEST_TASK_TRAINING_RESULT[
        "paths"
    ][
        "best_state"
    ]
)


print(
    "\n训练摘要："
)

print(
    TEST_TASK_TRAINING_RESULT[
        "paths"
    ][
        "training_summary"
    ]
)


print(
    "\n训练进度："
)

print(
    TEST_TASK_TRAINING_RESULT[
        "paths"
    ][
        "training_progress"
    ]
)

开始第一套内层MPNN监督训练
task id                       = Data_split_OuterFold1_InnerFold1
device                        = cpu
MPNN supervised train rows    = 6918
MPNN early-stop rows          = 1730
inner ML valid rows used      = 0
outer test rows used          = 0
resume from checkpoint        = False
resume after epoch            = 0
start epoch                   = 1
maximum epochs                = 150
early stopping patience       = 20
Epoch 001/150 | Train Loss: 0.890614 | Early-stop Loss: 0.766090 | Best: 0.766090 @ Epoch 001 | LR: 0.00100000
Epoch 002/150 | Train Loss: 0.658052 | Early-stop Loss: 0.531043 | Best: 0.531043 @ Epoch 002 | LR: 0.00100000
Epoch 003/150 | Train Loss: 0.478444 | Early-stop Loss: 0.429739 | Best: 0.429739 @ Epoch 003 | LR: 0.00100000
Epoch 004/150 | Train Loss: 0.419355 | Early-stop Loss: 0.388997 | Best: 0.388997 @ Epoch 004 | LR: 0.00100000
Epoch 005/150 | Train Loss: 0.366229 | Early-stop Loss: 0.334483 | Best: 0.334483 @ Epoch 005 | LR: 0.00100000
Epoch 006

In [7]:
# Cell 6 | 加载第一套最佳MPNN并提取内层训练与验证130维表征


def save_numpy_atomic(
    array: np.ndarray,
    output_path: Union[str, Path],
) -> None:
    """原子方式保存NumPy数组，降低中断导致文件损坏的风险。"""
    output_path = Path(
        output_path
    )

    temporary_path = (
        output_path.parent
        / (
            output_path.name
            + ".tmp"
        )
    )

    with open(
        temporary_path,
        "wb",
    ) as file:
        np.save(
            file,
            array,
            allow_pickle=False,
        )

    os.replace(
        temporary_path,
        output_path,
    )


def extract_mpnn_features_without_targets(
    model: nn.Module,
    feature_loader: DataLoader,
    expected_rows: int,
    feature_name: str,
) -> np.ndarray:
    """
    从不返回目标值的DataLoader中提取130维MPNN表征。
    """
    model.eval()

    feature_blocks = []

    extracted_rows = 0


    with torch.no_grad():
        for (
            cation_batch,
            anion_batch,
            environment_batch,
        ) in feature_loader:

            cation_batch = (
                cation_batch.to(
                    DEVICE
                )
            )

            anion_batch = (
                anion_batch.to(
                    DEVICE
                )
            )

            environment_batch = (
                environment_batch.to(
                    DEVICE
                )
            )


            feature_batch = (
                model.extract_features(
                    cation_batch,
                    anion_batch,
                    environment_batch,
                )
            )


            if (
                feature_batch.ndim != 2
                or feature_batch.shape[1] != 130
            ):
                raise RuntimeError(
                    f"{feature_name}批次维度错误："
                    f"{tuple(feature_batch.shape)}"
                )


            if not torch.isfinite(
                feature_batch
            ).all():
                raise RuntimeError(
                    f"{feature_name}中出现NaN或无穷值。"
                )


            feature_numpy = (
                feature_batch
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )


            feature_blocks.append(
                feature_numpy
            )


            extracted_rows += int(
                feature_numpy.shape[0]
            )


    if extracted_rows != expected_rows:
        raise RuntimeError(
            f"{feature_name}提取行数错误："
            f"实际={extracted_rows}，"
            f"预期={expected_rows}"
        )


    features = np.concatenate(
        feature_blocks,
        axis=0,
    )


    if features.shape != (
        expected_rows,
        130,
    ):
        raise RuntimeError(
            f"{feature_name}最终形状错误："
            f"{features.shape}"
        )


    if not np.isfinite(
        features
    ).all():
        raise RuntimeError(
            f"{feature_name}最终结果"
            "存在NaN或无穷值。"
        )


    return features


# ============================================================
# 1. 读取并核对第一套任务的最佳模型
# ============================================================

TEST_TASK_OUTPUT_DIR = Path(
    TEST_TASK_PREPROCESSING[
        "path_map"
    ][
        "preprocessing_audit"
    ]
).parent


TEST_BEST_STATE_PATH = (
    TEST_TASK_OUTPUT_DIR
    / "mpnn_best_state.pt"
)


if not TEST_BEST_STATE_PATH.exists():
    raise FileNotFoundError(
        "找不到第一套任务的最佳MPNN模型：\n"
        f"{TEST_BEST_STATE_PATH}"
    )


best_state_payload = (
    load_torch_object(
        TEST_BEST_STATE_PATH
    )
)


required_best_state_keys = {
    "task_id",
    "split_name",
    "outer_fold",
    "inner_fold",
    "task_seed",
    "descriptor_dim",
    "best_epoch",
    "best_early_stop_loss",
    "mpnn_supervised_train_rows",
    "mpnn_early_stop_rows",
    "inner_ml_validation_used",
    "outer_test_used",
    "model_state_dict",
}


missing_best_state_keys = (
    required_best_state_keys
    - set(
        best_state_payload.keys()
    )
)


if missing_best_state_keys:
    raise KeyError(
        "最佳模型文件缺少字段："
        f"{sorted(missing_best_state_keys)}"
    )


if (
    best_state_payload[
        "task_id"
    ]
    != TEST_TASK_PREPROCESSING[
        "task_id"
    ]
):
    raise RuntimeError(
        "最佳模型文件属于其他任务。"
    )


if int(
    best_state_payload[
        "task_seed"
    ]
) != int(
    TEST_TASK_PREPROCESSING[
        "task_seed"
    ]
):
    raise RuntimeError(
        "最佳模型随机种子"
        "与当前任务不一致。"
    )


if int(
    best_state_payload[
        "descriptor_dim"
    ]
) != 130:
    raise RuntimeError(
        "最佳模型表征维度不是130。"
    )


if bool(
    best_state_payload[
        "inner_ml_validation_used"
    ]
):
    raise RuntimeError(
        "最佳模型记录显示内层ML验证集"
        "参与了MPNN训练。"
    )


if bool(
    best_state_payload[
        "outer_test_used"
    ]
):
    raise RuntimeError(
        "最佳模型记录显示外层测试集"
        "参与了MPNN训练。"
    )


if int(
    best_state_payload[
        "mpnn_supervised_train_rows"
    ]
) != 6918:
    raise RuntimeError(
        "最佳模型监督训练样本数不是6918。"
    )


if int(
    best_state_payload[
        "mpnn_early_stop_rows"
    ]
) != 1730:
    raise RuntimeError(
        "最佳模型早停样本数不是1730。"
    )


# ============================================================
# 2. 建立全新模型并载入最佳状态
# ============================================================

TEST_FEATURE_EXTRACTOR = (
    build_fresh_mpnn_model(
        task_seed=int(
            best_state_payload[
                "task_seed"
            ]
        )
    )
)


TEST_FEATURE_EXTRACTOR.load_state_dict(
    best_state_payload[
        "model_state_dict"
    ],
    strict=True,
)


TEST_FEATURE_EXTRACTOR.to(
    DEVICE
)

TEST_FEATURE_EXTRACTOR.eval()


loaded_parameter_counts = (
    count_model_parameters(
        TEST_FEATURE_EXTRACTOR
    )
)


if (
    loaded_parameter_counts[
        "total_parameters"
    ]
    != 298817
):
    raise RuntimeError(
        "加载后的MPNN参数量错误。"
    )


# ============================================================
# 3. 在完全不读取目标值的情况下提取两套表征
# ============================================================

feature_extraction_started_at = (
    datetime.now().isoformat(
        timespec="seconds"
    )
)


X_inner_train_mpnn = (
    extract_mpnn_features_without_targets(
        model=(
            TEST_FEATURE_EXTRACTOR
        ),
        feature_loader=(
            TEST_INNER_TRAIN_FEATURE_LOADER
        ),
        expected_rows=8648,
        feature_name=(
            "X_inner_train_mpnn"
        ),
    )
)


X_inner_valid_mpnn = (
    extract_mpnn_features_without_targets(
        model=(
            TEST_FEATURE_EXTRACTOR
        ),
        feature_loader=(
            TEST_INNER_VALID_FEATURE_LOADER
        ),
        expected_rows=4324,
        feature_name=(
            "X_inner_valid_mpnn"
        ),
    )
)


feature_extraction_completed_at = (
    datetime.now().isoformat(
        timespec="seconds"
    )
)


# ============================================================
# 4. 检查130维表征中的最后两列确实是标准化T/P
# ============================================================

train_tp_match = bool(
    np.allclose(
        X_inner_train_mpnn[
            :,
            -2:
        ],
        TEST_INNER_TRAIN_TP_SCALED,
        rtol=1e-6,
        atol=1e-6,
    )
)


valid_tp_match = bool(
    np.allclose(
        X_inner_valid_mpnn[
            :,
            -2:
        ],
        TEST_INNER_VALID_TP_SCALED,
        rtol=1e-6,
        atol=1e-6,
    )
)


if not train_tp_match:
    raise RuntimeError(
        "内层训练表征最后两列"
        "与标准化T/P不一致。"
    )


if not valid_tp_match:
    raise RuntimeError(
        "内层验证表征最后两列"
        "与标准化T/P不一致。"
    )


# ============================================================
# 5. 表征提取完成后，才读取下游ML所需目标
# ============================================================

inner_train_target_manifest = (
    TEST_TASK_PREPROCESSING[
        "inner_train_manifest"
    ]
)


inner_valid_target_manifest = (
    TEST_OUTER_TRAIN_MANIFEST.iloc[
        TEST_INNER_VALID_RELATIVE_IDX
    ]
    .reset_index(
        drop=True
    )
    .copy()
)


y_inner_train_lg_eta = (
    inner_train_target_manifest[
        "lg_eta"
    ]
    .to_numpy(
        dtype=np.float32
    )
)


y_inner_valid_lg_eta = (
    inner_valid_target_manifest[
        "lg_eta"
    ]
    .to_numpy(
        dtype=np.float32
    )
)


if y_inner_train_lg_eta.shape != (
    8648,
):
    raise RuntimeError(
        "内层训练目标形状错误："
        f"{y_inner_train_lg_eta.shape}"
    )


if y_inner_valid_lg_eta.shape != (
    4324,
):
    raise RuntimeError(
        "内层验证目标形状错误："
        f"{y_inner_valid_lg_eta.shape}"
    )


if not np.isfinite(
    y_inner_train_lg_eta
).all():
    raise RuntimeError(
        "内层训练目标中存在NaN或无穷值。"
    )


if not np.isfinite(
    y_inner_valid_lg_eta
).all():
    raise RuntimeError(
        "内层验证目标中存在NaN或无穷值。"
    )


# 再次核对目标顺序对应的全局行号。
if not np.array_equal(
    inner_train_target_manifest[
        "original_feature_row_idx"
    ].to_numpy(
        dtype=np.int64
    ),
    TEST_TASK_CONTEXT[
        "inner_train_global_idx"
    ],
):
    raise RuntimeError(
        "内层训练目标顺序"
        "与内层训练表征顺序不一致。"
    )


if not np.array_equal(
    inner_valid_target_manifest[
        "original_feature_row_idx"
    ].to_numpy(
        dtype=np.int64
    ),
    TEST_TASK_CONTEXT[
        "inner_valid_global_idx"
    ],
):
    raise RuntimeError(
        "内层验证目标顺序"
        "与内层验证表征顺序不一致。"
    )


# ============================================================
# 6. 保存下游调参所需的最小文件集合
# ============================================================

X_INNER_TRAIN_PATH = (
    TEST_TASK_OUTPUT_DIR
    / "X_inner_train_mpnn.npy"
)

X_INNER_VALID_PATH = (
    TEST_TASK_OUTPUT_DIR
    / "X_inner_valid_mpnn.npy"
)

Y_INNER_TRAIN_PATH = (
    TEST_TASK_OUTPUT_DIR
    / "y_inner_train_lg_eta.npy"
)

Y_INNER_VALID_PATH = (
    TEST_TASK_OUTPUT_DIR
    / "y_inner_valid_lg_eta.npy"
)

INNER_FEATURE_METADATA_PATH = (
    TEST_TASK_OUTPUT_DIR
    / "inner_feature_metadata.json"
)

INNER_OUTPUT_AUDIT_PATH = (
    TEST_TASK_OUTPUT_DIR
    / "inner_output_audit.json"
)


save_numpy_atomic(
    array=X_inner_train_mpnn,
    output_path=X_INNER_TRAIN_PATH,
)

save_numpy_atomic(
    array=X_inner_valid_mpnn,
    output_path=X_INNER_VALID_PATH,
)

save_numpy_atomic(
    array=y_inner_train_lg_eta,
    output_path=Y_INNER_TRAIN_PATH,
)

save_numpy_atomic(
    array=y_inner_valid_lg_eta,
    output_path=Y_INNER_VALID_PATH,
)


# ============================================================
# 7. 重新从磁盘读取并完成最终审计
# ============================================================

X_inner_train_reloaded = np.load(
    X_INNER_TRAIN_PATH,
    allow_pickle=False,
)

X_inner_valid_reloaded = np.load(
    X_INNER_VALID_PATH,
    allow_pickle=False,
)

y_inner_train_reloaded = np.load(
    Y_INNER_TRAIN_PATH,
    allow_pickle=False,
)

y_inner_valid_reloaded = np.load(
    Y_INNER_VALID_PATH,
    allow_pickle=False,
)


expected_output_shapes = {
    "X_inner_train_mpnn": [
        8648,
        130,
    ],

    "X_inner_valid_mpnn": [
        4324,
        130,
    ],

    "y_inner_train_lg_eta": [
        8648,
    ],

    "y_inner_valid_lg_eta": [
        4324,
    ],
}


actual_output_shapes = {
    "X_inner_train_mpnn": list(
        X_inner_train_reloaded.shape
    ),

    "X_inner_valid_mpnn": list(
        X_inner_valid_reloaded.shape
    ),

    "y_inner_train_lg_eta": list(
        y_inner_train_reloaded.shape
    ),

    "y_inner_valid_lg_eta": list(
        y_inner_valid_reloaded.shape
    ),
}


if actual_output_shapes != expected_output_shapes:
    raise RuntimeError(
        "保存后的输出形状不正确：\n"
        f"实际={actual_output_shapes}\n"
        f"预期={expected_output_shapes}"
    )


output_paths = {
    "X_inner_train_mpnn": (
        X_INNER_TRAIN_PATH
    ),

    "X_inner_valid_mpnn": (
        X_INNER_VALID_PATH
    ),

    "y_inner_train_lg_eta": (
        Y_INNER_TRAIN_PATH
    ),

    "y_inner_valid_lg_eta": (
        Y_INNER_VALID_PATH
    ),
}


output_sha256 = {
    output_name: calculate_sha256(
        output_path
    )

    for output_name, output_path
    in output_paths.items()
}


# ============================================================
# 8. 保存表征元数据
# ============================================================

inner_feature_metadata = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "status": (
        "feature_extraction_completed"
    ),

    "task_id": (
        TEST_TASK_PREPROCESSING[
            "task_id"
        ]
    ),

    "split_name": (
        TEST_TASK_PREPROCESSING[
            "split_name"
        ]
    ),

    "outer_fold": int(
        TEST_TASK_PREPROCESSING[
            "outer_fold"
        ]
    ),

    "inner_fold": int(
        TEST_TASK_PREPROCESSING[
            "inner_fold"
        ]
    ),

    "task_seed": int(
        TEST_TASK_PREPROCESSING[
            "task_seed"
        ]
    ),

    "feature_extraction": {
        "started_at": (
            feature_extraction_started_at
        ),

        "completed_at": (
            feature_extraction_completed_at
        ),

        "best_model_path": str(
            TEST_BEST_STATE_PATH
        ),

        "best_model_sha256": (
            calculate_sha256(
                TEST_BEST_STATE_PATH
            )
        ),

        "best_epoch": int(
            best_state_payload[
                "best_epoch"
            ]
        ),

        "best_early_stop_loss": float(
            best_state_payload[
                "best_early_stop_loss"
            ]
        ),

        "descriptor_dim": 130,

        "cation_embedding_dim": 64,

        "anion_embedding_dim": 64,

        "environment_dim": 2,
    },

    "output_shapes": (
        actual_output_shapes
    ),

    "output_dtypes": {
        "X_inner_train_mpnn": str(
            X_inner_train_reloaded.dtype
        ),

        "X_inner_valid_mpnn": str(
            X_inner_valid_reloaded.dtype
        ),

        "y_inner_train_lg_eta": str(
            y_inner_train_reloaded.dtype
        ),

        "y_inner_valid_lg_eta": str(
            y_inner_valid_reloaded.dtype
        ),
    },

    "output_sha256": (
        output_sha256
    ),

    "output_paths": {
        key: str(
            value
        )
        for key, value
        in output_paths.items()
    },
}


save_json_atomic(
    data=inner_feature_metadata,
    output_path=(
        INNER_FEATURE_METADATA_PATH
    ),
)


# ============================================================
# 9. 保存无泄露输出审计
# ============================================================

inner_output_audit = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "audit_status": "passed",

    "task_id": (
        TEST_TASK_PREPROCESSING[
            "task_id"
        ]
    ),

    "checks": {
        "best_state_loaded": True,

        "best_epoch": int(
            best_state_payload[
                "best_epoch"
            ]
        ),

        "inner_train_feature_shape_correct": (
            X_inner_train_reloaded.shape
            == (
                8648,
                130,
            )
        ),

        "inner_valid_feature_shape_correct": (
            X_inner_valid_reloaded.shape
            == (
                4324,
                130,
            )
        ),

        "inner_train_target_shape_correct": (
            y_inner_train_reloaded.shape
            == (
                8648,
            )
        ),

        "inner_valid_target_shape_correct": (
            y_inner_valid_reloaded.shape
            == (
                4324,
            )
        ),

        "all_features_finite": bool(
            np.isfinite(
                X_inner_train_reloaded
            ).all()
            and np.isfinite(
                X_inner_valid_reloaded
            ).all()
        ),

        "all_targets_finite": bool(
            np.isfinite(
                y_inner_train_reloaded
            ).all()
            and np.isfinite(
                y_inner_valid_reloaded
            ).all()
        ),

        "train_tp_columns_match": (
            train_tp_match
        ),

        "valid_tp_columns_match": (
            valid_tp_match
        ),

        "train_feature_target_order_match": (
            True
        ),

        "valid_feature_target_order_match": (
            True
        ),
    },

    "leakage_control": {
        "inner_validation_target_read_"
        "before_feature_extraction": False,

        "inner_validation_target_read_"
        "after_feature_extraction": True,

        "inner_validation_target_used_for_"
        "mpnn_training": False,

        "inner_validation_target_used_for_"
        "mpnn_early_stopping": False,

        "inner_validation_target_used_for_"
        "mpnn_feature_extraction": False,

        "inner_validation_target_saved_for_"
        "downstream_ml_validation": True,

        "outer_test_structure_read": False,

        "outer_test_target_read": False,

        "outer_test_feature_extracted": False,

        "existing_outer_mpnn_features_read": (
            False
        ),

        "old_global_mpnn_feature_read": False,

        "external_10point_holdout_read": False,
    },

    "outputs": {
        **{
            key: str(
                value
            )
            for key, value
            in output_paths.items()
        },

        "feature_metadata": str(
            INNER_FEATURE_METADATA_PATH
        ),

        "output_audit": str(
            INNER_OUTPUT_AUDIT_PATH
        ),
    },
}


if not all(
    value
    for value in inner_output_audit[
        "checks"
    ].values()
):
    failed_checks = [
        key
        for key, value
        in inner_output_audit[
            "checks"
        ].items()
        if not value
    ]

    raise RuntimeError(
        "第一套内层表征输出审计失败："
        f"{failed_checks}"
    )


save_json_atomic(
    data=inner_output_audit,
    output_path=(
        INNER_OUTPUT_AUDIT_PATH
    ),
)


# ============================================================
# 10. 清理临时重新读取对象
# ============================================================

del (
    X_inner_train_reloaded,
    X_inner_valid_reloaded,
    y_inner_train_reloaded,
    y_inner_valid_reloaded,
)

gc.collect()


# ============================================================
# 11. 输出检查
# ============================================================

print("=" * 100)
print("Cell 6 第一套内层训练与验证MPNN表征提取完成 ✓")
print("=" * 100)

print(
    "task id                      =",
    inner_output_audit[
        "task_id"
    ],
)

print(
    "best epoch                   =",
    best_state_payload[
        "best_epoch"
    ],
)

print(
    "best early-stop loss         =",
    best_state_payload[
        "best_early_stop_loss"
    ],
)

print(
    "X inner train shape          =",
    X_inner_train_mpnn.shape,
)

print(
    "X inner valid shape          =",
    X_inner_valid_mpnn.shape,
)

print(
    "y inner train shape          =",
    y_inner_train_lg_eta.shape,
)

print(
    "y inner valid shape          =",
    y_inner_valid_lg_eta.shape,
)

print(
    "train T/P columns match      =",
    train_tp_match,
)

print(
    "valid T/P columns match      =",
    valid_tp_match,
)

print(
    "all outputs finite           =",
    True,
)

print(
    "inner valid target used by MPNN =",
    False,
)

print(
    "inner valid target saved for ML =",
    True,
)

print(
    "outer test target read       =",
    False,
)

print(
    "outer test feature extracted =",
    False,
)

print(
    "output audit status          =",
    inner_output_audit[
        "audit_status"
    ],
)


print(
    "\n内层训练表征："
)

print(
    X_INNER_TRAIN_PATH
)


print(
    "\n内层验证表征："
)

print(
    X_INNER_VALID_PATH
)


print(
    "\n内层训练目标："
)

print(
    Y_INNER_TRAIN_PATH
)


print(
    "\n内层验证目标："
)

print(
    Y_INNER_VALID_PATH
)


print(
    "\n表征元数据："
)

print(
    INNER_FEATURE_METADATA_PATH
)


print(
    "\n输出审计："
)

print(
    INNER_OUTPUT_AUDIT_PATH
)

Cell 6 第一套内层训练与验证MPNN表征提取完成 ✓
task id                      = Data_split_OuterFold1_InnerFold1
best epoch                   = 146
best early-stop loss         = 0.037401667413856254
X inner train shape          = (8648, 130)
X inner valid shape          = (4324, 130)
y inner train shape          = (8648,)
y inner valid shape          = (4324,)
train T/P columns match      = True
valid T/P columns match      = True
all outputs finite           = True
inner valid target used by MPNN = False
inner valid target saved for ML = True
outer test target read       = False
outer test feature extracted = False
output audit status          = passed

内层训练表征：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\inner_foldwise_mpnn_tuning_features\Data_split\OuterFold_1\InnerFold_1\X_inner_train_mpnn.npy

内层验证表征：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\in

In [8]:
# Cell 7 | 完成第一套内层MPNN总审计并测试下游调参读取接口


# ============================================================
# 1. 第一套任务全部预期文件
# ============================================================

FIRST_TASK_REQUIRED_PATHS = {
    "inner_train_manifest": (
        TEST_TASK_OUTPUT_DIR
        / "inner_train_manifest.csv"
    ),

    "inner_valid_structure_manifest": (
        TEST_TASK_OUTPUT_DIR
        / "inner_valid_structure_manifest_no_targets.csv"
    ),

    "mpnn_supervised_train_manifest": (
        TEST_TASK_OUTPUT_DIR
        / "mpnn_supervised_train_manifest.csv"
    ),

    "mpnn_early_stop_manifest": (
        TEST_TASK_OUTPUT_DIR
        / "mpnn_early_stop_manifest.csv"
    ),

    "mpnn_internal_indices": (
        TEST_TASK_OUTPUT_DIR
        / "mpnn_internal_indices.npz"
    ),

    "tp_scaler": (
        TEST_TASK_OUTPUT_DIR
        / "tp_scaler.joblib"
    ),

    "target_scaler": (
        TEST_TASK_OUTPUT_DIR
        / "target_scaler.joblib"
    ),

    "preprocessing_audit": (
        TEST_TASK_OUTPUT_DIR
        / "mpnn_preprocessing_audit.json"
    ),

    "dataloader_audit": (
        TEST_TASK_OUTPUT_DIR
        / "mpnn_dataloader_audit.json"
    ),

    "model_structure_audit": (
        TEST_TASK_OUTPUT_DIR
        / "mpnn_model_structure_audit.json"
    ),

    "training_history": (
        TEST_TASK_OUTPUT_DIR
        / "mpnn_training_history.csv"
    ),

    "training_summary": (
        TEST_TASK_OUTPUT_DIR
        / "mpnn_training_summary.json"
    ),

    "training_progress": (
        TEST_TASK_OUTPUT_DIR
        / "mpnn_training_progress.json"
    ),

    "best_state": (
        TEST_TASK_OUTPUT_DIR
        / "mpnn_best_state.pt"
    ),

    "X_inner_train_mpnn": (
        TEST_TASK_OUTPUT_DIR
        / "X_inner_train_mpnn.npy"
    ),

    "X_inner_valid_mpnn": (
        TEST_TASK_OUTPUT_DIR
        / "X_inner_valid_mpnn.npy"
    ),

    "y_inner_train_lg_eta": (
        TEST_TASK_OUTPUT_DIR
        / "y_inner_train_lg_eta.npy"
    ),

    "y_inner_valid_lg_eta": (
        TEST_TASK_OUTPUT_DIR
        / "y_inner_valid_lg_eta.npy"
    ),

    "inner_feature_metadata": (
        TEST_TASK_OUTPUT_DIR
        / "inner_feature_metadata.json"
    ),

    "inner_output_audit": (
        TEST_TASK_OUTPUT_DIR
        / "inner_output_audit.json"
    ),
}


missing_first_task_files = [
    path
    for path in FIRST_TASK_REQUIRED_PATHS.values()
    if not path.exists()
]


if missing_first_task_files:
    raise FileNotFoundError(
        "第一套任务缺少以下输出文件：\n"
        + "\n".join(
            str(path)
            for path in missing_first_task_files
        )
    )


checkpoint_path = (
    TEST_TASK_OUTPUT_DIR
    / "mpnn_training_checkpoint.pt"
)


if checkpoint_path.exists():
    raise RuntimeError(
        "第一套任务已经完成训练，"
        "但训练断点文件仍然存在：\n"
        f"{checkpoint_path}"
    )


# ============================================================
# 2. 读取各阶段JSON审计文件
# ============================================================

preprocessing_audit = read_json(
    FIRST_TASK_REQUIRED_PATHS[
        "preprocessing_audit"
    ]
)

dataloader_audit = read_json(
    FIRST_TASK_REQUIRED_PATHS[
        "dataloader_audit"
    ]
)

model_structure_audit = read_json(
    FIRST_TASK_REQUIRED_PATHS[
        "model_structure_audit"
    ]
)

training_summary = read_json(
    FIRST_TASK_REQUIRED_PATHS[
        "training_summary"
    ]
)

training_progress = read_json(
    FIRST_TASK_REQUIRED_PATHS[
        "training_progress"
    ]
)

feature_metadata = read_json(
    FIRST_TASK_REQUIRED_PATHS[
        "inner_feature_metadata"
    ]
)

output_audit = read_json(
    FIRST_TASK_REQUIRED_PATHS[
        "inner_output_audit"
    ]
)


expected_task_id = (
    TEST_TASK_PREPROCESSING[
        "task_id"
    ]
)


json_identity_checks = {
    "preprocessing_task_match": (
        preprocessing_audit.get(
            "task_id"
        )
        == expected_task_id
    ),

    "dataloader_task_match": (
        dataloader_audit.get(
            "task_id"
        )
        == expected_task_id
    ),

    "model_structure_task_match": (
        model_structure_audit.get(
            "task_id"
        )
        == expected_task_id
    ),

    "training_summary_task_match": (
        training_summary.get(
            "task_id"
        )
        == expected_task_id
    ),

    "training_progress_task_match": (
        training_progress.get(
            "task_id"
        )
        == expected_task_id
    ),

    "feature_metadata_task_match": (
        feature_metadata.get(
            "task_id"
        )
        == expected_task_id
    ),

    "output_audit_task_match": (
        output_audit.get(
            "task_id"
        )
        == expected_task_id
    ),
}


if not all(
    json_identity_checks.values()
):
    failed_checks = [
        key
        for key, value
        in json_identity_checks.items()
        if not value
    ]

    raise RuntimeError(
        "部分阶段文件属于其他任务："
        f"{failed_checks}"
    )


expected_status_checks = {
    "preprocessing_passed": (
        preprocessing_audit.get(
            "status"
        )
        == "preprocessing_passed"
    ),

    "dataloader_passed": (
        dataloader_audit.get(
            "status"
        )
        == "dataloader_passed"
    ),

    "model_structure_passed": (
        model_structure_audit.get(
            "status"
        )
        == "model_structure_passed"
    ),

    "training_completed": (
        training_summary.get(
            "status"
        )
        == "training_completed"
    ),

    "progress_completed": (
        training_progress.get(
            "status"
        )
        == "training_completed"
    ),

    "feature_extraction_completed": (
        feature_metadata.get(
            "status"
        )
        == "feature_extraction_completed"
    ),

    "output_audit_passed": (
        output_audit.get(
            "audit_status"
        )
        == "passed"
    ),
}


if not all(
    expected_status_checks.values()
):
    failed_checks = [
        key
        for key, value
        in expected_status_checks.items()
        if not value
    ]

    raise RuntimeError(
        "部分阶段状态未通过："
        f"{failed_checks}"
    )


# ============================================================
# 3. 训练历史与最佳Epoch检查
# ============================================================

training_history = read_csv_robust(
    FIRST_TASK_REQUIRED_PATHS[
        "training_history"
    ]
)


required_history_columns = [
    "task_id",
    "epoch",
    "train_loss",
    "early_stop_loss",
    "best_early_stop_loss",
    "best_epoch",
    "learning_rate",
    "train_rows_used",
    "early_stop_rows_used",
    "inner_ml_valid_rows_used",
    "outer_test_rows_used",
]


missing_history_columns = [
    column
    for column in required_history_columns
    if column not in training_history.columns
]


if missing_history_columns:
    raise KeyError(
        "训练历史缺少字段："
        f"{missing_history_columns}"
    )


final_epoch = int(
    training_summary[
        "result"
    ][
        "final_epoch"
    ]
)


best_epoch = int(
    training_summary[
        "result"
    ][
        "best_epoch"
    ]
)


best_early_stop_loss = float(
    training_summary[
        "result"
    ][
        "best_early_stop_loss"
    ]
)


if len(
    training_history
) != final_epoch:
    raise RuntimeError(
        "训练历史行数与final_epoch不一致。"
    )


expected_epochs = np.arange(
    1,
    final_epoch + 1,
    dtype=np.int64,
)


actual_epochs = pd.to_numeric(
    training_history[
        "epoch"
    ],
    errors="raise",
).to_numpy(
    dtype=np.int64
)


if not np.array_equal(
    actual_epochs,
    expected_epochs,
):
    raise RuntimeError(
        "训练历史Epoch不是从1开始连续排列。"
    )


history_best_row_position = int(
    pd.to_numeric(
        training_history[
            "early_stop_loss"
        ],
        errors="raise",
    ).to_numpy(
        dtype=float
    ).argmin()
)


history_best_epoch = int(
    training_history.iloc[
        history_best_row_position
    ][
        "epoch"
    ]
)


history_best_loss = float(
    training_history.iloc[
        history_best_row_position
    ][
        "early_stop_loss"
    ]
)


if history_best_epoch != best_epoch:
    raise RuntimeError(
        "训练历史中的最佳Epoch"
        "与训练摘要不一致。"
    )


if not np.isclose(
    history_best_loss,
    best_early_stop_loss,
    rtol=1e-10,
    atol=1e-10,
):
    raise RuntimeError(
        "训练历史中的最佳早停损失"
        "与训练摘要不一致。"
    )


if not training_history[
    "train_rows_used"
].eq(
    6918
).all():
    raise RuntimeError(
        "部分Epoch的监督训练样本数不是6918。"
    )


if not training_history[
    "early_stop_rows_used"
].eq(
    1730
).all():
    raise RuntimeError(
        "部分Epoch的早停样本数不是1730。"
    )


if not training_history[
    "inner_ml_valid_rows_used"
].eq(
    0
).all():
    raise RuntimeError(
        "训练历史显示内层ML验证数据"
        "参与了MPNN训练。"
    )


if not training_history[
    "outer_test_rows_used"
].eq(
    0
).all():
    raise RuntimeError(
        "训练历史显示外层测试数据"
        "参与了MPNN训练。"
    )


# ============================================================
# 4. 标准化器和内部索引检查
# ============================================================

saved_tp_scaler = joblib.load(
    FIRST_TASK_REQUIRED_PATHS[
        "tp_scaler"
    ]
)

saved_target_scaler = joblib.load(
    FIRST_TASK_REQUIRED_PATHS[
        "target_scaler"
    ]
)


if fitted_sample_count(
    saved_tp_scaler
) != 6918:
    raise RuntimeError(
        "保存的T/P标准化器"
        "拟合样本数不是6918。"
    )


if fitted_sample_count(
    saved_target_scaler
) != 6918:
    raise RuntimeError(
        "保存的目标标准化器"
        "拟合样本数不是6918。"
    )


with np.load(
    FIRST_TASK_REQUIRED_PATHS[
        "mpnn_internal_indices"
    ],
    allow_pickle=False,
) as internal_indices_file:

    internal_index_keys = set(
        internal_indices_file.files
    )


    expected_internal_index_keys = {
        "inner_train_relative_to_outer_train_idx",
        "inner_valid_relative_to_outer_train_idx",
        "mpnn_supervised_train_relative_to_inner_train_idx",
        "mpnn_early_stop_relative_to_inner_train_idx",
        "inner_train_global_idx",
        "inner_valid_global_idx",
        "mpnn_supervised_train_global_idx",
        "mpnn_early_stop_global_idx",
    }


    if (
        internal_index_keys
        != expected_internal_index_keys
    ):
        raise KeyError(
            "内部索引文件键名不完整。\n"
            f"实际={sorted(internal_index_keys)}\n"
            f"预期={sorted(expected_internal_index_keys)}"
        )


    saved_inner_train_global_idx = (
        internal_indices_file[
            "inner_train_global_idx"
        ].astype(
            np.int64
        )
    )


    saved_inner_valid_global_idx = (
        internal_indices_file[
            "inner_valid_global_idx"
        ].astype(
            np.int64
        )
    )


    saved_mpnn_train_global_idx = (
        internal_indices_file[
            "mpnn_supervised_train_global_idx"
        ].astype(
            np.int64
        )
    )


    saved_mpnn_stop_global_idx = (
        internal_indices_file[
            "mpnn_early_stop_global_idx"
        ].astype(
            np.int64
        )
    )


if len(
    saved_inner_train_global_idx
) != 8648:
    raise RuntimeError(
        "内部索引中的内层训练行数不是8648。"
    )


if len(
    saved_inner_valid_global_idx
) != 4324:
    raise RuntimeError(
        "内部索引中的内层验证行数不是4324。"
    )


if len(
    saved_mpnn_train_global_idx
) != 6918:
    raise RuntimeError(
        "内部索引中的MPNN训练行数不是6918。"
    )


if len(
    saved_mpnn_stop_global_idx
) != 1730:
    raise RuntimeError(
        "内部索引中的MPNN早停行数不是1730。"
    )


index_overlap_checks = {
    "inner_train_valid_overlap": int(
        np.intersect1d(
            saved_inner_train_global_idx,
            saved_inner_valid_global_idx,
        ).size
    ),

    "mpnn_train_stop_overlap": int(
        np.intersect1d(
            saved_mpnn_train_global_idx,
            saved_mpnn_stop_global_idx,
        ).size
    ),

    "mpnn_train_inner_valid_overlap": int(
        np.intersect1d(
            saved_mpnn_train_global_idx,
            saved_inner_valid_global_idx,
        ).size
    ),

    "mpnn_stop_inner_valid_overlap": int(
        np.intersect1d(
            saved_mpnn_stop_global_idx,
            saved_inner_valid_global_idx,
        ).size
    ),
}


if any(
    value != 0
    for value in index_overlap_checks.values()
):
    raise RuntimeError(
        "第一套任务内部索引存在重叠："
        f"{index_overlap_checks}"
    )


mpnn_internal_union = np.sort(
    np.concatenate(
        [
            saved_mpnn_train_global_idx,
            saved_mpnn_stop_global_idx,
        ]
    )
)


if not np.array_equal(
    mpnn_internal_union,
    np.sort(
        saved_inner_train_global_idx
    ),
):
    raise RuntimeError(
        "MPNN监督训练集和早停集"
        "没有完整覆盖内层ML训练集。"
    )


# ============================================================
# 5. 清单中的目标列边界检查
# ============================================================

saved_inner_train_manifest = (
    read_csv_robust(
        FIRST_TASK_REQUIRED_PATHS[
            "inner_train_manifest"
        ]
    )
)


saved_inner_valid_structure_manifest = (
    read_csv_robust(
        FIRST_TASK_REQUIRED_PATHS[
            "inner_valid_structure_manifest"
        ]
    )
)


if "lg_eta" not in (
    saved_inner_train_manifest.columns
):
    raise RuntimeError(
        "内层训练清单缺少lg_eta。"
    )


for forbidden_target_column in [
    "eta",
    "lg_eta",
]:
    if forbidden_target_column in (
        saved_inner_valid_structure_manifest.columns
    ):
        raise RuntimeError(
            "无目标内层验证结构清单"
            f"意外包含{forbidden_target_column}。"
        )


if len(
    saved_inner_train_manifest
) != 8648:
    raise RuntimeError(
        "内层训练清单行数不是8648。"
    )


if len(
    saved_inner_valid_structure_manifest
) != 4324:
    raise RuntimeError(
        "无目标内层验证结构清单"
        "行数不是4324。"
    )


# ============================================================
# 6. 模拟下游调参Notebook读取四个最小输入
# ============================================================

X_train_for_tuning = np.load(
    FIRST_TASK_REQUIRED_PATHS[
        "X_inner_train_mpnn"
    ],
    allow_pickle=False,
)

X_valid_for_tuning = np.load(
    FIRST_TASK_REQUIRED_PATHS[
        "X_inner_valid_mpnn"
    ],
    allow_pickle=False,
)

y_train_for_tuning = np.load(
    FIRST_TASK_REQUIRED_PATHS[
        "y_inner_train_lg_eta"
    ],
    allow_pickle=False,
)

y_valid_for_tuning = np.load(
    FIRST_TASK_REQUIRED_PATHS[
        "y_inner_valid_lg_eta"
    ],
    allow_pickle=False,
)


expected_downstream_shapes = {
    "X_train": (
        8648,
        130,
    ),

    "X_valid": (
        4324,
        130,
    ),

    "y_train": (
        8648,
    ),

    "y_valid": (
        4324,
    ),
}


actual_downstream_shapes = {
    "X_train": (
        X_train_for_tuning.shape
    ),

    "X_valid": (
        X_valid_for_tuning.shape
    ),

    "y_train": (
        y_train_for_tuning.shape
    ),

    "y_valid": (
        y_valid_for_tuning.shape
    ),
}


if (
    actual_downstream_shapes
    != expected_downstream_shapes
):
    raise RuntimeError(
        "下游调参接口形状错误：\n"
        f"实际={actual_downstream_shapes}\n"
        f"预期={expected_downstream_shapes}"
    )


if not all(
    np.isfinite(array).all()
    for array in [
        X_train_for_tuning,
        X_valid_for_tuning,
        y_train_for_tuning,
        y_valid_for_tuning,
    ]
):
    raise RuntimeError(
        "下游调参输入中存在NaN或无穷值。"
    )


# 模拟后续Transformer等模型的折内标准化。
# 这里只测试接口，不保存标准化器、不训练模型。
downstream_interface_scaler = (
    StandardScaler()
)


X_train_interface_scaled = (
    downstream_interface_scaler
    .fit_transform(
        X_train_for_tuning
    )
)


X_valid_interface_scaled = (
    downstream_interface_scaler
    .transform(
        X_valid_for_tuning
    )
)


downstream_scaler_fitted_rows = (
    fitted_sample_count(
        downstream_interface_scaler
    )
)


if (
    downstream_scaler_fitted_rows
    != 8648
):
    raise RuntimeError(
        "下游接口测试中的标准化器"
        "没有只在8648条内层训练数据上拟合。"
    )


if not (
    np.isfinite(
        X_train_interface_scaled
    ).all()
    and np.isfinite(
        X_valid_interface_scaled
    ).all()
):
    raise RuntimeError(
        "下游接口标准化结果"
        "存在NaN或无穷值。"
    )


# 检查目标值和已保存清单顺序一致。
expected_y_train = (
    saved_inner_train_manifest[
        "lg_eta"
    ]
    .to_numpy(
        dtype=np.float32
    )
)


expected_y_valid = (
    TEST_OUTER_TRAIN_MANIFEST.iloc[
        TEST_INNER_VALID_RELATIVE_IDX
    ][
        "lg_eta"
    ]
    .to_numpy(
        dtype=np.float32
    )
)


train_target_order_match = bool(
    np.array_equal(
        y_train_for_tuning,
        expected_y_train,
    )
)


valid_target_order_match = bool(
    np.array_equal(
        y_valid_for_tuning,
        expected_y_valid,
    )
)


if not train_target_order_match:
    raise RuntimeError(
        "下游接口中的训练目标顺序不正确。"
    )


if not valid_target_order_match:
    raise RuntimeError(
        "下游接口中的验证目标顺序不正确。"
    )


# ============================================================
# 7. 文件清单和哈希
# ============================================================

FIRST_TASK_INVENTORY_PATH = (
    TEST_TASK_OUTPUT_DIR
    / "first_task_output_inventory.csv"
)

FIRST_TASK_COMPLETE_AUDIT_PATH = (
    TEST_TASK_OUTPUT_DIR
    / "first_task_complete_audit.json"
)

DOWNSTREAM_INTERFACE_MANIFEST_PATH = (
    TEST_TASK_OUTPUT_DIR
    / "downstream_tuning_interface_manifest.json"
)


inventory_rows = []


for (
    file_role,
    file_path,
) in FIRST_TASK_REQUIRED_PATHS.items():

    file_path = Path(
        file_path
    )


    inventory_rows.append(
        {
            "Task_id": (
                expected_task_id
            ),

            "File_role": (
                file_role
            ),

            "File_name": (
                file_path.name
            ),

            "Full_path": str(
                file_path
            ),

            "Size_bytes": int(
                file_path.stat().st_size
            ),

            "SHA256": (
                calculate_sha256(
                    file_path
                )
            ),

            "Exists": True,
        }
    )


df_first_task_inventory = (
    pd.DataFrame(
        inventory_rows
    )
)


df_first_task_inventory.to_csv(
    FIRST_TASK_INVENTORY_PATH,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 8. 下游调参读取接口manifest
# ============================================================

downstream_interface_manifest = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "status": (
        "downstream_interface_passed"
    ),

    "task_id": (
        expected_task_id
    ),

    "allowed_downstream_inputs": {
        "X_inner_train_mpnn": str(
            FIRST_TASK_REQUIRED_PATHS[
                "X_inner_train_mpnn"
            ]
        ),

        "X_inner_valid_mpnn": str(
            FIRST_TASK_REQUIRED_PATHS[
                "X_inner_valid_mpnn"
            ]
        ),

        "y_inner_train_lg_eta": str(
            FIRST_TASK_REQUIRED_PATHS[
                "y_inner_train_lg_eta"
            ]
        ),

        "y_inner_valid_lg_eta": str(
            FIRST_TASK_REQUIRED_PATHS[
                "y_inner_valid_lg_eta"
            ]
        ),
    },

    "input_shapes": {
        key: list(
            value
        )
        for key, value
        in actual_downstream_shapes.items()
    },

    "input_sha256": {
        file_role: calculate_sha256(
            FIRST_TASK_REQUIRED_PATHS[
                file_role
            ]
        )

        for file_role in [
            "X_inner_train_mpnn",
            "X_inner_valid_mpnn",
            "y_inner_train_lg_eta",
            "y_inner_valid_lg_eta",
        ]
    },

    "standardization_policy": {
        "interface_test_performed": True,

        "scaler_fitted_only_on_"
        "inner_training_features": True,

        "scaler_fitted_rows": int(
            downstream_scaler_fitted_rows
        ),

        "inner_validation_used_for_"
        "scaler_fit": False,

        "note": (
            "The final tuning notebook will apply "
            "model-specific preprocessing. "
            "This test only verifies that a scaler "
            "can be fitted on inner training rows "
            "and applied to inner validation rows."
        ),
    },

    "target_order_checks": {
        "train_target_order_match": (
            train_target_order_match
        ),

        "valid_target_order_match": (
            valid_target_order_match
        ),
    },

    "forbidden_downstream_inputs": [
        "X_outer_train_mpnn.npy",
        "X_outer_test_mpnn.npy",
        "y_outer_test_lg_eta.npy",
        "mpnn_features_eta.npy",
        "external_10point_holdout",
    ],

    "leakage_control": {
        "inner_validation_used_for_"
        "mpnn_training": False,

        "inner_validation_used_for_"
        "mpnn_early_stopping": False,

        "inner_validation_target_read_"
        "only_after_feature_extraction": True,

        "outer_test_structure_read": False,

        "outer_test_target_read": False,

        "outer_test_feature_read": False,

        "external_10point_holdout_read": False,
    },
}


save_json_atomic(
    data=(
        downstream_interface_manifest
    ),
    output_path=(
        DOWNSTREAM_INTERFACE_MANIFEST_PATH
    ),
)


# ============================================================
# 9. 第一套任务总审计
# ============================================================

first_task_complete_audit = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "audit_status": "passed",

    "task_id": (
        expected_task_id
    ),

    "stage_status_checks": (
        expected_status_checks
    ),

    "identity_checks": (
        json_identity_checks
    ),

    "training_checks": {
        "history_rows": int(
            len(
                training_history
            )
        ),

        "final_epoch": int(
            final_epoch
        ),

        "best_epoch": int(
            best_epoch
        ),

        "best_early_stop_loss": float(
            best_early_stop_loss
        ),

        "training_rows_per_epoch": 6918,

        "early_stop_rows_per_epoch": 1730,

        "inner_validation_rows_used_per_epoch": 0,

        "outer_test_rows_used_per_epoch": 0,

        "checkpoint_removed": bool(
            not checkpoint_path.exists()
        ),
    },

    "index_overlap_checks": (
        index_overlap_checks
    ),

    "scaler_checks": {
        "mpnn_tp_scaler_fitted_rows": (
            fitted_sample_count(
                saved_tp_scaler
            )
        ),

        "mpnn_target_scaler_fitted_rows": (
            fitted_sample_count(
                saved_target_scaler
            )
        ),

        "downstream_interface_scaler_"
        "fitted_rows": int(
            downstream_scaler_fitted_rows
        ),
    },

    "output_checks": {
        "required_file_count": int(
            len(
                FIRST_TASK_REQUIRED_PATHS
            )
        ),

        "all_required_files_exist": True,

        "feature_metadata_status": (
            feature_metadata.get(
                "status"
            )
        ),

        "output_audit_status": (
            output_audit.get(
                "audit_status"
            )
        ),

        "downstream_interface_status": (
            downstream_interface_manifest[
                "status"
            ]
        ),

        "train_target_order_match": (
            train_target_order_match
        ),

        "valid_target_order_match": (
            valid_target_order_match
        ),
    },

    "leakage_control": {
        "inner_validation_used_for_"
        "mpnn_backpropagation": False,

        "inner_validation_used_for_"
        "mpnn_early_stopping": False,

        "inner_validation_used_for_"
        "mpnn_scheduler": False,

        "inner_validation_target_used_"
        "during_feature_extraction": False,

        "outer_test_structure_read": False,

        "outer_test_target_read": False,

        "outer_test_feature_read": False,

        "existing_outer_mpnn_features_read": (
            False
        ),

        "old_global_mpnn_feature_read": False,

        "external_10point_holdout_read": False,
    },

    "outputs": {
        "inventory": str(
            FIRST_TASK_INVENTORY_PATH
        ),

        "complete_audit": str(
            FIRST_TASK_COMPLETE_AUDIT_PATH
        ),

        "downstream_interface_manifest": str(
            DOWNSTREAM_INTERFACE_MANIFEST_PATH
        ),
    },
}


if not all(
    first_task_complete_audit[
        "stage_status_checks"
    ].values()
):
    raise RuntimeError(
        "第一套任务阶段状态检查未全部通过。"
    )


if not all(
    first_task_complete_audit[
        "identity_checks"
    ].values()
):
    raise RuntimeError(
        "第一套任务身份检查未全部通过。"
    )


if any(
    value != 0
    for value in index_overlap_checks.values()
):
    raise RuntimeError(
        "第一套任务仍存在索引重叠。"
    )


save_json_atomic(
    data=(
        first_task_complete_audit
    ),
    output_path=(
        FIRST_TASK_COMPLETE_AUDIT_PATH
    ),
)


# ============================================================
# 10. 清理接口测试临时对象
# ============================================================

del (
    X_train_interface_scaled,
    X_valid_interface_scaled,
    downstream_interface_scaler,
)

gc.collect()


# ============================================================
# 11. 输出结果
# ============================================================

print("=" * 100)
print("Cell 7 第一套内层MPNN任务总审计和下游接口测试通过 ✓")
print("=" * 100)

print(
    "task id                         =",
    expected_task_id,
)

print(
    "required output files           =",
    len(
        FIRST_TASK_REQUIRED_PATHS
    ),
)

print(
    "training history rows           =",
    len(
        training_history
    ),
)

print(
    "final epoch                     =",
    final_epoch,
)

print(
    "best epoch                      =",
    best_epoch,
)

print(
    "best early-stop loss            =",
    best_early_stop_loss,
)

print(
    "MPNN T/P scaler fitted rows     =",
    fitted_sample_count(
        saved_tp_scaler
    ),
)

print(
    "MPNN target scaler fitted rows  =",
    fitted_sample_count(
        saved_target_scaler
    ),
)

print(
    "downstream scaler fitted rows   =",
    downstream_scaler_fitted_rows,
)

print(
    "inner train/valid overlap       =",
    index_overlap_checks[
        "inner_train_valid_overlap"
    ],
)

print(
    "MPNN train/stop overlap         =",
    index_overlap_checks[
        "mpnn_train_stop_overlap"
    ],
)

print(
    "MPNN train/inner valid overlap  =",
    index_overlap_checks[
        "mpnn_train_inner_valid_overlap"
    ],
)

print(
    "MPNN stop/inner valid overlap   =",
    index_overlap_checks[
        "mpnn_stop_inner_valid_overlap"
    ],
)

print(
    "train target order match        =",
    train_target_order_match,
)

print(
    "valid target order match        =",
    valid_target_order_match,
)

print(
    "checkpoint exists               =",
    checkpoint_path.exists(),
)

print(
    "downstream interface status     =",
    downstream_interface_manifest[
        "status"
    ],
)

print(
    "first task audit status         =",
    first_task_complete_audit[
        "audit_status"
    ],
)

print(
    "inner validation used by MPNN   =",
    False,
)

print(
    "outer test used                 =",
    False,
)


print(
    "\n输出文件清单："
)

print(
    FIRST_TASK_INVENTORY_PATH
)


print(
    "\n第一套任务总审计："
)

print(
    FIRST_TASK_COMPLETE_AUDIT_PATH
)


print(
    "\n下游调参接口manifest："
)

print(
    DOWNSTREAM_INTERFACE_MANIFEST_PATH
)

Cell 7 第一套内层MPNN任务总审计和下游接口测试通过 ✓
task id                         = Data_split_OuterFold1_InnerFold1
required output files           = 20
training history rows           = 150
final epoch                     = 150
best epoch                      = 146
best early-stop loss            = 0.037401667413856254
MPNN T/P scaler fitted rows     = 6918
MPNN target scaler fitted rows  = 6918
downstream scaler fitted rows   = 8648
inner train/valid overlap       = 0
MPNN train/stop overlap         = 0
MPNN train/inner valid overlap  = 0
MPNN stop/inner valid overlap   = 0
train target order match        = True
valid target order match        = True
checkpoint exists               = False
downstream interface status     = downstream_interface_passed
first task audit status         = passed
inner validation used by MPNN   = False
outer test used                 = False

输出文件清单：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\in

In [9]:
# Cell 8 | 一次性、可恢复地完成其余全部内层MPNN训练与表征提取


# 设置为30的原因：
# 代码会自动跳过已经完成的任务，并选择全部尚未完成的任务。
# 当前第一套任务已完成，因此本次实际预计运行29套。
MAX_NEW_TASKS_THIS_RUN = 30


BATCH_STATUS_TABLE_PATH = (
    INNER_MPNN_LOG_DIR
    / "inner_mpnn_batch_task_status_Data_IL.csv"
)

BATCH_RUN_STATE_PATH = (
    INNER_MPNN_LOG_DIR
    / "inner_mpnn_batch_run_state_Data_IL.json"
)


# ============================================================
# 1. 通用保存函数
# ============================================================

def batch_save_json_atomic(
    data,
    output_path,
):
    """原子方式保存JSON。"""
    output_path = Path(
        output_path
    )

    temporary_path = (
        output_path.parent
        / (
            output_path.name
            + ".tmp"
        )
    )

    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            to_jsonable(
                data
            ),
            file,
            ensure_ascii=False,
            indent=2,
        )

    os.replace(
        temporary_path,
        output_path,
    )


def batch_save_torch_atomic(
    data,
    output_path,
):
    """原子方式保存PyTorch文件。"""
    output_path = Path(
        output_path
    )

    temporary_path = (
        output_path.parent
        / (
            output_path.name
            + ".tmp"
        )
    )

    torch.save(
        data,
        temporary_path,
    )

    os.replace(
        temporary_path,
        output_path,
    )


def batch_save_numpy_atomic(
    array,
    output_path,
):
    """原子方式保存NumPy数组。"""
    output_path = Path(
        output_path
    )

    temporary_path = (
        output_path.parent
        / (
            output_path.name
            + ".tmp"
        )
    )

    with open(
        temporary_path,
        "wb",
    ) as file:
        np.save(
            file,
            array,
            allow_pickle=False,
        )

    os.replace(
        temporary_path,
        output_path,
    )


def batch_clone_state_dict_to_cpu(
    state_dict,
):
    """复制模型状态至CPU。"""
    return {
        key: (
            value
            .detach()
            .cpu()
            .clone()
        )
        for key, value
        in state_dict.items()
    }


def batch_move_optimizer_state_to_device(
    optimizer,
    device,
):
    """将恢复后的优化器状态移至当前设备。"""
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(
                value
            ):
                state[key] = value.to(
                    device
                )


# ============================================================
# 2. 每套任务核心文件
# ============================================================

def get_task_core_paths(
    split_name,
    outer_fold,
    inner_fold,
):
    """返回一套任务应保存的20个核心文件。"""
    output_dir = (
        get_inner_task_output_dir(
            split_name=split_name,
            outer_fold=outer_fold,
            inner_fold=inner_fold,
        )
    )

    return {
        "inner_train_manifest": (
            output_dir
            / "inner_train_manifest.csv"
        ),

        "inner_valid_structure_manifest": (
            output_dir
            / "inner_valid_structure_manifest_no_targets.csv"
        ),

        "mpnn_supervised_train_manifest": (
            output_dir
            / "mpnn_supervised_train_manifest.csv"
        ),

        "mpnn_early_stop_manifest": (
            output_dir
            / "mpnn_early_stop_manifest.csv"
        ),

        "mpnn_internal_indices": (
            output_dir
            / "mpnn_internal_indices.npz"
        ),

        "tp_scaler": (
            output_dir
            / "tp_scaler.joblib"
        ),

        "target_scaler": (
            output_dir
            / "target_scaler.joblib"
        ),

        "preprocessing_audit": (
            output_dir
            / "mpnn_preprocessing_audit.json"
        ),

        "dataloader_audit": (
            output_dir
            / "mpnn_dataloader_audit.json"
        ),

        "model_structure_audit": (
            output_dir
            / "mpnn_model_structure_audit.json"
        ),

        "training_history": (
            output_dir
            / "mpnn_training_history.csv"
        ),

        "training_summary": (
            output_dir
            / "mpnn_training_summary.json"
        ),

        "training_progress": (
            output_dir
            / "mpnn_training_progress.json"
        ),

        "best_state": (
            output_dir
            / "mpnn_best_state.pt"
        ),

        "X_inner_train_mpnn": (
            output_dir
            / "X_inner_train_mpnn.npy"
        ),

        "X_inner_valid_mpnn": (
            output_dir
            / "X_inner_valid_mpnn.npy"
        ),

        "y_inner_train_lg_eta": (
            output_dir
            / "y_inner_train_lg_eta.npy"
        ),

        "y_inner_valid_lg_eta": (
            output_dir
            / "y_inner_valid_lg_eta.npy"
        ),

        "inner_feature_metadata": (
            output_dir
            / "inner_feature_metadata.json"
        ),

        "inner_output_audit": (
            output_dir
            / "inner_output_audit.json"
        ),
    }


def inspect_task_completion(
    split_name,
    outer_fold,
    inner_fold,
):
    """判断任务当前处于完成、断点或待运行状态。"""
    task_id = (
        f"{split_name}_"
        f"OuterFold{outer_fold}_"
        f"InnerFold{inner_fold}"
    )

    paths = get_task_core_paths(
        split_name=split_name,
        outer_fold=outer_fold,
        inner_fold=inner_fold,
    )

    output_dir = (
        get_inner_task_output_dir(
            split_name=split_name,
            outer_fold=outer_fold,
            inner_fold=inner_fold,
        )
    )

    checkpoint_path = (
        output_dir
        / "mpnn_training_checkpoint.pt"
    )

    all_core_files_exist = all(
        path.exists()
        for path in paths.values()
    )

    output_audit_passed = False

    if paths[
        "inner_output_audit"
    ].exists():
        try:
            audit = read_json(
                paths[
                    "inner_output_audit"
                ]
            )

            output_audit_passed = bool(
                audit.get(
                    "audit_status"
                )
                == "passed"
                and audit.get(
                    "task_id"
                )
                == task_id
            )

        except Exception:
            output_audit_passed = False

    training_completed = False

    if paths[
        "training_summary"
    ].exists():
        try:
            summary = read_json(
                paths[
                    "training_summary"
                ]
            )

            training_completed = bool(
                summary.get(
                    "status"
                )
                == "training_completed"
                and summary.get(
                    "task_id"
                )
                == task_id
            )

        except Exception:
            training_completed = False

    if (
        all_core_files_exist
        and output_audit_passed
    ):
        status = "completed"

    elif checkpoint_path.exists():
        status = "checkpoint_available"

    elif training_completed:
        status = (
            "training_completed_"
            "feature_extraction_pending"
        )

    elif output_dir.exists():
        status = "partial_outputs"

    else:
        status = "pending"

    return {
        "task_id": task_id,
        "status": status,

        "completed": bool(
            status == "completed"
        ),

        "checkpoint_exists": bool(
            checkpoint_path.exists()
        ),

        "all_core_files_exist": bool(
            all_core_files_exist
        ),

        "training_completed": bool(
            training_completed
        ),

        "output_audit_passed": bool(
            output_audit_passed
        ),

        "output_dir": output_dir,
        "paths": paths,
        "checkpoint_path": checkpoint_path,
    }


# ============================================================
# 3. 每套任务DataLoader与模型结构审计
# ============================================================

def save_generic_task_interface_audits(
    preprocessing_context,
    dataloader_context,
):
    """保存一套任务的DataLoader与模型结构审计。"""
    task_id = (
        preprocessing_context[
            "task_id"
        ]
    )

    output_dir = Path(
        preprocessing_context[
            "path_map"
        ][
            "preprocessing_audit"
        ]
    ).parent

    dataloader_audit_path = (
        output_dir
        / "mpnn_dataloader_audit.json"
    )

    model_structure_audit_path = (
        output_dir
        / "mpnn_model_structure_audit.json"
    )

    (
        train_cation_batch,
        train_anion_batch,
        train_environment_batch,
        train_target_batch,
    ) = next(
        iter(
            dataloader_context[
                "mpnn_train_eval_loader"
            ]
        )
    )

    (
        valid_cation_batch,
        valid_anion_batch,
        valid_environment_batch,
    ) = next(
        iter(
            dataloader_context[
                "inner_valid_feature_loader"
            ]
        )
    )

    if hasattr(
        dataloader_context[
            "inner_valid_feature_dataset"
        ],
        "target_tensor",
    ):
        raise RuntimeError(
            f"{task_id}：内层验证特征数据集"
            "意外包含目标张量。"
        )

    dataloader_audit = {
        "created_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),

        "run_code_version": (
            RUN_CODE_VERSION
        ),

        "status": "dataloader_passed",

        "task_id": task_id,

        "counts": (
            dataloader_context[
                "counts"
            ]
        ),

        "loader_batches": {
            "mpnn_train_batches": int(
                len(
                    dataloader_context[
                        "mpnn_train_loader"
                    ]
                )
            ),

            "mpnn_early_stop_batches": int(
                len(
                    dataloader_context[
                        "mpnn_early_stop_loader"
                    ]
                )
            ),

            "inner_train_feature_batches": int(
                len(
                    dataloader_context[
                        "inner_train_feature_loader"
                    ]
                )
            ),

            "inner_valid_feature_batches": int(
                len(
                    dataloader_context[
                        "inner_valid_feature_loader"
                    ]
                )
            ),
        },

        "first_supervised_batch": {
            "environment_shape": list(
                train_environment_batch.shape
            ),

            "target_shape": list(
                train_target_batch.shape
            ),
        },

        "first_inner_valid_feature_batch": {
            "environment_shape": list(
                valid_environment_batch.shape
            ),

            "returned_items": 3,

            "target_returned": False,
        },

        "leakage_control": {
            "inner_valid_dataset_has_target_tensor": False,
            "inner_valid_loader_returns_target": False,
            "outer_test_dataset_created": False,
            "outer_test_loader_created": False,
            "outer_test_target_tensor_created": False,
        },
    }

    batch_save_json_atomic(
        data=dataloader_audit,
        output_path=dataloader_audit_path,
    )

    temporary_model = (
        build_fresh_mpnn_model(
            task_seed=int(
                preprocessing_context[
                    "task_seed"
                ]
            )
        )
    )

    parameter_counts = (
        count_model_parameters(
            temporary_model
        )
    )

    if (
        parameter_counts[
            "total_parameters"
        ]
        != 298817
        or parameter_counts[
            "trainable_parameters"
        ]
        != 298817
    ):
        raise RuntimeError(
            f"{task_id}：MPNN参数量异常。"
        )

    train_cation_batch = (
        train_cation_batch.to(
            DEVICE
        )
    )

    train_anion_batch = (
        train_anion_batch.to(
            DEVICE
        )
    )

    train_environment_batch = (
        train_environment_batch.to(
            DEVICE
        )
    )

    valid_cation_batch = (
        valid_cation_batch.to(
            DEVICE
        )
    )

    valid_anion_batch = (
        valid_anion_batch.to(
            DEVICE
        )
    )

    valid_environment_batch = (
        valid_environment_batch.to(
            DEVICE
        )
    )

    temporary_model.eval()

    with torch.no_grad():
        train_feature_batch = (
            temporary_model.extract_features(
                train_cation_batch,
                train_anion_batch,
                train_environment_batch,
            )
        )

        valid_feature_batch = (
            temporary_model.extract_features(
                valid_cation_batch,
                valid_anion_batch,
                valid_environment_batch,
            )
        )

        prediction_batch = (
            temporary_model(
                train_cation_batch,
                train_anion_batch,
                train_environment_batch,
            )
        )

    if (
        train_feature_batch.shape[1] != 130
        or valid_feature_batch.shape[1] != 130
        or prediction_batch.shape[1] != 1
    ):
        raise RuntimeError(
            f"{task_id}：模型前向传播维度异常。"
        )

    all_forward_values_finite = bool(
        torch.isfinite(
            train_feature_batch
        ).all()
        and torch.isfinite(
            valid_feature_batch
        ).all()
        and torch.isfinite(
            prediction_batch
        ).all()
    )

    if not all_forward_values_finite:
        raise RuntimeError(
            f"{task_id}：模型前向传播出现NaN或无穷值。"
        )

    model_structure_audit = {
        "created_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),

        "run_code_version": (
            RUN_CODE_VERSION
        ),

        "status": "model_structure_passed",

        "task_id": task_id,

        "architecture": {
            "cation_embedding_dim": 64,
            "anion_embedding_dim": 64,
            "environment_dim": 2,
            "descriptor_dim": 130,
            "prediction_dim": 1,
            "total_parameters": 298817,
            "trainable_parameters": 298817,
        },

        "forward_checks": {
            "train_feature_shape": list(
                train_feature_batch.shape
            ),

            "valid_feature_shape": list(
                valid_feature_batch.shape
            ),

            "prediction_shape": list(
                prediction_batch.shape
            ),

            "all_finite": (
                all_forward_values_finite
            ),
        },

        "leakage_control": {
            "model_trained_in_structure_check": False,
            "backward_called": False,
            "inner_validation_target_used": False,
            "outer_test_target_read": False,
        },
    }

    batch_save_json_atomic(
        data=model_structure_audit,
        output_path=model_structure_audit_path,
    )

    del (
        temporary_model,
        train_cation_batch,
        train_anion_batch,
        train_environment_batch,
        train_target_batch,
        valid_cation_batch,
        valid_anion_batch,
        valid_environment_batch,
        train_feature_batch,
        valid_feature_batch,
        prediction_batch,
    )

    gc.collect()

    return {
        "dataloader_audit": (
            dataloader_audit_path
        ),

        "model_structure_audit": (
            model_structure_audit_path
        ),
    }


# ============================================================
# 4. 通用、可恢复的内层MPNN训练函数
# ============================================================

def train_inner_mpnn_batch_task(
    preprocessing_context,
    dataloader_context,
):
    """训练或恢复一套内层MPNN。"""
    task_id = (
        preprocessing_context[
            "task_id"
        ]
    )

    task_seed = int(
        preprocessing_context[
            "task_seed"
        ]
    )

    output_dir = Path(
        preprocessing_context[
            "path_map"
        ][
            "preprocessing_audit"
        ]
    ).parent

    history_path = (
        output_dir
        / "mpnn_training_history.csv"
    )

    best_state_path = (
        output_dir
        / "mpnn_best_state.pt"
    )

    summary_path = (
        output_dir
        / "mpnn_training_summary.json"
    )

    progress_path = (
        output_dir
        / "mpnn_training_progress.json"
    )

    checkpoint_path = (
        output_dir
        / "mpnn_training_checkpoint.pt"
    )

    expected_train_rows = int(
        preprocessing_context[
            "preprocessing_audit"
        ][
            "counts"
        ][
            "mpnn_supervised_train_rows"
        ]
    )

    expected_stop_rows = int(
        preprocessing_context[
            "preprocessing_audit"
        ][
            "counts"
        ][
            "mpnn_early_stop_rows"
        ]
    )

    # 已完成训练但尚未提取表征时，直接加载最佳状态。
    if summary_path.exists():
        existing_summary = read_json(
            summary_path
        )

        if (
            existing_summary.get(
                "status"
            )
            == "training_completed"
            and existing_summary.get(
                "task_id"
            )
            == task_id
        ):
            if not best_state_path.exists():
                raise FileNotFoundError(
                    f"{task_id}：训练摘要显示已完成，"
                    "但缺少最佳模型文件。"
                )

            best_payload = load_torch_object(
                best_state_path
            )

            completed_model = (
                build_fresh_mpnn_model(
                    task_seed=task_seed
                )
            )

            completed_model.load_state_dict(
                best_payload[
                    "model_state_dict"
                ],
                strict=True,
            )

            completed_model.to(
                DEVICE
            )

            completed_model.eval()

            return {
                "model": completed_model,

                "training_summary": (
                    existing_summary
                ),

                "resumed_completed_training": True,
            }

    seed_everything(
        task_seed
    )

    model = build_fresh_mpnn_model(
        task_seed=task_seed
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=INITIAL_LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    criterion = nn.MSELoss()

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=8,
            min_lr=1e-7,
        )
    )

    train_dataset = (
        dataloader_context[
            "mpnn_supervised_train_dataset"
        ]
    )

    stop_loader = (
        dataloader_context[
            "mpnn_early_stop_loader"
        ]
    )

    if len(
        train_dataset
    ) != expected_train_rows:
        raise RuntimeError(
            f"{task_id}：监督训练数据集行数异常。"
        )

    if len(
        stop_loader.dataset
    ) != expected_stop_rows:
        raise RuntimeError(
            f"{task_id}：早停数据集行数异常。"
        )

    start_epoch = 1
    best_epoch = 0
    best_loss = float("inf")
    best_state_dict = None
    no_improve_count = 0
    history_rows = []
    cumulative_minutes_before = 0.0
    resumed = False
    resumed_after_epoch = 0

    checkpoint = None

    if checkpoint_path.exists():
        checkpoint = load_torch_object(
            checkpoint_path
        )

        if (
            checkpoint.get(
                "task_id"
            )
            != task_id
            or checkpoint.get(
                "run_code_version"
            )
            != RUN_CODE_VERSION
        ):
            raise RuntimeError(
                f"{task_id}：训练断点身份"
                "或代码版本不一致。"
            )

        model.load_state_dict(
            checkpoint[
                "model_state_dict"
            ]
        )

        optimizer.load_state_dict(
            checkpoint[
                "optimizer_state_dict"
            ]
        )

        batch_move_optimizer_state_to_device(
            optimizer=optimizer,
            device=DEVICE,
        )

        scheduler.load_state_dict(
            checkpoint[
                "scheduler_state_dict"
            ]
        )

        resumed_after_epoch = int(
            checkpoint[
                "epoch_completed"
            ]
        )

        start_epoch = (
            resumed_after_epoch
            + 1
        )

        best_epoch = int(
            checkpoint[
                "best_epoch"
            ]
        )

        best_loss = float(
            checkpoint[
                "best_val_loss"
            ]
        )

        best_state_dict = (
            checkpoint[
                "best_state_dict"
            ]
        )

        no_improve_count = int(
            checkpoint[
                "no_improve_count"
            ]
        )

        history_rows = list(
            checkpoint[
                "training_history"
            ]
        )

        cumulative_minutes_before = float(
            checkpoint[
                "cumulative_training_minutes"
            ]
        )

        resumed = True

    train_generator = torch.Generator()

    if resumed:
        train_generator.set_state(
            checkpoint[
                "train_generator_state"
            ]
        )

    else:
        train_generator.manual_seed(
            task_seed
        )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=(
            collate_with_target
        ),
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        generator=train_generator,
        drop_last=False,
    )

    if start_epoch > MAX_EPOCHS:
        raise RuntimeError(
            f"{task_id}：断点中的Epoch"
            "已达到最大训练轮数。"
        )

    session_start = time.perf_counter()
    stopped_early = False
    epochs_run_this_session = 0

    print("\n" + "=" * 100)
    print(f"开始训练：{task_id}")
    print("=" * 100)

    print(
        "supervised train rows =",
        expected_train_rows,
    )

    print(
        "early-stop rows       =",
        expected_stop_rows,
    )

    print(
        "inner valid rows used =",
        0,
    )

    print(
        "outer test rows used  =",
        0,
    )

    print(
        "resume checkpoint     =",
        resumed,
    )

    print(
        "resume after epoch    =",
        resumed_after_epoch,
    )

    print(
        "start epoch           =",
        start_epoch,
    )

    for epoch in range(
        start_epoch,
        MAX_EPOCHS + 1,
    ):
        epochs_run_this_session += 1

        # ----------------------------------------------------
        # 监督训练阶段
        # ----------------------------------------------------

        model.train()

        train_loss_sum = 0.0
        train_rows_seen = 0

        for (
            cation_batch,
            anion_batch,
            environment_batch,
            target_batch,
        ) in train_loader:

            cation_batch = (
                cation_batch.to(
                    DEVICE
                )
            )

            anion_batch = (
                anion_batch.to(
                    DEVICE
                )
            )

            environment_batch = (
                environment_batch.to(
                    DEVICE
                )
            )

            target_batch = (
                target_batch.to(
                    DEVICE
                )
            )

            if not torch.isfinite(
                target_batch
            ).all():
                raise RuntimeError(
                    f"{task_id} Epoch {epoch}："
                    "训练目标出现NaN或无穷值。"
                )

            optimizer.zero_grad(
                set_to_none=True
            )

            prediction = model(
                cation_batch,
                anion_batch,
                environment_batch,
            )

            loss = criterion(
                prediction,
                target_batch,
            )

            if not torch.isfinite(
                loss
            ):
                raise RuntimeError(
                    f"{task_id} Epoch {epoch}："
                    "训练损失异常。"
                )

            loss.backward()
            optimizer.step()

            batch_rows = int(
                target_batch.shape[0]
            )

            train_loss_sum += (
                float(
                    loss.item()
                )
                * batch_rows
            )

            train_rows_seen += (
                batch_rows
            )

        if train_rows_seen != (
            expected_train_rows
        ):
            raise RuntimeError(
                f"{task_id} Epoch {epoch}："
                "监督训练样本数错误。"
            )

        average_train_loss = (
            train_loss_sum
            / train_rows_seen
        )

        # ----------------------------------------------------
        # MPNN早停阶段
        # ----------------------------------------------------

        model.eval()

        stop_loss_sum = 0.0
        stop_rows_seen = 0

        with torch.no_grad():
            for (
                cation_batch,
                anion_batch,
                environment_batch,
                target_batch,
            ) in stop_loader:

                cation_batch = (
                    cation_batch.to(
                        DEVICE
                    )
                )

                anion_batch = (
                    anion_batch.to(
                        DEVICE
                    )
                )

                environment_batch = (
                    environment_batch.to(
                        DEVICE
                    )
                )

                target_batch = (
                    target_batch.to(
                        DEVICE
                    )
                )

                if not torch.isfinite(
                    target_batch
                ).all():
                    raise RuntimeError(
                        f"{task_id} Epoch {epoch}："
                        "早停目标出现NaN或无穷值。"
                    )

                prediction = model(
                    cation_batch,
                    anion_batch,
                    environment_batch,
                )

                stop_loss = criterion(
                    prediction,
                    target_batch,
                )

                if not torch.isfinite(
                    stop_loss
                ):
                    raise RuntimeError(
                        f"{task_id} Epoch {epoch}："
                        "早停损失异常。"
                    )

                batch_rows = int(
                    target_batch.shape[0]
                )

                stop_loss_sum += (
                    float(
                        stop_loss.item()
                    )
                    * batch_rows
                )

                stop_rows_seen += (
                    batch_rows
                )

        if stop_rows_seen != (
            expected_stop_rows
        ):
            raise RuntimeError(
                f"{task_id} Epoch {epoch}："
                "早停样本数错误。"
            )

        average_stop_loss = (
            stop_loss_sum
            / stop_rows_seen
        )

        scheduler.step(
            average_stop_loss
        )

        current_lr = float(
            optimizer.param_groups[
                0
            ][
                "lr"
            ]
        )

        improved = bool(
            average_stop_loss
            < best_loss
            - MIN_DELTA
        )

        if improved:
            best_loss = float(
                average_stop_loss
            )

            best_epoch = int(
                epoch
            )

            best_state_dict = (
                batch_clone_state_dict_to_cpu(
                    model.state_dict()
                )
            )

            no_improve_count = 0

            best_payload = {
                "property": "viscosity",

                "target_col": "lg_eta",

                "representation_method": (
                    "inner-fold-specific "
                    "supervised MPNN"
                ),

                "task_id": task_id,

                "split_name": (
                    preprocessing_context[
                        "split_name"
                    ]
                ),

                "outer_fold": int(
                    preprocessing_context[
                        "outer_fold"
                    ]
                ),

                "inner_fold": int(
                    preprocessing_context[
                        "inner_fold"
                    ]
                ),

                "task_seed": int(
                    task_seed
                ),

                "model_class": "ILPredictor",

                "encoder_class": "MPNNEncoder",

                "descriptor_dim": 130,

                "best_epoch": int(
                    best_epoch
                ),

                "best_early_stop_loss": float(
                    best_loss
                ),

                "mpnn_supervised_train_rows": int(
                    expected_train_rows
                ),

                "mpnn_early_stop_rows": int(
                    expected_stop_rows
                ),

                "inner_ml_validation_used": False,

                "outer_test_used": False,

                "model_state_dict": (
                    best_state_dict
                ),
            }

            batch_save_torch_atomic(
                data=best_payload,
                output_path=best_state_path,
            )

        else:
            no_improve_count += 1

        history_row = {
            "task_id": task_id,

            "split_name": (
                preprocessing_context[
                    "split_name"
                ]
            ),

            "outer_fold": int(
                preprocessing_context[
                    "outer_fold"
                ]
            ),

            "inner_fold": int(
                preprocessing_context[
                    "inner_fold"
                ]
            ),

            "task_seed": int(
                task_seed
            ),

            "epoch": int(
                epoch
            ),

            "train_loss": float(
                average_train_loss
            ),

            "early_stop_loss": float(
                average_stop_loss
            ),

            "best_early_stop_loss": float(
                best_loss
            ),

            "best_epoch": int(
                best_epoch
            ),

            "learning_rate": float(
                current_lr
            ),

            "improved": int(
                improved
            ),

            "no_improve_count": int(
                no_improve_count
            ),

            "train_rows_used": int(
                train_rows_seen
            ),

            "early_stop_rows_used": int(
                stop_rows_seen
            ),

            "inner_ml_valid_rows_used": 0,

            "outer_test_rows_used": 0,
        }

        history_rows.append(
            history_row
        )

        pd.DataFrame(
            history_rows
        ).to_csv(
            history_path,
            index=False,
            encoding="utf-8-sig",
        )

        cumulative_minutes = (
            cumulative_minutes_before
            + (
                time.perf_counter()
                - session_start
            ) / 60.0
        )

        checkpoint_payload = {
            "task_id": task_id,

            "run_code_version": (
                RUN_CODE_VERSION
            ),

            "epoch_completed": int(
                epoch
            ),

            "model_state_dict": (
                batch_clone_state_dict_to_cpu(
                    model.state_dict()
                )
            ),

            "optimizer_state_dict": (
                optimizer.state_dict()
            ),

            "scheduler_state_dict": (
                scheduler.state_dict()
            ),

            "best_epoch": int(
                best_epoch
            ),

            "best_val_loss": float(
                best_loss
            ),

            "best_state_dict": (
                best_state_dict
            ),

            "no_improve_count": int(
                no_improve_count
            ),

            "training_history": (
                history_rows
            ),

            "train_generator_state": (
                train_generator.get_state()
            ),

            "cumulative_training_minutes": float(
                cumulative_minutes
            ),
        }

        batch_save_torch_atomic(
            data=checkpoint_payload,
            output_path=checkpoint_path,
        )

        progress_record = {
            "updated_at": (
                datetime.now().isoformat(
                    timespec="seconds"
                )
            ),

            "status": "training",

            "task_id": task_id,

            "epoch_completed": int(
                epoch
            ),

            "best_epoch": int(
                best_epoch
            ),

            "best_early_stop_loss": float(
                best_loss
            ),

            "no_improve_count": int(
                no_improve_count
            ),

            "current_learning_rate": float(
                current_lr
            ),

            "cumulative_training_minutes": float(
                cumulative_minutes
            ),

            "checkpoint_path": str(
                checkpoint_path
            ),
        }

        batch_save_json_atomic(
            data=progress_record,
            output_path=progress_path,
        )

        if (
            epoch == start_epoch
            or epoch % 10 == 0
            or improved
            or epoch == MAX_EPOCHS
        ):
            print(
                f"{task_id} | "
                f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
                f"Train {average_train_loss:.6f} | "
                f"Stop {average_stop_loss:.6f} | "
                f"Best {best_loss:.6f} "
                f"@ {best_epoch:03d} | "
                f"LR {current_lr:.8f}"
            )

        if (
            no_improve_count
            >= EARLY_STOP_PATIENCE
        ):
            stopped_early = True

            print(
                f"{task_id} early stopping "
                f"at Epoch {epoch}; "
                f"best epoch={best_epoch}"
            )

            break

    if best_state_dict is None:
        raise RuntimeError(
            f"{task_id}：没有得到最佳模型。"
        )

    model.load_state_dict(
        best_state_dict
    )

    model.to(
        DEVICE
    )

    model.eval()

    final_epoch = int(
        history_rows[
            -1
        ][
            "epoch"
        ]
    )

    cumulative_minutes = (
        cumulative_minutes_before
        + (
            time.perf_counter()
            - session_start
        ) / 60.0
    )

    final_best_payload = {
        "property": "viscosity",

        "target_col": "lg_eta",

        "representation_method": (
            "inner-fold-specific "
            "supervised MPNN"
        ),

        "task_id": task_id,

        "split_name": (
            preprocessing_context[
                "split_name"
            ]
        ),

        "outer_fold": int(
            preprocessing_context[
                "outer_fold"
            ]
        ),

        "inner_fold": int(
            preprocessing_context[
                "inner_fold"
            ]
        ),

        "task_seed": int(
            task_seed
        ),

        "model_class": "ILPredictor",

        "encoder_class": "MPNNEncoder",

        "descriptor_dim": 130,

        "best_epoch": int(
            best_epoch
        ),

        "best_early_stop_loss": float(
            best_loss
        ),

        "mpnn_supervised_train_rows": int(
            expected_train_rows
        ),

        "mpnn_early_stop_rows": int(
            expected_stop_rows
        ),

        "inner_ml_validation_used": False,

        "outer_test_used": False,

        "model_state_dict": (
            batch_clone_state_dict_to_cpu(
                model.state_dict()
            )
        ),
    }

    batch_save_torch_atomic(
        data=final_best_payload,
        output_path=best_state_path,
    )

    training_summary = {
        "created_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),

        "run_code_version": (
            RUN_CODE_VERSION
        ),

        "status": "training_completed",

        "task_id": task_id,

        "split_name": (
            preprocessing_context[
                "split_name"
            ]
        ),

        "outer_fold": int(
            preprocessing_context[
                "outer_fold"
            ]
        ),

        "inner_fold": int(
            preprocessing_context[
                "inner_fold"
            ]
        ),

        "task_seed": int(
            task_seed
        ),

        "device": str(
            DEVICE
        ),

        "training_configuration": {
            "optimizer": "Adam",

            "initial_learning_rate": float(
                INITIAL_LEARNING_RATE
            ),

            "weight_decay": float(
                WEIGHT_DECAY
            ),

            "scheduler": (
                "ReduceLROnPlateau"
            ),

            "scheduler_factor": 0.5,

            "scheduler_patience": 8,

            "maximum_epochs": int(
                MAX_EPOCHS
            ),

            "early_stopping_patience": int(
                EARLY_STOP_PATIENCE
            ),

            "batch_size": int(
                BATCH_SIZE
            ),
        },

        "data_counts": {
            "mpnn_supervised_train_rows": int(
                expected_train_rows
            ),

            "mpnn_early_stop_rows": int(
                expected_stop_rows
            ),

            "inner_ml_valid_rows_used": 0,

            "outer_test_rows_used": 0,
        },

        "result": {
            "resumed_from_checkpoint": bool(
                resumed
            ),

            "resumed_after_epoch": int(
                resumed_after_epoch
            ),

            "epochs_run_this_session": int(
                epochs_run_this_session
            ),

            "final_epoch": int(
                final_epoch
            ),

            "best_epoch": int(
                best_epoch
            ),

            "best_early_stop_loss": float(
                best_loss
            ),

            "stopped_early": bool(
                stopped_early
            ),

            "cumulative_training_minutes": float(
                cumulative_minutes
            ),
        },

        "leakage_control": {
            "inner_validation_available_"
            "in_source_manifest": True,

            "inner_validation_used_for_"
            "backpropagation": False,

            "inner_validation_used_for_"
            "early_stopping": False,

            "inner_validation_used_for_"
            "scheduler": False,

            "outer_test_used_for_"
            "backpropagation": False,

            "outer_test_used_for_"
            "early_stopping": False,

            "outer_test_used_for_"
            "scheduler": False,

            "existing_outer_mpnn_features_read": False,

            "old_global_mpnn_feature_read": False,

            "external_10point_holdout_read": False,
        },
    }

    batch_save_json_atomic(
        data=training_summary,
        output_path=summary_path,
    )

    final_progress = {
        "updated_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),

        "status": "training_completed",

        "task_id": task_id,

        "final_epoch": int(
            final_epoch
        ),

        "best_epoch": int(
            best_epoch
        ),

        "best_early_stop_loss": float(
            best_loss
        ),

        "stopped_early": bool(
            stopped_early
        ),

        "cumulative_training_minutes": float(
            cumulative_minutes
        ),

        "checkpoint_removed": True,
    }

    batch_save_json_atomic(
        data=final_progress,
        output_path=progress_path,
    )

    if checkpoint_path.exists():
        checkpoint_path.unlink()

    return {
        "model": model,

        "training_summary": (
            training_summary
        ),

        "resumed_completed_training": False,
    }


# ============================================================
# 5. 通用表征提取和输出审计
# ============================================================

def extract_batch_task_features(
    preprocessing_context,
    dataloader_context,
    trained_model,
):
    """提取并保存一套任务的内层训练和验证表征。"""
    task_id = (
        preprocessing_context[
            "task_id"
        ]
    )

    split_name = (
        preprocessing_context[
            "split_name"
        ]
    )

    outer_fold = int(
        preprocessing_context[
            "outer_fold"
        ]
    )

    inner_fold = int(
        preprocessing_context[
            "inner_fold"
        ]
    )

    output_dir = Path(
        preprocessing_context[
            "path_map"
        ][
            "preprocessing_audit"
        ]
    ).parent

    n_train = len(
        preprocessing_context[
            "inner_train_manifest"
        ]
    )

    n_valid = len(
        preprocessing_context[
            "inner_valid_structure_manifest"
        ]
    )

    X_train = (
        extract_mpnn_features_without_targets(
            model=trained_model,

            feature_loader=(
                dataloader_context[
                    "inner_train_feature_loader"
                ]
            ),

            expected_rows=n_train,

            feature_name=(
                f"{task_id} X_inner_train"
            ),
        )
    )

    X_valid = (
        extract_mpnn_features_without_targets(
            model=trained_model,

            feature_loader=(
                dataloader_context[
                    "inner_valid_feature_loader"
                ]
            ),

            expected_rows=n_valid,

            feature_name=(
                f"{task_id} X_inner_valid"
            ),
        )
    )

    train_tp_match = bool(
        np.allclose(
            X_train[
                :,
                -2:
            ],

            preprocessing_context[
                "inner_train_tp_scaled"
            ],

            rtol=1e-6,
            atol=1e-6,
        )
    )

    valid_tp_match = bool(
        np.allclose(
            X_valid[
                :,
                -2:
            ],

            preprocessing_context[
                "inner_valid_tp_scaled"
            ],

            rtol=1e-6,
            atol=1e-6,
        )
    )

    if not (
        train_tp_match
        and valid_tp_match
    ):
        raise RuntimeError(
            f"{task_id}：表征最后两列"
            "与标准化T/P不一致。"
        )

    # 表征固定以后，才取得下游ML训练和验证目标。
    train_manifest = (
        preprocessing_context[
            "inner_train_manifest"
        ]
    )

    valid_relative_idx = (
        INNER_TASK_INDEX_CACHE[
            (
                split_name,
                outer_fold,
                inner_fold,
            )
        ][
            "inner_valid_relative_idx"
        ]
    )

    outer_train_manifest = (
        OUTER_CONTEXT_CACHE[
            (
                split_name,
                outer_fold,
            )
        ][
            "outer_train_manifest"
        ]
    )

    valid_target_manifest = (
        outer_train_manifest.iloc[
            valid_relative_idx
        ]
        .reset_index(
            drop=True
        )
        .copy()
    )

    y_train = (
        train_manifest[
            "lg_eta"
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    y_valid = (
        valid_target_manifest[
            "lg_eta"
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    if X_train.shape != (
        n_train,
        130,
    ):
        raise RuntimeError(
            f"{task_id}：训练表征形状错误。"
        )

    if X_valid.shape != (
        n_valid,
        130,
    ):
        raise RuntimeError(
            f"{task_id}：验证表征形状错误。"
        )

    if y_train.shape != (
        n_train,
    ):
        raise RuntimeError(
            f"{task_id}：训练目标形状错误。"
        )

    if y_valid.shape != (
        n_valid,
    ):
        raise RuntimeError(
            f"{task_id}：验证目标形状错误。"
        )

    if not (
        np.isfinite(
            X_train
        ).all()
        and np.isfinite(
            X_valid
        ).all()
        and np.isfinite(
            y_train
        ).all()
        and np.isfinite(
            y_valid
        ).all()
    ):
        raise RuntimeError(
            f"{task_id}：特征或目标中"
            "存在NaN或无穷值。"
        )

    train_order_match = bool(
        np.array_equal(
            train_manifest[
                "original_feature_row_idx"
            ].to_numpy(
                dtype=np.int64
            ),

            INNER_TASK_INDEX_CACHE[
                (
                    split_name,
                    outer_fold,
                    inner_fold,
                )
            ][
                "inner_train_global_idx"
            ],
        )
    )

    valid_order_match = bool(
        np.array_equal(
            valid_target_manifest[
                "original_feature_row_idx"
            ].to_numpy(
                dtype=np.int64
            ),

            INNER_TASK_INDEX_CACHE[
                (
                    split_name,
                    outer_fold,
                    inner_fold,
                )
            ][
                "inner_valid_global_idx"
            ],
        )
    )

    if not (
        train_order_match
        and valid_order_match
    ):
        raise RuntimeError(
            f"{task_id}：特征和目标顺序不一致。"
        )

    paths = get_task_core_paths(
        split_name=split_name,
        outer_fold=outer_fold,
        inner_fold=inner_fold,
    )

    batch_save_numpy_atomic(
        X_train,
        paths[
            "X_inner_train_mpnn"
        ],
    )

    batch_save_numpy_atomic(
        X_valid,
        paths[
            "X_inner_valid_mpnn"
        ],
    )

    batch_save_numpy_atomic(
        y_train,
        paths[
            "y_inner_train_lg_eta"
        ],
    )

    batch_save_numpy_atomic(
        y_valid,
        paths[
            "y_inner_valid_lg_eta"
        ],
    )

    training_summary = read_json(
        paths[
            "training_summary"
        ]
    )

    best_state_payload = (
        load_torch_object(
            paths[
                "best_state"
            ]
        )
    )

    output_arrays = {
        "X_inner_train_mpnn": X_train,
        "X_inner_valid_mpnn": X_valid,
        "y_inner_train_lg_eta": y_train,
        "y_inner_valid_lg_eta": y_valid,
    }

    output_file_roles = [
        "X_inner_train_mpnn",
        "X_inner_valid_mpnn",
        "y_inner_train_lg_eta",
        "y_inner_valid_lg_eta",
    ]

    output_sha256 = {
        role: calculate_sha256(
            paths[
                role
            ]
        )
        for role in output_file_roles
    }

    feature_metadata = {
        "created_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),

        "run_code_version": (
            RUN_CODE_VERSION
        ),

        "status": (
            "feature_extraction_completed"
        ),

        "task_id": task_id,

        "split_name": split_name,

        "outer_fold": outer_fold,

        "inner_fold": inner_fold,

        "task_seed": int(
            preprocessing_context[
                "task_seed"
            ]
        ),

        "feature_extraction": {
            "best_model_path": str(
                paths[
                    "best_state"
                ]
            ),

            "best_model_sha256": (
                calculate_sha256(
                    paths[
                        "best_state"
                    ]
                )
            ),

            "best_epoch": int(
                best_state_payload[
                    "best_epoch"
                ]
            ),

            "best_early_stop_loss": float(
                best_state_payload[
                    "best_early_stop_loss"
                ]
            ),

            "descriptor_dim": 130,

            "cation_embedding_dim": 64,

            "anion_embedding_dim": 64,

            "environment_dim": 2,
        },

        "output_shapes": {
            key: list(
                value.shape
            )
            for key, value
            in output_arrays.items()
        },

        "output_dtypes": {
            key: str(
                value.dtype
            )
            for key, value
            in output_arrays.items()
        },

        "output_sha256": (
            output_sha256
        ),

        "output_paths": {
            role: str(
                paths[
                    role
                ]
            )
            for role in output_file_roles
        },
    }

    batch_save_json_atomic(
        data=feature_metadata,
        output_path=paths[
            "inner_feature_metadata"
        ],
    )

    output_checks = {
        "train_feature_shape_correct": bool(
            X_train.shape
            == (
                n_train,
                130,
            )
        ),

        "valid_feature_shape_correct": bool(
            X_valid.shape
            == (
                n_valid,
                130,
            )
        ),

        "train_target_shape_correct": bool(
            y_train.shape
            == (
                n_train,
            )
        ),

        "valid_target_shape_correct": bool(
            y_valid.shape
            == (
                n_valid,
            )
        ),

        "all_features_finite": bool(
            np.isfinite(
                X_train
            ).all()
            and np.isfinite(
                X_valid
            ).all()
        ),

        "all_targets_finite": bool(
            np.isfinite(
                y_train
            ).all()
            and np.isfinite(
                y_valid
            ).all()
        ),

        "train_tp_columns_match": (
            train_tp_match
        ),

        "valid_tp_columns_match": (
            valid_tp_match
        ),

        "train_feature_target_order_match": (
            train_order_match
        ),

        "valid_feature_target_order_match": (
            valid_order_match
        ),
    }

    if not all(
        output_checks.values()
    ):
        failed_checks = [
            key
            for key, value
            in output_checks.items()
            if not value
        ]

        raise RuntimeError(
            f"{task_id}：输出审计失败："
            f"{failed_checks}"
        )

    output_audit = {
        "created_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),

        "run_code_version": (
            RUN_CODE_VERSION
        ),

        "audit_status": "passed",

        "task_id": task_id,

        "checks": output_checks,

        "leakage_control": {
            "inner_validation_label_available_"
            "in_source_manifest": True,

            "inner_validation_label_used_for_"
            "mpnn_training": False,

            "inner_validation_label_used_for_"
            "mpnn_early_stopping": False,

            "inner_validation_label_used_for_"
            "mpnn_scheduler": False,

            "inner_validation_label_used_for_"
            "mpnn_feature_extraction": False,

            "inner_validation_label_used_after_"
            "feature_extraction_for_downstream_"
            "validation": True,

            "outer_test_structure_used": False,

            "outer_test_target_used": False,

            "outer_test_feature_extracted": False,

            "existing_outer_mpnn_features_read": False,

            "old_global_mpnn_feature_read": False,

            "external_10point_holdout_read": False,
        },

        "outputs": {
            role: str(
                path
            )
            for role, path
            in paths.items()
        },

        "training_result": {
            "best_epoch": int(
                training_summary[
                    "result"
                ][
                    "best_epoch"
                ]
            ),

            "best_early_stop_loss": float(
                training_summary[
                    "result"
                ][
                    "best_early_stop_loss"
                ]
            ),
        },
    }

    batch_save_json_atomic(
        data=output_audit,
        output_path=paths[
            "inner_output_audit"
        ],
    )

    return {
        "feature_metadata": (
            feature_metadata
        ),

        "output_audit": (
            output_audit
        ),
    }


# ============================================================
# 6. 生成全部30套任务状态表
# ============================================================

def build_batch_status_table():
    """读取硬盘状态，生成全部30套任务的当前状态表。"""
    rows = []

    ordered_tasks = (
        df_inner_mpnn_tasks
        .copy()
    )

    ordered_tasks[
        "_split_order"
    ] = (
        ordered_tasks[
            "Split"
        ].map(
            {
                "Data_split": 0,
                "IL_split": 1,
            }
        )
    )

    ordered_tasks = (
        ordered_tasks
        .sort_values(
            [
                "_split_order",
                "Outer_fold",
                "Inner_fold",
            ]
        )
        .drop(
            columns=[
                "_split_order",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    for task_row in (
        ordered_tasks.itertuples(
            index=False
        )
    ):
        completion = (
            inspect_task_completion(
                split_name=(
                    task_row.Split
                ),

                outer_fold=int(
                    task_row.Outer_fold
                ),

                inner_fold=int(
                    task_row.Inner_fold
                ),
            )
        )

        rows.append(
            {
                "Task_id": (
                    completion[
                        "task_id"
                    ]
                ),

                "Split": (
                    task_row.Split
                ),

                "Outer_fold": int(
                    task_row.Outer_fold
                ),

                "Inner_fold": int(
                    task_row.Inner_fold
                ),

                "Task_seed": int(
                    task_row.Task_seed
                ),

                "Status": (
                    completion[
                        "status"
                    ]
                ),

                "Completed": bool(
                    completion[
                        "completed"
                    ]
                ),

                "Checkpoint_exists": bool(
                    completion[
                        "checkpoint_exists"
                    ]
                ),

                "Training_completed": bool(
                    completion[
                        "training_completed"
                    ]
                ),

                "All_core_files_exist": bool(
                    completion[
                        "all_core_files_exist"
                    ]
                ),

                "Output_audit_passed": bool(
                    completion[
                        "output_audit_passed"
                    ]
                ),

                "Output_dir": str(
                    completion[
                        "output_dir"
                    ]
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


# ============================================================
# 7. 确定本次要运行的全部未完成任务
# ============================================================

status_before = (
    build_batch_status_table()
)

pending_before = (
    status_before.loc[
        ~status_before[
            "Completed"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

tasks_this_run = (
    pending_before
    .head(
        MAX_NEW_TASKS_THIS_RUN
    )
    .copy()
)

run_started_at = (
    datetime.now().isoformat(
        timespec="seconds"
    )
)

batch_run_state = {
    "updated_at": run_started_at,

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "status": "running",

    "maximum_new_tasks_this_run": int(
        MAX_NEW_TASKS_THIS_RUN
    ),

    "completed_before_run": int(
        status_before[
            "Completed"
        ].sum()
    ),

    "pending_before_run": int(
        (
            ~status_before[
                "Completed"
            ]
        ).sum()
    ),

    "planned_task_ids": (
        tasks_this_run[
            "Task_id"
        ].tolist()
    ),

    "completed_this_run": [],

    "failed_task": None,

    "error": None,
}

batch_save_json_atomic(
    data=batch_run_state,
    output_path=BATCH_RUN_STATE_PATH,
)

print("=" * 100)
print("Cell 8 全部剩余内层MPNN任务运行开始")
print("=" * 100)

print(
    "completed before run =",
    batch_run_state[
        "completed_before_run"
    ],
)

print(
    "pending before run   =",
    batch_run_state[
        "pending_before_run"
    ],
)

print(
    "tasks this run       =",
    len(
        tasks_this_run
    ),
)

print(
    "planned task ids     ="
)

for task_id in (
    batch_run_state[
        "planned_task_ids"
    ]
):
    print(
        "  ",
        task_id,
    )

if len(
    tasks_this_run
) == 0:
    print(
        "\n全部30套任务此前已经完成，"
        "本次不需要继续训练。"
    )


# ============================================================
# 8. 顺序运行全部未完成任务
# ============================================================

for task_row in (
    tasks_this_run.itertuples(
        index=False
    )
):
    split_name = (
        task_row.Split
    )

    outer_fold = int(
        task_row.Outer_fold
    )

    inner_fold = int(
        task_row.Inner_fold
    )

    task_id = (
        task_row.Task_id
    )

    try:
        print("\n" + "#" * 100)

        print(
            f"处理任务：{task_id}"
        )

        print("#" * 100)

        preprocessing_context = (
            build_inner_mpnn_preprocessing_context(
                split_name=split_name,
                outer_fold=outer_fold,
                inner_fold=inner_fold,
                save_outputs=True,
            )
        )

        dataloader_context = (
            build_task_dataloaders(
                preprocessing_context=(
                    preprocessing_context
                )
            )
        )

        save_generic_task_interface_audits(
            preprocessing_context=(
                preprocessing_context
            ),

            dataloader_context=(
                dataloader_context
            ),
        )

        training_result = (
            train_inner_mpnn_batch_task(
                preprocessing_context=(
                    preprocessing_context
                ),

                dataloader_context=(
                    dataloader_context
                ),
            )
        )

        extract_batch_task_features(
            preprocessing_context=(
                preprocessing_context
            ),

            dataloader_context=(
                dataloader_context
            ),

            trained_model=(
                training_result[
                    "model"
                ]
            ),
        )

        completion_after_task = (
            inspect_task_completion(
                split_name=split_name,
                outer_fold=outer_fold,
                inner_fold=inner_fold,
            )
        )

        if not completion_after_task[
            "completed"
        ]:
            raise RuntimeError(
                f"{task_id}：训练和表征提取结束后，"
                "任务仍未达到completed状态。"
            )

        batch_run_state[
            "completed_this_run"
        ].append(
            task_id
        )

        batch_run_state[
            "updated_at"
        ] = (
            datetime.now().isoformat(
                timespec="seconds"
            )
        )

        batch_save_json_atomic(
            data=batch_run_state,
            output_path=(
                BATCH_RUN_STATE_PATH
            ),
        )

        print(
            f"{task_id} 完成 ✓"
        )

        del (
            preprocessing_context,
            dataloader_context,
            training_result,
        )

        gc.collect()

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    except BaseException as error:
        batch_run_state[
            "updated_at"
        ] = (
            datetime.now().isoformat(
                timespec="seconds"
            )
        )

        batch_run_state[
            "status"
        ] = "failed"

        batch_run_state[
            "failed_task"
        ] = task_id

        batch_run_state[
            "error"
        ] = {
            "type": type(
                error
            ).__name__,

            "message": str(
                error
            ),

            "traceback": (
                traceback.format_exc()
            ),
        }

        batch_save_json_atomic(
            data=batch_run_state,
            output_path=(
                BATCH_RUN_STATE_PATH
            ),
        )

        current_status = (
            build_batch_status_table()
        )

        current_status.to_csv(
            BATCH_STATUS_TABLE_PATH,
            index=False,
            encoding="utf-8-sig",
        )

        raise


# ============================================================
# 9. 保存运行后的30套任务状态
# ============================================================

status_after = (
    build_batch_status_table()
)

status_after.to_csv(
    BATCH_STATUS_TABLE_PATH,
    index=False,
    encoding="utf-8-sig",
)

completed_after_count = int(
    status_after[
        "Completed"
    ].sum()
)

pending_after_count = int(
    (
        ~status_after[
            "Completed"
        ]
    ).sum()
)

checkpoint_after_count = int(
    status_after[
        "Checkpoint_exists"
    ].sum()
)

audit_passed_after_count = int(
    status_after[
        "Output_audit_passed"
    ].sum()
)

batch_run_state[
    "updated_at"
] = (
    datetime.now().isoformat(
        timespec="seconds"
    )
)

batch_run_state[
    "status"
] = (
    "all_tasks_completed"
    if pending_after_count == 0
    else "batch_completed"
)

batch_run_state[
    "completed_after_run"
] = (
    completed_after_count
)

batch_run_state[
    "pending_after_run"
] = (
    pending_after_count
)

batch_run_state[
    "checkpoint_after_run"
] = (
    checkpoint_after_count
)

batch_run_state[
    "output_audit_passed_after_run"
] = (
    audit_passed_after_count
)

batch_run_state[
    "status_table_path"
] = str(
    BATCH_STATUS_TABLE_PATH
)

batch_save_json_atomic(
    data=batch_run_state,
    output_path=BATCH_RUN_STATE_PATH,
)


# ============================================================
# 10. 最终强制检查
# ============================================================

if pending_after_count == 0:
    if completed_after_count != 30:
        raise RuntimeError(
            "待运行任务已为0，"
            f"但Completed任务数为{completed_after_count}，"
            "并非30。"
        )

    if audit_passed_after_count != 30:
        raise RuntimeError(
            "30套任务已运行结束，"
            f"但通过输出审计的任务只有"
            f"{audit_passed_after_count}套。"
        )

    if checkpoint_after_count != 0:
        raise RuntimeError(
            "全部任务已完成，"
            f"但仍有{checkpoint_after_count}个断点文件。"
        )


# ============================================================
# 11. 输出最终结果
# ============================================================

print("\n" + "=" * 100)
print("Cell 8 全部剩余内层MPNN任务运行结束 ✓")
print("=" * 100)

print(
    "completed before run =",
    batch_run_state[
        "completed_before_run"
    ],
)

print(
    "completed this run   =",
    len(
        batch_run_state[
            "completed_this_run"
        ]
    ),
)

print(
    "completed after run  =",
    completed_after_count,
)

print(
    "pending after run    =",
    pending_after_count,
)

print(
    "output audits passed =",
    audit_passed_after_count,
)

print(
    "checkpoints remaining =",
    checkpoint_after_count,
)

print(
    "batch status         =",
    batch_run_state[
        "status"
    ],
)

print(
    "\n本次完成任务："
)

for task_id in (
    batch_run_state[
        "completed_this_run"
    ]
):
    print(
        "  ",
        task_id,
    )

print(
    "\n全部30套任务状态汇总："
)

display(
    status_after.groupby(
        [
            "Split",
            "Status",
        ],
        as_index=False,
    ).size()
)

print(
    "\n全部30套任务明细："
)

display(
    status_after[
        [
            "Task_id",
            "Split",
            "Outer_fold",
            "Inner_fold",
            "Status",
            "Completed",
            "Checkpoint_exists",
            "Output_audit_passed",
        ]
    ]
)

print(
    "\n任务状态表："
)

print(
    BATCH_STATUS_TABLE_PATH
)

print(
    "\n批量运行状态："
)

print(
    BATCH_RUN_STATE_PATH
)

Cell 8 全部剩余内层MPNN任务运行开始
completed before run = 1
pending before run   = 29
tasks this run       = 29
planned task ids     =
   Data_split_OuterFold1_InnerFold2
   Data_split_OuterFold1_InnerFold3
   Data_split_OuterFold2_InnerFold1
   Data_split_OuterFold2_InnerFold2
   Data_split_OuterFold2_InnerFold3
   Data_split_OuterFold3_InnerFold1
   Data_split_OuterFold3_InnerFold2
   Data_split_OuterFold3_InnerFold3
   Data_split_OuterFold4_InnerFold1
   Data_split_OuterFold4_InnerFold2
   Data_split_OuterFold4_InnerFold3
   Data_split_OuterFold5_InnerFold1
   Data_split_OuterFold5_InnerFold2
   Data_split_OuterFold5_InnerFold3
   IL_split_OuterFold1_InnerFold1
   IL_split_OuterFold1_InnerFold2
   IL_split_OuterFold1_InnerFold3
   IL_split_OuterFold2_InnerFold1
   IL_split_OuterFold2_InnerFold2
   IL_split_OuterFold2_InnerFold3
   IL_split_OuterFold3_InnerFold1
   IL_split_OuterFold3_InnerFold2
   IL_split_OuterFold3_InnerFold3
   IL_split_OuterFold4_InnerFold1
   IL_split_OuterFold4_InnerFold

,Split,Status,size
0,Data_split,completed,15
1,IL_split,completed,15



全部30套任务明细：


,Task_id,Split,Outer_fold,Inner_fold,Status,Completed,Checkpoint_exists,Output_audit_passed
0,Data_split_OuterFold1_InnerFold1,Data_split,1,1,completed,True,False,True
1,Data_split_OuterFold1_InnerFold2,Data_split,1,2,completed,True,False,True
2,Data_split_OuterFold1_InnerFold3,Data_split,1,3,completed,True,False,True
3,Data_split_OuterFold2_InnerFold1,Data_split,2,1,completed,True,False,True
4,Data_split_OuterFold2_InnerFold2,Data_split,2,2,completed,True,False,True
5,Data_split_OuterFold2_InnerFold3,Data_split,2,3,completed,True,False,True
6,Data_split_OuterFold3_InnerFold1,Data_split,3,1,completed,True,False,True
7,Data_split_OuterFold3_InnerFold2,Data_split,3,2,completed,True,False,True
8,Data_split_OuterFold3_InnerFold3,Data_split,3,3,completed,True,False,True
9,Data_split_OuterFold4_InnerFold1,Data_split,4,1,completed,True,False,True



任务状态表：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\inner_foldwise_mpnn_tuning_features\_run_logs\inner_mpnn_batch_task_status_Data_IL.csv

批量运行状态：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\inner_foldwise_mpnn_tuning_features\_run_logs\inner_mpnn_batch_run_state_Data_IL.json
